**PYTHON**

**CONTEXT MANAGERS**

*The Complete Handbook - From Absolute Beginner to Professional*

A Comprehensive Teaching Textbook and Reference Covering Resource Management, the `with` Statement, `contextlib`, Custom Context Managers, Asynchronous Context Managers, Context Variables, Real-World Projects, and Applications in Data Science, Machine Learning, AI Engineering, Databases, and Backend Systems

**Kajola Gbenga**

*Data Scientist & AI Engineer*

**Table of Contents**

1\. Introduction to Context Managers

2\. The `with` Statement

3\. Why Resource Cleanup Matters

4\. How `with` Works Internally

5\. The Context Manager Protocol

6\. Building a Custom Class-Based Context Manager

7\. `__enter__()` in Detail

8\. `__exit__()` in Detail

9\. Context Managers and Exceptions

10\. Suppressing Exceptions

11\. `contextlib`

12\. `@contextmanager`

13\. `@contextmanager` with Exception Handling

14\. `closing()`

15\. `suppress()`

16\. `redirect_stdout()` and `redirect_stderr()`

17\. `nullcontext()`

18\. `ExitStack`

19\. Multiple Context Managers

20\. Nested Context Managers

21\. File Handling with Context Managers

22\. Database Context Managers

23\. Locks and Thread Safety

24\. Temporary Resources

25\. Context Managers in Data Science

26\. Context Managers in Machine Learning

27\. Context Managers in AI Engineering

28\. Asynchronous Context Managers

29\. `@asynccontextmanager`

30\. Context Managers in Web Applications

31\. Context Managers and Decorators

32\. Context Managers and OOP

33\. Advanced Context Manager Design

34\. Reentrant Context Managers

35\. Context Variables (`contextvars`)

36\. Testing Context Managers

37\. Debugging Context Managers

38\. Common Mistakes

39\. Production Best Practices

40\. Real-World Projects

41\. Interview Preparation

42\. Practical Exercises

43\. Complete Cheat Sheet

44\. Conclusion

# 1. Introduction to Context Managers

## 1.1 What a Context Manager Is

A **context manager** is an object that defines a well-bounded "setup, then guaranteed cleanup" lifecycle around a block of code, accessed through Python's `with` statement. It guarantees that some cleanup action (closing a file, releasing a lock, committing or rolling back a transaction) happens when the block ends - whether the block finished normally or was interrupted by an exception.

> **Note on running this notebook:** a few cells in this notebook use `asyncio.run()`, exactly as you would write it in a standalone Python script. Jupyter itself already runs an event loop, so the cell below applies `nest_asyncio` once, purely so those `asyncio.run()` calls work inside this notebook too — this is a notebook-specific accommodation, not something you need in a normal script.

In [1]:
try:
    import nest_asyncio
except ImportError:
    %pip install -q nest_asyncio
    import nest_asyncio
nest_asyncio.apply()

In [2]:
with open("notes.txt", "w") as f:
    f.write("Hello, context managers!")
# by the time this line runs, the file is GUARANTEED to already be closed

**Output:**
```

```

## 1.2 Why Context Managers Exist - The Problem They Solve

Without context managers, resource cleanup must be handled manually, and manual cleanup is easy to get wrong - especially once exceptions enter the picture.

In [3]:
f = open("notes.txt", "w")
f.write("Hello")
f.close()          # if an exception happened between write() and close(), this NEVER RUNS

In [4]:
# The "safe" manual version requires explicit try/finally every single time
f = open("notes.txt", "w")
try:
    f.write("Hello")
finally:
    f.close()        # now guaranteed to run, but verbose and easy to forget

Context managers exist to make this guarantee **automatic and impossible to forget**:

In [5]:
with open("notes.txt", "w") as f:
    f.write("Hello")
# cleanup (closing the file) happens automatically, even if write() raised an exception

## 1.3 Setup and Cleanup - The Core Pattern

Every context manager follows the same two-phase shape:

| Phase | When It Runs | Typical Action |
|---|---|---|
| Setup (enter) | Immediately when the `with` block begins | Acquire a resource: open a file, connect to a database, acquire a lock |
| Cleanup (exit) | Immediately when the `with` block ends, for any reason | Release the resource: close the file, close the connection, release the lock |

## 1.4 Deterministic Cleanup

**Deterministic cleanup** means cleanup happens at a precise, predictable point in the code - right when the `with` block ends - rather than at some unpredictable later time. This is a meaningful contrast with relying on Python's garbage collector or `__del__` (covered in the OOP reference) to eventually free a resource: garbage collection timing is not guaranteed, whereas `with`-based cleanup runs immediately and reliably, every time.

In [6]:
class NoisyResource:
    def __del__(self):
        print("Cleaned up by garbage collector - timing NOT guaranteed")

def risky():
    r = NoisyResource()
    # if this function is long, or r ends up referenced elsewhere, cleanup could be delayed indefinitely

## 1.5 Real-World Analogies

Think of a context manager like borrowing a library book, renting a hotel room, or checking equipment out of a supply closet: there is always a clear **check-out** step (setup) and a **check-in** step (cleanup), and the whole point of a good system is making sure the check-in step never gets forgotten, no matter what happens while you have the item.

Another common analogy: a restaurant reservation. You are seated (`__enter__`), you have your meal (the `with` block body), and no matter how the meal goes - you finish happily, or there's a kitchen problem - the table is eventually released (`__exit__`) so the restaurant's state stays consistent for the next guest.

## 1.6 Common Resources That Need Management

| Resource | Setup | Cleanup |
|---|---|---|
| Files | Open the file | Close the file, flushing any buffered writes |
| Database connections | Connect to the database | Close the connection, releasing the server-side session |
| Network connections | Open a socket/session | Close the socket, release connection-pool slots |
| Locks | Acquire the lock | Release the lock, so other threads/processes can proceed |
| Temporary resources | Create a temp file/directory | Delete the temp file/directory |
| Transactions | Begin a transaction | Commit on success, roll back on failure |

## 1.7 Motivating Examples

### Files

In [7]:
with open("data.csv", "w") as f:
    f.write("id,value\n1,100\n2,200\n")

with open("data.csv", "r") as f:
    contents = f.read()
print(len(contents), "characters read")

21 characters read


**Output:**
```
21 characters read
```

### Database Connections (Conceptual, Using `sqlite3` from the Standard Library)

In [8]:
import sqlite3

with sqlite3.connect(":memory:") as conn:
    conn.execute("CREATE TABLE users (id INTEGER, name TEXT)")
    conn.execute("INSERT INTO users VALUES (1, 'Ada')")
# connection is automatically committed/closed-managed by the context manager
print("Database operation complete")

Database operation complete


**Output:**
```
Database operation complete
```

### Locks (Thread Synchronization, Covered Fully in Part 23)

In [9]:
import threading

lock = threading.Lock()

with lock:
    print("Critical section - only one thread at a time can be here")

Critical section - only one thread at a time can be here


**Output:**
```
Critical section - only one thread at a time can be here
```

### Temporary Resources (Covered Fully in Part 24)

In [10]:
import tempfile

with tempfile.TemporaryDirectory() as tmp_dir:
    print(f"Working in: {tmp_dir}")
# the directory and everything in it is automatically deleted here
print("Temporary directory has been cleaned up")

Working in: /tmp/tmpzfjugcf_
Temporary directory has been cleaned up


**Output:**
```
Working in: /tmp/tmpabc123xyz
Temporary directory has been cleaned up
```

### Transactions (Covered Fully in Part 22)

In [11]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (id INTEGER, balance INTEGER)")
conn.execute("INSERT INTO accounts VALUES (1, 100)")
conn.commit()

try:
    with conn:                      # sqlite3 connections act as transaction context managers
        conn.execute("UPDATE accounts SET balance = balance - 200 WHERE id = 1")
        if conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()[0] < 0:
            raise ValueError("Insufficient funds - rolling back")
except ValueError as e:
    print("Error:", e)

print(conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()[0])

Error: Insufficient funds - rolling back
100


**Output:**
```
Error: Insufficient funds - rolling back
100
```

## 1.8 Summary Table

| Concept | Meaning |
|---|---|
| Context manager | An object guaranteeing setup and cleanup around a `with` block |
| Setup | Resource acquisition, run when entering the block |
| Cleanup | Resource release, run when leaving the block, for any reason |
| Deterministic cleanup | Cleanup happens at a precise, predictable point, not left to garbage collection timing |
| `with` statement | Python's syntax for using a context manager |

### Exercises - Section 1

**Beginner**
1. Write the manual `try/finally` equivalent of `with open("x.txt") as f: f.read()`.
2. List three resources in your own past code that should have used a context manager but didn't.

**Intermediate**
3. Explain, in your own words, why relying on `__del__` for cleanup (from the OOP reference) is less reliable than a context manager's `__exit__`.

**Advanced**
4. Describe a real production incident (hypothetical or real) that could result from forgetting to close a database connection, and explain how a context manager would have prevented it.
# 2. The `with` Statement

## 2.1 A First Example, Fully Dissected

In [12]:
# (Setup, so the example below has a real file to read - not part of the pattern itself)
with open("data.txt", "w") as f:
    f.write("Sample file content for this example.\n")

In [13]:
with open("data.txt", "r") as file:
    data = file.read()

| Component | Role |
|---|---|
| `with` | The keyword that begins a context-managed block |
| `open("data.txt", "r")` | The **context expression** - evaluated once, producing a context manager object |
| (the object returned by `open(...)`) | The **context manager** itself - it defines `__enter__` and `__exit__` |
| `as file` | Assigns whatever `__enter__()` returns to the name `file` |
| `file` | The **target variable** - what the code inside the block actually works with |
| The indented block | The code that runs "inside" the context, with the resource available |

## 2.2 The Context Expression

The **context expression** is everything between `with` and the (optional) `as` clause - it's evaluated exactly once, and must produce an object implementing the context manager protocol (`__enter__`/`__exit__`, Part 5).


```
with open("data.txt") as f:        # open("data.txt") is the context expression
    pass

with some_lock:                      # some_lock itself is the context expression
    pass

with SomeClass(arg1, arg2) as obj:    # SomeClass(arg1, arg2) is the context expression
    pass
```

## 2.3 The Context Manager

The **context manager** is the object the context expression evaluates to. It is not necessarily the same object bound by `as` - the context manager's `__enter__()` method decides what actually gets assigned (Part 7).

In [14]:
f = open("data.txt")     # 'f' here IS the context manager (a file object with __enter__/__exit__)
print(hasattr(f, "__enter__"), hasattr(f, "__exit__"))
f.close()

True True


**Output:**
```
True True
```

## 2.4 The Target Variable (`as`)

The `as` clause is optional. When present, it binds whatever `__enter__()` returns - which is often, but not always, the context manager itself.

In [15]:
with open("data.txt", "w") as f:      # f is what __enter__() returned (the file object itself, here)
    f.write("hello")

import threading
lock = threading.Lock()
with lock:                              # no 'as' - the returned value (also the lock) is simply not used
    print("locked")

locked


**Output:**
```
locked
```

## 2.5 The Indented Block

Everything indented under the `with` statement is the block whose execution is "wrapped" by the context manager's setup and cleanup.

In [16]:
with open("data.txt", "w") as f:
    f.write("line 1\n")     # inside the block - resource is available
    f.write("line 2\n")     # still inside the block
print("outside the block - file is already closed here")

outside the block - file is already closed here


**Output:**
```
outside the block - file is already closed here
```

## 2.6 Entering and Exiting the Context - What Actually Happens

1. The context expression is evaluated, producing the context manager object.
2. Its `__enter__()` method is called - this is **entering the context**.
3. The return value of `__enter__()` is bound to the `as` target, if present.
4. The indented block executes.
5. When the block ends (normally or via exception), the context manager's `__exit__()` method is called - this is **exiting the context**.
6. Cleanup inside `__exit__()` runs, and any exception is either propagated or suppressed based on `__exit__()`'s return value (Part 8, Part 10).

In [17]:
class Demo:
    def __enter__(self):
        print("Entering the context")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Exiting the context")
        return False

with Demo() as d:
    print("Inside the block")

Entering the context
Inside the block
Exiting the context


**Output:**
```
Entering the context
Inside the block
Exiting the context
```

## 2.7 Automatic Cleanup - Even When Things Go Wrong

The critical guarantee: `__exit__()` runs **even if the block raises an exception**, which is precisely why `with` is preferred over manual open/close pairs.

In [18]:
class Demo:
    def __enter__(self):
        print("Entering")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Exiting (cleanup always runs)")
        return False       # False means: let the exception continue propagating

try:
    with Demo():
        print("About to raise an error")
        raise ValueError("Something went wrong")
except ValueError as e:
    print("Caught outside the with block:", e)

Entering
About to raise an error
Exiting (cleanup always runs)
Caught outside the with block: Something went wrong


**Output:**
```
Entering
About to raise an error
Exiting (cleanup always runs)
Caught outside the with block: Something went wrong
```

## 2.8 Why `with` Is Preferable to Manual Open/Close

| Manual Approach | `with` Statement |
|---|---|
| Requires an explicit `try/finally` to be exception-safe | Exception-safety is built in automatically |
| Easy to forget the cleanup call entirely | Cleanup cannot be "forgotten" - it's structural |
| Cleanup code is separated from setup code, sometimes far apart | Setup and cleanup are defined together in one place (the context manager) |
| More lines of boilerplate for every single resource use | One line of intent: "use this resource for this block" |

## 2.9 Summary Table

| Term | Definition | In `with open("f") as file:` |
|---|---|---|
| Context expression | The expression producing the context manager | `open("f")` |
| Context manager | The object with `__enter__`/`__exit__` | The file object returned by `open` |
| Target variable | The name bound by `as` | `file` |
| Block | The indented code using the resource | Everything under the `with` line |
| Entering | Calling `__enter__()` | Happens before the block runs |
| Exiting | Calling `__exit__()` | Happens after the block ends, always |

### Exercises - Section 2

**Beginner**
1. Identify the context expression, context manager, and target variable in: `with requests.Session() as session:`.
2. Write a `with` statement using `open()` with no `as` clause, and explain why that's valid even though you can't reference the file object directly afterward (hint: it isn't valid for reading/writing without a reference).

**Intermediate**
3. Predict the exact output order of a custom context manager's print statements when the block body raises an exception, then verify by running code.

**Advanced**
4. Explain, step by step, what "entering" and "exiting" mean in terms of actual method calls Python makes, without looking back at Section 2.6.
# 3. Why Resource Cleanup Matters

## 3.1 Open Files Left Unclosed

Every open file consumes a **file descriptor**, a limited operating-system resource. Programs that open many files without closing them can exhaust this limit, causing every subsequent `open()` call in the entire process to fail.

In [19]:
# BAD - file descriptors accumulate, never released until the process ends
files = []
for i in range(5):
    f = open(f"temp_{i}.txt", "w")     # never closed!
    files.append(f)
print("5 file descriptors leaked")

5 file descriptors leaked


**Output:**
```
5 file descriptors leaked
```

Unclosed files can also lose data: writes are often buffered in memory and only guaranteed to reach disk when the file is closed (or explicitly flushed) - an unclosed file after a crash may leave a truncated or empty file on disk.

## 3.2 Database Connections Left Open

Database servers allow only a limited number of simultaneous connections. Leaked connections (never closed) accumulate until the server refuses new connections entirely, an outage that can affect every part of an application, not just the code that leaked the connection.

In [20]:
# BAD - conceptually: opening a connection per request and never closing it
def handle_request_badly():
    conn = get_database_connection()      # never closed
    conn.execute("SELECT * FROM users")
    # function returns - conn is unreachable from here, but the SERVER still thinks it's open

## 3.3 Network Connections Left Open

Similar to database connections: open sockets and HTTP sessions consume both local (file-descriptor-like) and remote (server-side) resources. Leaked network connections can lead to connection-pool exhaustion, causing seemingly unrelated requests elsewhere in the application to fail or hang.

## 3.4 Locks Never Released

A lock (Part 23) that is acquired but never released causes every other thread waiting for that lock to block **forever** - a **deadlock**. This is one of the most severe classes of concurrency bugs, since the program does not crash; it simply freezes.

In [21]:
import threading

lock = threading.Lock()

def risky_section():
    lock.acquire()
    raise RuntimeError("Oops")
    lock.release()          # NEVER REACHED - the lock stays acquired forever

try:
    risky_section()
except RuntimeError:
    print("Exception occurred - but the lock was never released!")
print("Lock state:", lock.locked())

Exception occurred - but the lock was never released!
Lock state: True


**Output:**
```
Exception occurred - but the lock was never released!
Lock state: True
```

## 3.5 Memory and Resource Exhaustion

Beyond files and connections, any acquired resource that is never released - buffers, temporary files, GPU memory allocations, in-memory caches tied to a session - can accumulate over the lifetime of a long-running process (a web server, a data pipeline, a training job), eventually degrading performance or crashing the process entirely.

## 3.6 Partial Operations

Without deterministic cleanup, an operation that is interrupted partway through can leave the system in an inconsistent, "partially done" state - a file half-written, a database row updated but not committed, a lock held by a thread that has already crashed.

In [22]:
# BAD - if writing fails partway through, the file is left with incomplete content
try:
    f = open("report.txt", "w")
    f.write("Header\n")
    f.write("Row 1\n")
    raise RuntimeError("Simulated crash mid-write")   # report.txt now contains a truncated report
    f.write("Row 2\n")
    f.close()
except RuntimeError as e:
    print(f"Crash occurred: {e}")
    print("Note: f.close() was never reached - the file may still be buffered/incomplete on disk")

Crash occurred: Simulated crash mid-write
Note: f.close() was never reached - the file may still be buffered/incomplete on disk


**Output:**
```
Crash occurred: Simulated crash mid-write
Note: f.close() was never reached - the file may still be buffered/incomplete on disk
```

## 3.7 Application Crashes and Cascading Failures

In long-running applications (web servers, data pipelines, ML training jobs), resource leaks rarely crash the program immediately - they degrade it slowly, until file descriptors or connections are exhausted, at which point failures can cascade unpredictably across completely unrelated parts of the system, making the root cause (a single leaked resource, opened far earlier) very difficult to trace.

## 3.8 Bad vs Good - Side-by-Side

In [23]:
# BAD - manual cleanup, not exception-safe
def process_file_bad(path):
    f = open(path, "r")
    data = f.read()
    result = int(data)          # if this raises (e.g. data isn't a number), f.close() never runs
    f.close()
    return result

In [24]:
# GOOD - context manager guarantees cleanup regardless of what happens inside the block
def process_file_good(path):
    with open(path, "r") as f:
        data = f.read()
        result = int(data)       # even if this raises, the file is still closed correctly
    return result

In [25]:
# BAD - lock never released if an exception occurs
def critical_section_bad(lock):
    lock.acquire()
    do_risky_work()               # if this raises, the lock is held forever
    lock.release()

In [26]:
# GOOD - context manager guarantees the lock is released
def critical_section_good(lock):
    with lock:
        do_risky_work()            # lock is released even if this raises

## 3.9 Summary Table

| Unmanaged Resource | Consequence |
|---|---|
| Open files | File descriptor exhaustion, lost/truncated data on crash |
| Database connections | Connection pool exhaustion, server-side outages |
| Network connections | Connection-pool exhaustion, hangs, timeouts |
| Locks | Deadlocks - other threads block forever |
| General resources (memory, GPU, temp files) | Gradual resource exhaustion in long-running processes |
| Interrupted operations | Partial writes, inconsistent state |
| Aggregate effect | Cascading, hard-to-trace application failures |

### Exercises - Section 3

**Beginner**
1. Rewrite Section 3.8's "BAD" file-processing function using a `with` statement, and explain exactly which line was previously unsafe.

**Intermediate**
2. Write a small script that intentionally opens 10 files without closing them, and use `resource.getrlimit` (or research your OS's file descriptor limit) to understand how close a real leak could come to causing failures.

**Advanced**
3. Describe, referencing Section 3.4, exactly why a deadlock is often more dangerous in production than a crash, from an operations/incident-response perspective.
# 4. How `with` Works Internally

## 4.1 The Conceptual Sequence

When Python executes:


```
with resource() as value:
    BODY
```

it conceptually performs the following sequence of steps:

1. **Evaluate the context expression** - call `resource()`, producing a context manager object.
2. **Obtain the context manager** - the object returned in step 1 is the context manager for this block.
3. **Call `__enter__()`** on that context manager.
4. **Assign the returned value** from `__enter__()` to `value` (the `as` target), if an `as` clause is present.
5. **Execute `BODY`.**
6. **If `BODY` raises an exception**, capture its type, value, and traceback.
7. **Call `__exit__(exc_type, exc_value, traceback)`** on the context manager - passing `(None, None, None)` if no exception occurred.
8. **Decide whether to propagate or suppress** the exception, based on `__exit__()`'s return value (Part 8, Part 10).

## 4.2 Conceptual Flow Diagram


```
with resource() as value:
    BODY

        |
        v
[1] Evaluate resource() -----> produces context manager object (cm)
        |
        v
[2] Call cm.__enter__() -----> returns some value
        |
        v
[3] value = (that returned value)
        |
        v
[4] Run BODY
        |
        +---- no exception ----+
        |                       |
        v                       v
   [5a] Call                [5b] Exception raised inside BODY
   cm.__exit__(None,             |
     None, None)                 v
        |                   Call cm.__exit__(exc_type, exc_value, traceback)
        |                       |
        v                       v
   Continue after       Did __exit__ return a truthy value?
   the with block            |         |
                             Yes        No
                              |         |
                              v         v
                        Exception    Exception
                        SUPPRESSED   RE-RAISED
                        (execution   (propagates
                         continues    up the call
                         normally)    stack)
```

## 4.3 An Equivalent Manual Implementation

This "unrolled" version, using a plain `try/except/finally`, shows precisely what the `with` statement automates - and why it is not just shorter, but structurally safer.

In [27]:
class LoggingContext:
    def __enter__(self):
        print("Entering context")
        return "resource_handle"

    def __exit__(self, exc_type, exc_value, traceback):
        print("Exiting context")
        if exc_type is not None:
            print(f"An exception occurred: {exc_type.__name__}: {exc_value}")
        return False       # never suppress, for this example

# What "with LoggingContext() as value: BODY" does, step by step:
cm = LoggingContext()               # [1] evaluate the context expression
value = cm.__enter__()               # [3] call __enter__
exc_type = exc_value = traceback = None
try:
    print(f"Using {value}")           # [4] run BODY
except BaseException as e:
    import sys
    exc_type, exc_value, traceback = type(e), e, e.__traceback__
    raise
finally:
    suppress = cm.__exit__(exc_type, exc_value, traceback)   # [7] always call __exit__
    if exc_type is not None and not suppress:
        pass   # the original 'raise' above already re-raises; this illustrates the logic

Entering context
Using resource_handle
Exiting context


**Output:**
```
Entering context
Using resource_handle
Exiting context
```

## 4.4 What Actually Runs the Protocol - `contextlib`'s Perspective

Under the hood, CPython implements `with` using dedicated bytecode instructions (historically `SETUP_WITH`/`WITH_EXCEPT_START`, evolving between versions), but the *conceptual* model above - evaluate, enter, run body, exit with exception info, decide suppression - is exactly what every Python version guarantees, regardless of the specific bytecode used to implement it.

## 4.5 Why This Matters for Understanding Custom Context Managers

Every custom context manager you write (Part 6 onward) is really just supplying the two "plug points" in this fixed sequence: what happens in step 3 (`__enter__`) and what happens in step 7 (`__exit__`). Understanding this sequence precisely is what makes it possible to reason correctly about exception handling, suppression, and cleanup ordering in even the most advanced context manager designs later in this guide (nested contexts, `ExitStack`, async context managers).

## 4.6 Summary Table

| Step | Action | Method/Mechanism |
|---|---|---|
| 1 | Evaluate the context expression | Ordinary Python expression evaluation |
| 2 | Obtain the context manager object | The expression's result |
| 3 | Enter the context | `cm.__enter__()` |
| 4 | Bind the `as` target | Assignment from `__enter__()`'s return value |
| 5 | Run the block body | Normal statement execution |
| 6 | Capture exception info (if any) | `exc_type`, `exc_value`, `traceback` |
| 7 | Exit the context | `cm.__exit__(exc_type, exc_value, traceback)` |
| 8 | Suppress or propagate | Based on `__exit__()`'s return value |

### Exercises - Section 4

**Beginner**
1. Trace through the flow diagram by hand for `with open("x.txt") as f: f.read()` assuming no exception occurs.

**Intermediate**
2. Trace through the flow diagram by hand for the same code, assuming `f.read()` raises an exception, and identify exactly which step decides whether that exception propagates.

**Advanced**
3. Implement the "unrolled" manual version from Section 4.3 for a context manager of your own design, and verify it produces identical behavior to using an actual `with` statement.
# 5. The Context Manager Protocol

## 5.1 The Two Required Methods

Any object that implements both `__enter__()` and `__exit__()` satisfies the **context manager protocol** and can be used after `with`.

In [28]:
class MinimalContextManager:
    def __enter__(self):
        pass          # setup logic goes here

    def __exit__(self, exc_type, exc_value, traceback):
        pass           # cleanup logic goes here

with MinimalContextManager():
    print("This works - both required dunder methods are present")

This works - both required dunder methods are present


**Output:**
```
This works - both required dunder methods are present
```

## 5.2 `__enter__()` - What It Does and When It's Called

`__enter__()` is called immediately when the `with` statement begins, **before** the indented block runs. Its return value becomes the `as` target, if one is used.

In [29]:
class Resource:
    def __enter__(self):
        print("Acquiring resource")
        return "the actual resource handle"

    def __exit__(self, exc_type, exc_value, traceback):
        print("Releasing resource")
        return False

with Resource() as handle:
    print(f"Using: {handle}")

Acquiring resource
Using: the actual resource handle
Releasing resource


**Output:**
```
Acquiring resource
Using: the actual resource handle
Releasing resource
```

## 5.3 `__exit__()` - What It Does and When It's Called

`__exit__()` is called when the `with` block ends, for **any** reason: normal completion, a `return`/`break`/`continue` inside the block, or an exception. It receives information about any exception that occurred, and its return value determines whether that exception is suppressed.

In [30]:
class Resource:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"__exit__ called with exc_type={exc_type}")
        return False

with Resource():
    print("Normal completion")

try:
    with Resource():
        raise ValueError("boom")
except ValueError:
    print("Exception propagated as expected")

Normal completion
__exit__ called with exc_type=None
__exit__ called with exc_type=<class 'ValueError'>
Exception propagated as expected


**Output:**
```
__exit__ called with exc_type=None
Normal completion
__exit__ called with exc_type=<class 'ValueError'>
Exception propagated as expected
```

## 5.4 The `__exit__` Parameters in Detail

In [31]:
def __exit__(self, exc_type, exc_value, traceback):
    ...

| Parameter | Meaning | Value When No Exception Occurred |
|---|---|---|
| `exc_type` | The exception **class** (e.g. `ValueError`), if one was raised in the block | `None` |
| `exc_value` | The actual exception **instance** (e.g. `ValueError("boom")`) | `None` |
| `traceback` | A traceback object describing where the exception occurred | `None` |

In [32]:
class Inspector:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("exc_type:", exc_type)
        print("exc_value:", exc_value)
        print("traceback is None:", traceback is None)
        return False

try:
    with Inspector():
        raise KeyError("missing_key")
except KeyError:
    pass

exc_type: <class 'KeyError'>
exc_value: 'missing_key'
traceback is None: False


**Output:**
```
exc_type: <class 'KeyError'>
exc_value: 'missing_key'
traceback is None: False
```

## 5.5 Why `__exit__()` Returns a Boolean

`__exit__()`'s return value is used by Python to decide what to do with any exception that occurred in the block:

- A **truthy** return value (most commonly `True`) tells Python: "I have handled this exception - **suppress** it; do not propagate it further."
- A **falsy** return value (most commonly `False`, or simply omitting `return` entirely, which returns `None`) tells Python: "let this exception continue propagating normally, as if this context manager weren't here."

This is covered in complete depth, including when suppression is appropriate, in Part 8 and Part 10.

In [33]:
class SuppressingContext:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is ValueError:
            print("Suppressing this ValueError")
            return True         # exception will NOT propagate
        return False              # anything else propagates normally

with SuppressingContext():
    raise ValueError("this will be silently absorbed")

print("Execution continues here - the exception never escaped the with block")

Suppressing this ValueError
Execution continues here - the exception never escaped the with block


**Output:**
```
Suppressing this ValueError
Execution continues here - the exception never escaped the with block
```

## 5.6 Summary Table

| Method | Called When | Receives | Returns |
|---|---|---|---|
| `__enter__(self)` | At the start of the `with` block | Nothing (besides `self`) | The value bound to `as`, if used |
| `__exit__(self, exc_type, exc_value, traceback)` | At the end of the `with` block, always | Exception info (or `None, None, None`) | A boolean-like value controlling exception suppression |

### Exercises - Section 5

**Beginner**
1. Write a minimal context manager (like Section 5.1) and add print statements inside both `__enter__` and `__exit__` to observe the exact call order relative to the block's body.

**Intermediate**
2. Write a context manager whose `__exit__` prints all three exception parameters, and trigger it with two different exception types to compare the output.

**Advanced**
3. Write a context manager that suppresses `ZeroDivisionError` specifically but lets every other exception type propagate, and test it with both a `ZeroDivisionError` and a `TypeError`.
# 6. Building a Custom Class-Based Context Manager

## 6.1 A Complete, Simple Example

In [34]:
class ManagedFile:
    def __init__(self, filename, mode):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        print(f"Opening {self.filename}")
        self.file = open(self.filename, self.mode)
        return self.file

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Closing {self.filename}")
        if self.file:
            self.file.close()
        return False

with ManagedFile("greeting.txt", "w") as f:
    f.write("Hello, custom context manager!")

Opening greeting.txt
Closing greeting.txt


**Output:**
```
Opening greeting.txt
Closing greeting.txt
```

## 6.2 Every Line, Explained

| Code Line | Explanation |
|---|---|
| `def __init__(self, filename, mode):` | Ordinary initialization (from the OOP reference) - stores the parameters needed later, but does **not** open the file yet. |
| `self.file = None` | A placeholder; the actual file handle doesn't exist until `__enter__` runs. |
| `def __enter__(self):` | Called when the `with` block starts. |
| `self.file = open(self.filename, self.mode)` | The actual **resource acquisition** step - this is where setup happens. |
| `return self.file` | This value is what gets bound to the `as` target - here, the file object itself, not `self`. |
| `def __exit__(self, exc_type, exc_value, traceback):` | Called when the `with` block ends, for any reason. |
| `if self.file: self.file.close()` | The actual **cleanup** step. |
| `return False` | Never suppress exceptions from this context manager - let any error inside the block propagate normally. |

## 6.3 Demonstrating Normal Execution

In [35]:
with ManagedFile("data.txt", "w") as f:
    f.write("line 1\n")
    f.write("line 2\n")
print("File successfully written and closed")

Opening data.txt
Closing data.txt
File successfully written and closed


**Output:**
```
Opening data.txt
Closing data.txt
File successfully written and closed
```

## 6.4 Demonstrating Exception Execution - Cleanup Still Runs

In [36]:
try:
    with ManagedFile("data.txt", "w") as f:
        f.write("line 1\n")
        raise RuntimeError("Something went wrong mid-write")
        f.write("line 2\n")     # never reached
except RuntimeError as e:
    print("Caught outside the with block:", e)

Opening data.txt
Closing data.txt
Caught outside the with block: Something went wrong mid-write


**Output:**
```
Opening data.txt
Closing data.txt
Caught outside the with block: Something went wrong mid-write
```

Notice `"Closing data.txt"` still printed **before** the exception reached the `except` block outside - proving `__exit__()` (and therefore `self.file.close()`) ran even though the block was interrupted by an exception.

## 6.5 A Second Example - Resource With Setup Validation

In [37]:
class DatabaseConnection:
    def __init__(self, host):
        self.host = host
        self.connected = False

    def __enter__(self):
        print(f"Connecting to {self.host}")
        if not self.host:
            raise ValueError("Host cannot be empty")
        self.connected = True
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if self.connected:
            print(f"Disconnecting from {self.host}")
            self.connected = False
        return False

    def query(self, sql):
        if not self.connected:
            raise RuntimeError("Not connected")
        return f"Result of: {sql}"

with DatabaseConnection("db.example.com") as db:
    print(db.query("SELECT * FROM users"))

Connecting to db.example.com
Result of: SELECT * FROM users
Disconnecting from db.example.com


**Output:**
```
Connecting to db.example.com
Result of: SELECT * FROM users
Disconnecting from db.example.com
```

**Important detail:** if `__enter__()` itself raises an exception (as it would for `DatabaseConnection("")`), `__exit__()` is **never called** - there is nothing to clean up yet, since the context was never successfully entered.

In [38]:
try:
    with DatabaseConnection("") as db:
        print("This never runs")
except ValueError as e:
    print("Error during __enter__:", e)

Connecting to 
Error during __enter__: Host cannot be empty


**Output:**
```
Connecting to
Error during __enter__: Host cannot be empty
```

## 6.6 Summary Table

| Step | Method | Purpose |
|---|---|---|
| 1. Setup | `__enter__` | Acquire the resource; return the value to be used inside the block |
| 2. Use | (the `with` block body) | Work with the acquired resource |
| 3. Cleanup | `__exit__` | Release the resource, regardless of success or failure |
| Special case | If `__enter__` raises | `__exit__` is never called - nothing was successfully acquired |

### Exercises - Section 6

**Beginner**
1. Write a `ManagedFile`-style context manager for opening and closing a file, but add a print statement in `__init__` too, and observe exactly when it prints relative to `__enter__`.

**Intermediate**
2. Extend `DatabaseConnection` with a `transaction()` method that itself returns a nested context manager (preview of Part 22).

**Advanced**
3. Write a context manager whose `__enter__` can fail (raise an exception) under some condition, and prove with a print statement that `__exit__` is never called in that specific case.
# 7. `__enter__()` in Detail

## 7.1 What Happens When Entering a Context

`__enter__()` runs synchronously, before the `with` block's body, and whatever it returns becomes available to the block (if an `as` clause captures it). `__enter__()` is where resource acquisition, validation, or any "opening" logic belongs.

In [39]:
class Timer:
    def __enter__(self):
        import time
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        import time
        elapsed = time.perf_counter() - self._start
        print(f"Elapsed: {elapsed:.4f} seconds")
        return False

with Timer():
    total = sum(range(1_000_000))

Elapsed: 0.0125 seconds


**Output:**
```
Elapsed: 0.0123 seconds
```

## 7.2 Returning `self`

The most common pattern: `__enter__()` returns `self`, giving the block access to the full context manager object, including any of its own methods.

In [40]:
class APIClient:
    def __enter__(self):
        print("Connecting to API")
        return self          # returning self - block gets the whole client object

    def __exit__(self, exc_type, exc_value, traceback):
        print("Closing API connection")
        return False

    def get(self, endpoint):
        return f"GET {endpoint} -> 200 OK"

with APIClient() as client:
    print(client.get("/users"))       # 'client' has all of APIClient's methods available

Connecting to API
GET /users -> 200 OK
Closing API connection


**Output:**
```
Connecting to API
GET /users -> 200 OK
Closing API connection
```

## 7.3 Returning Another Object

`__enter__()` does not have to return `self` - it can return an entirely different object, which is often more appropriate when the context manager's job is purely to manage the lifecycle of *something else*.

In [41]:
class FileOpener:
    def __init__(self, path, mode):
        self.path = path
        self.mode = mode
        self._file = None

    def __enter__(self):
        self._file = open(self.path, self.mode)
        return self._file         # returns the FILE, not the FileOpener instance itself

    def __exit__(self, exc_type, exc_value, traceback):
        self._file.close()
        return False

with FileOpener("output.txt", "w") as f:
    print(type(f).__name__)          # 'f' is a plain file object, not a FileOpener
    f.write("data")

TextIOWrapper


**Output:**
```
TextIOWrapper
```

## 7.4 Returning Nothing (`None`)

If `__enter__()` has no `return` statement (or explicitly `return None`), the `as` target (if used) will simply be `None` - this is appropriate when the block doesn't need a handle to anything, only the guarantee that setup/cleanup will happen.

In [42]:
class Lock:
    def __enter__(self):
        print("Acquiring lock")
        # no return statement - implicitly returns None

    def __exit__(self, exc_type, exc_value, traceback):
        print("Releasing lock")
        return False

with Lock() as result:
    print("result is:", result)

Acquiring lock
result is: None
Releasing lock


**Output:**
```
Acquiring lock
result is: None
Releasing lock
```

## 7.5 Comparing `with MyContext():` and `with MyContext() as resource:`

In [43]:
class Demo:
    def __enter__(self):
        print("Entering")
        return "some value"

    def __exit__(self, exc_type, exc_value, traceback):
        print("Exiting")
        return False

# Without 'as' - the return value of __enter__ is simply discarded
with Demo():
    print("Using the context, but not the returned value")

# With 'as' - the return value of __enter__ is captured and usable
with Demo() as resource:
    print(f"Using the context, with access to: {resource}")

Entering
Using the context, but not the returned value
Exiting
Entering
Using the context, with access to: some value
Exiting


**Output:**
```
Entering
Using the context, but not the returned value
Exiting
Entering
Using the context, with access to: some value
Exiting
```

Both forms enter and exit the context identically - the only difference is whether the code inside the block has a name to refer to the value `__enter__()` produced. Use `with Ctx():` (no `as`) when the block only needs the side effects of entering/exiting (e.g. holding a lock), and `with Ctx() as x:` when the block needs to actually use something the context manager provides (a file, a connection, a client).

## 7.6 Summary Table

| Pattern | When to Use |
|---|---|
| `return self` | The block needs access to the context manager's own methods/attributes |
| `return other_object` | The context manager's job is to manage a *different* object's lifecycle |
| `return None` (or no return) | The block only needs the setup/cleanup side effect, not a handle to anything |
| `with Ctx():` (no `as`) | The returned value isn't needed inside the block |
| `with Ctx() as x:` | The block needs to use whatever `__enter__()` provides |

### Exercises - Section 7

**Beginner**
1. Write a context manager whose `__enter__` returns `self`, and one whose `__enter__` returns a different, unrelated object, and compare using both with `as`.

**Intermediate**
2. Write a `with` statement without `as` for a context manager that returns a useful value from `__enter__`, and explain what information is lost by omitting `as`.

**Advanced**
3. Design a context manager where returning `self` from `__enter__` would be a mistake, and explain specifically why returning a different object is the better design there.
# 8. `__exit__()` in Detail

## 8.1 The Full Signature

In [44]:
def __exit__(self, exc_type, exc_value, traceback):
    ...

Every `__exit__` method must accept exactly these three parameters (beyond `self`) - Python always calls it with exactly three arguments, using `None, None, None` when no exception occurred.

## 8.2 `exc_type`, `exc_value`, `traceback` - Precisely

| Parameter | Type | Description |
|---|---|---|
| `exc_type` | A class (or `None`) | The exception class that was raised, e.g. `ValueError`, `KeyError` |
| `exc_value` | An exception instance (or `None`) | The actual exception object, e.g. `ValueError("bad input")` |
| `traceback` | A traceback object (or `None`) | Describes the call stack at the point the exception was raised; rarely inspected directly, but available for advanced logging/debugging |

In [45]:
class Inspector:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"type: {exc_type}")
        print(f"value: {exc_value}")
        print(f"traceback line: {traceback.tb_lineno if traceback else None}")
        return False

try:
    with Inspector():
        raise ValueError("test error")
except ValueError:
    pass

type: <class 'ValueError'>
value: test error
traceback line: 13


**Output:**
```
type: <class 'ValueError'>
value: test error
traceback line: 44
```

## 8.3 Normal Execution - All Three Parameters Are `None`

In [46]:
class Inspector:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Exception occurred:", exc_type is not None)
        return False

with Inspector():
    print("No error here")

No error here
Exception occurred: False


**Output:**
```
No error here
Exception occurred: False
```

## 8.4 Exception Execution - Parameters Are Populated

In [47]:
with Inspector():
    print("About to raise")
    raise KeyError("oops")

About to raise
Exception occurred: True


KeyError: 'oops'

**Output:**
```
About to raise
Exception occurred: True
```

```
Traceback (most recent call last):
  ...
KeyError: 'oops'
```

(The exception still propagates after `__exit__` runs, because this `Inspector.__exit__` returns `False`.)

## 8.5 Cleanup - Runs Regardless

Cleanup logic in `__exit__` should generally run unconditionally, whether or not an exception occurred, since the resource needs releasing either way.

In [48]:
class Resource:
    def __enter__(self):
        print("Acquired")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Released")     # this line runs whether or not an exception occurred
        return False

with Resource():
    pass

try:
    with Resource():
        raise RuntimeError("boom")
except RuntimeError:
    pass

Acquired
Released
Acquired
Released


**Output:**
```
Acquired
Released
Acquired
Released
```

## 8.6 Returning `None`

Returning `None` (or simply omitting a `return` statement, which is the same thing) is treated as **falsy**, so any exception continues propagating normally. This is the correct default for the overwhelming majority of context managers, since silently absorbing errors is rarely appropriate (Part 10).

In [49]:
class Simple:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        pass          # implicitly returns None

try:
    with Simple():
        raise ValueError("propagates as normal")
except ValueError as e:
    print("Caught:", e)

Caught: propagates as normal


**Output:**
```
Caught: propagates as normal
```

## 8.7 Returning `False`

Identical in effect to returning `None` - an explicit `False` is simply a more readable way of saying "do not suppress this exception," which is considered better style than relying on an implicit `None`.

In [50]:
class Explicit:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return False       # explicit, readable "never suppress" - preferred style

try:
    with Explicit():
        raise ValueError("still propagates")
except ValueError as e:
    print("Caught:", e)

Caught: still propagates


**Output:**
```
Caught: still propagates
```

## 8.8 Returning `True` - Precisely How Suppression Works

Returning `True` (or any truthy value) tells Python: "I have handled this exception myself - do not let it propagate any further." The exception simply disappears; execution resumes normally immediately after the `with` block, as if no exception had happened at all.

In [51]:
class Suppressor:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Suppressing {exc_type.__name__ if exc_type else 'nothing'}")
        return True        # ALWAYS suppress - use with extreme caution (see Part 10)

with Suppressor():
    raise ValueError("this disappears completely")

print("This line runs normally - the exception never escaped the with block")

Suppressing ValueError
This line runs normally - the exception never escaped the with block


**Output:**
```
Suppressing ValueError
This line runs normally - the exception never escaped the with block
```

**Critical precision:** suppression only ever applies to exceptions raised **inside the `with` block itself**. It has no effect at all when no exception occurred (there's nothing to suppress), and it cannot somehow "catch" an exception raised outside the block.

## 8.9 Summary Table

| Return Value | Effect on an Exception Raised in the Block |
|---|---|
| `None` (implicit, no `return`) | Propagates normally (not suppressed) |
| `False` | Propagates normally (not suppressed) - explicit and preferred over implicit `None` |
| `True` (or any truthy value) | **Suppressed** - the exception is discarded; execution continues after the block |
| (No exception occurred) | Return value is irrelevant - there's nothing to suppress |

### Exercises - Section 8

**Beginner**
1. Write a context manager whose `__exit__` prints all three parameters, then trigger it with `ZeroDivisionError`, `TypeError`, and no exception at all, comparing the three outputs.

**Intermediate**
2. Write a context manager that returns `True` from `__exit__` only for `FileNotFoundError`, and demonstrate that other exception types still propagate.

**Advanced**
3. Explain, in your own words, why returning `True` unconditionally from `__exit__` (as in Section 8.8) is almost always the wrong choice in real code, previewing the concerns covered fully in Part 10.
# 9. Context Managers and Exceptions

## 9.1 The Relationship Between `with`, `try`, `except`, and `finally`

A context manager's `__enter__`/`__exit__` pair is conceptually equivalent to a `try/finally` block, with `__exit__` additionally getting the option to inspect (and optionally suppress) any exception, which plain `finally` cannot do.

In [52]:
class SomeResource:
    def __enter__(self):
        print("Setup")
        return "resource_value"
    def __exit__(self, exc_type, exc_value, traceback):
        print("Cleanup")
        return False

def do_work(r):
    print(f"Working with {r}")

# Conceptually equivalent forms:

# Form 1: context manager
with SomeResource() as r:
    do_work(r)

# Form 2: manual try/finally (cannot inspect or suppress exceptions)
r_manager = SomeResource()
r = r_manager.__enter__()
try:
    do_work(r)
finally:
    r_manager.__exit__(None, None, None)   # simplified - real __exit__ gets actual exception info

Setup
Working with resource_value
Cleanup
Setup
Working with resource_value
Cleanup


**Output:**
```
Setup
Working with resource_value
Cleanup
Setup
Working with resource_value
Cleanup
```

## 9.2 A More Precise Equivalent, Including Exception Info

In [53]:
import sys

cm = SomeResource()
value = cm.__enter__()
exc_info = (None, None, None)
try:
    do_work(value)
except BaseException:
    exc_info = sys.exc_info()
    suppress = cm.__exit__(*exc_info)
    if not suppress:
        raise
else:
    cm.__exit__(None, None, None)

Setup
Working with resource_value
Cleanup


**Output:**
```
Setup
Working with resource_value
Cleanup
```

This is close to what Python does internally (Part 4), and it makes clear exactly why context managers are strictly more capable than plain `finally`: `finally` always runs, but it has no mechanism to inspect *which* exception occurred or to decide whether to suppress it - that decision-making capability is unique to `__exit__`'s return value.

## 9.3 Why Context Managers Are Especially Useful for Guaranteed Cleanup

The core guarantee context managers provide, precisely stated: **`__exit__()` is called exactly once, no matter how the block terminates** - whether it completes normally, raises any exception, or exits early via `return`, `break`, or `continue` (when the `with` block is inside a function or loop).

In [54]:
class Resource:
    def __enter__(self):
        print("Acquired")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Released")
        return False

def function_with_early_return():
    with Resource():
        print("Doing work")
        return "early exit"       # __exit__ STILL runs before the function actually returns
    print("This line never runs")

result = function_with_early_return()
print("Returned:", result)

Acquired
Doing work
Released
Returned: early exit


**Output:**
```
Acquired
Doing work
Released
Returned: early exit
```

In [55]:
for i in range(3):
    with Resource():
        if i == 1:
            break                  # __exit__ still runs before the loop actually breaks
        print(f"Iteration {i}")

Acquired
Iteration 0
Released
Acquired
Released


**Output:**
```
Acquired
Iteration 0
Released
Acquired
Released
```

## 9.4 Multiple Exceptions - What Happens if `__exit__` Itself Raises

If `__exit__()` raises its own new exception, that new exception **replaces** the original one (the original becomes available via the new exception's `__context__` attribute, following Python's standard exception-chaining behavior).

In [56]:
class DoubleTrouble:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        raise RuntimeError("Cleanup itself failed!")

try:
    with DoubleTrouble():
        raise ValueError("Original error")
except RuntimeError as e:
    print("Caught:", e)
    print("Original error was:", e.__context__)

Caught: Cleanup itself failed!
Original error was: Original error


**Output:**
```
Caught: Cleanup itself failed!
Original error was: Original error
```

**Best practice:** cleanup code inside `__exit__` should itself be defensive (e.g. wrapped in its own `try/except` if it might fail) so that a cleanup failure doesn't silently mask the original, often more important, exception.

## 9.5 Summary Table

| Concept | `try/finally` Alone | Context Manager (`with`) |
|---|---|---|
| Guaranteed cleanup | Yes | Yes |
| Can inspect the exception | No (not directly in `finally`) | Yes, via `exc_type`/`exc_value`/`traceback` |
| Can suppress the exception | No | Yes, via `__exit__`'s return value |
| Reusable setup/cleanup logic | Must duplicate `try/finally` at every call site | Encapsulated once, reused via `with` anywhere |
| Works with `return`/`break`/`continue` inside the block | Yes | Yes |

### Exercises - Section 9

**Beginner**
1. Write a function with a `with` block containing an early `return`, and confirm via print statements that `__exit__` still runs before the function actually returns.

**Intermediate**
2. Reproduce the "exception replaces exception" behavior from Section 9.4 with your own exception types, and inspect `__context__` to find the original error.

**Advanced**
3. Write a context manager whose `__exit__` performs cleanup that might itself fail, and wrap that cleanup in its own `try/except` so a cleanup failure never masks the original exception - demonstrate both succeeding and failing cleanup scenarios.
# 10. Suppressing Exceptions

## 10.1 When `__exit__()` Can Suppress an Exception

Any time `__exit__()` returns a truthy value, the exception that was active when `__exit__()` was called is discarded, and execution resumes immediately after the `with` block, as though the exception never happened.

In [57]:
class IgnoreErrors:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return True       # unconditionally suppress EVERYTHING

with IgnoreErrors():
    raise ValueError("gone without a trace")

print("Reached here - the exception was fully suppressed")

Reached here - the exception was fully suppressed

**Output:**
```
Reached here - the exception was fully suppressed
```

## 10.2 Selective Suppression - The Responsible Way

Suppression should almost always be **conditional**, based on the specific exception type, rather than blanket-suppressing everything.

In [58]:
class SuppressSpecific:
    def __init__(self, *exception_types):
        self.exception_types = exception_types

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is not None and issubclass(exc_type, self.exception_types):
            print(f"Suppressing expected {exc_type.__name__}")
            return True
        return False       # anything unexpected still propagates

with SuppressSpecific(FileNotFoundError):
    open("this_file_does_not_exist.txt")

print("Continued past a FileNotFoundError")

try:
    with SuppressSpecific(FileNotFoundError):
        raise ValueError("an unrelated, unexpected error")
except ValueError as e:
    print("Correctly NOT suppressed:", e)

Suppressing expected FileNotFoundError
Continued past a FileNotFoundError
Correctly NOT suppressed: an unrelated, unexpected error


**Output:**
```
Suppressing expected FileNotFoundError
Continued past a FileNotFoundError
Correctly NOT suppressed: an unrelated, unexpected error
```

## 10.3 When Suppression Is Appropriate

- The specific exception represents a genuinely **expected, recoverable** condition, and the calling code has a clear, correct fallback behavior for it (e.g., a cache-miss lookup that's expected to sometimes fail, or an optional cleanup step that may legitimately not apply).
- The exception type being suppressed is **narrow and explicit** - never suppress `Exception` or `BaseException` broadly.
- The suppression is a deliberate, documented part of the context manager's contract, not an accident of sloppy exception handling.

In [59]:
from contextlib import suppress    # covered fully in Part 15

# A GOOD use of suppression: deleting a file that may or may not exist is a common,
# genuinely expected case, and FileNotFoundError specifically is the right thing to ignore
import os
with suppress(FileNotFoundError):
    os.remove("maybe_does_not_exist.txt")
print("Cleanup attempted - missing file is not an error condition here")

Cleanup attempted - missing file is not an error condition here


**Output:**
```
Cleanup attempted - missing file is not an error condition here
```

## 10.4 When Suppression Is Dangerous

- **Blanket suppression** (`return True` unconditionally, or catching bare `Exception`) hides real bugs - a `TypeError` from a genuine programming mistake gets silently swallowed exactly like an expected, recoverable condition, making bugs far harder to detect and diagnose.
- Suppression that isn't clearly **documented** creates confusing, surprising behavior for anyone else using the context manager, who reasonably expects exceptions to propagate unless told otherwise.
- Suppressing exceptions inside **critical, correctness-sensitive code** (financial transactions, data validation, safety checks) can allow the program to continue operating on invalid or inconsistent state.

In [60]:
# DANGEROUS - unconditionally hides every possible error, including real bugs
class TooPermissive:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return True     # EVERYTHING silently disappears - a real bug here would never surface

with TooPermissive():
    result = 10 / 0            # ZeroDivisionError - a genuine bug - completely hidden
    print("This never prints, but no error is ever seen either")

print("Program continues, unaware anything went wrong at all")

Program continues, unaware anything went wrong at all


**Output:**
```
Program continues, unaware anything went wrong at all
```

## 10.5 Good vs Bad Exception Suppression - Side by Side

In [61]:
# GOOD - narrow, documented, genuinely expected condition
class SafeCacheLookup:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is KeyError:
            print("Cache miss - falling back to default (expected, documented behavior)")
            return True
        return False        # any other error is a real bug and should propagate

# BAD - broad, undocumented, hides everything including real programming errors
class UnsafeCatchAll:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return True         # no condition at all - ANY exception vanishes silently

## 10.6 Why Silently Hiding Errors Is Bad Practice

Silently swallowed exceptions are among the hardest bugs to diagnose in production systems: the program appears to run "successfully," logs show no errors, and yet the actual computed results are wrong, missing, or based on corrupted state - because the underlying failure that should have stopped execution was quietly discarded several layers below where anyone is looking.

## 10.7 Summary Table

| Practice | Verdict |
|---|---|
| Suppress a specific, narrow, expected exception type | Good - clear, documented, safe |
| Suppress based on runtime conditions plus exception type | Good - even more precise |
| Suppress `Exception` or `BaseException` broadly | Bad - hides real bugs indiscriminately |
| Suppress without clear documentation of the behavior | Bad - confusing and unpredictable for callers |
| Suppress in correctness-critical code paths | Dangerous - risks corrupted or invalid downstream state |

### Exercises - Section 10

**Beginner**
1. Write a context manager that suppresses only `ZeroDivisionError`, and test it against both a `ZeroDivisionError` and a `ValueError`.

**Intermediate**
2. Take Section 10.4's `TooPermissive` example and rewrite it to only suppress `ZeroDivisionError`, demonstrating that other bugs (like a `TypeError`) now correctly surface.

**Advanced**
3. Design a scenario (in comments or prose) where suppressing an exception seems tempting but is actually dangerous, and explain what the safer alternative design would be.
# 11. `contextlib`

## 11.1 Why the Standard Library Provides `contextlib`

Writing a full class with `__init__`, `__enter__`, and `__exit__` is sometimes more ceremony than a simple setup/cleanup pattern actually needs. The `contextlib` module, part of Python's standard library, provides ready-made tools that cover the most common context-manager needs, plus a much more concise way to write simple custom context managers using generator functions instead of full classes.

In [62]:
import contextlib
print([name for name in dir(contextlib) if not name.startswith("_")])

['AbstractAsyncContextManager', 'AbstractContextManager', 'AsyncContextDecorator', 'AsyncExitStack', 'ContextDecorator', 'ExitStack', 'GenericAlias', 'MethodType', 'abc', 'aclosing', 'asynccontextmanager', 'chdir', 'closing', 'contextmanager', 'deque', 'nullcontext', 'os', 'redirect_stderr', 'redirect_stdout', 'suppress', 'sys', 'wraps']


**Output:**
```
['AbstractAsyncContextManager', 'AbstractContextManager', 'AsyncContextDecorator', 'AsyncExitStack', 'ContextDecorator', 'ExitStack', 'asynccontextmanager', 'chdir', 'closing', 'contextmanager', 'nullcontext', 'redirect_stderr', 'redirect_stdout', 'suppress']
```

## 11.2 The Major Utilities at a Glance

| Utility | Purpose | Covered In |
|---|---|---|
| `@contextmanager` | Write a context manager as a generator function instead of a class | Part 12, 13 |
| `closing()` | Turn any object with a `.close()` method into a context manager | Part 14 |
| `suppress()` | A context-manager form of "ignore these specific exception types" | Part 15 |
| `redirect_stdout()` / `redirect_stderr()` | Temporarily redirect printed output | Part 16 |
| `nullcontext()` | A "do-nothing" context manager, useful for conditional context usage | Part 17 |
| `ExitStack` | Dynamically manage a variable number of context managers | Part 18 |
| `asynccontextmanager` | The `async` equivalent of `@contextmanager` | Part 29 |
| `AbstractContextManager` | A base class formalizing the sync context manager protocol (useful for `isinstance` checks or explicit inheritance) | (mentioned here) |

## 11.3 A Quick Preview

In [63]:
from contextlib import contextmanager

@contextmanager
def timer():
    import time
    start = time.perf_counter()
    yield
    print(f"Elapsed: {time.perf_counter() - start:.4f}s")

with timer():
    sum(range(1_000_000))

Elapsed: 0.0117s


**Output:**
```
Elapsed: 0.0119s
```

This single generator function does the work that would otherwise require a full class with `__init__`, `__enter__`, and `__exit__` - the full mechanics of `@contextmanager` are covered in depth in Part 12.

## 11.4 Why Prefer `contextlib` Utilities When Applicable

- **Less boilerplate** - a one-line `contextlib.closing(obj)` replaces an entire small class, for the common "just call `.close()` at the end" pattern.
- **Well-tested, standard behavior** - these utilities handle edge cases (like exceptions during setup or cleanup) correctly and consistently, reducing the chance of a subtle bug in a hand-rolled context manager.
- **Readability** - code reviewers and other Python developers instantly recognize `contextlib.suppress(...)` or `@contextmanager` and understand the intent immediately, without needing to read a custom class's implementation.

### Exercises - Section 11

**Beginner**
1. Run `dir(contextlib)` yourself and identify which utilities you've never heard of before this guide.

**Intermediate**
2. Match each utility in Section 11.2's table to a real scenario from your own code where it could replace a hand-written class-based context manager.

**Advanced**
3. Explain, in your own words, the tradeoff between using a `contextlib` utility versus writing a custom class-based context manager, referencing readability and control.
# 12. `@contextmanager`

## 12.1 Creating a Context Manager from a Generator Function

`contextlib.contextmanager` is a decorator that turns an ordinary generator function (one that uses `yield` exactly once) into a full context manager, without needing to write a class at all.

In [64]:
from contextlib import contextmanager

@contextmanager
def my_context():
    print("Setup")
    yield "the resource"
    print("Cleanup")

with my_context() as resource:
    print(f"Using {resource}")

Setup
Using the resource
Cleanup


**Output:**
```
Setup
Using the resource
Cleanup
```

## 12.2 Why `yield` Is Used

The single `yield` statement is the dividing line between setup and cleanup: everything **before** `yield` runs during `__enter__`, and everything **after** `yield` runs during `__exit__`. The value passed to `yield` becomes the value bound to the `as` target, exactly like a class-based context manager's `__enter__()` return value.

In [65]:
@contextmanager
def resource_manager():
    print("[before yield] acquiring resource")   # this is the "__enter__" phase
    resource = "resource_handle"
    yield resource                                  # this value becomes the 'as' target
    print("[after yield] releasing resource")      # this is the "__exit__" phase

with resource_manager() as r:
    print(f"[inside with block] using {r}")

[before yield] acquiring resource


[inside with block] using resource_handle
[after yield] releasing resource


**Output:**
```
[before yield] acquiring resource
[inside with block] using resource_handle
[after yield] releasing resource
```

## 12.3 Code Before `yield` - Setup

Everything before `yield` runs exactly once, at the moment `__enter__()` would be called - this is where resource acquisition belongs.

## 12.4 Code After `yield` - Cleanup

Everything after `yield` runs exactly once, at the moment `__exit__()` would be called - after the `with` block's body has finished (normally or via exception, see Part 13 for the exception-handling nuances).

## 12.5 How This Actually Works - The Generator's Perspective

Calling a generator function does **not** run any of its code immediately - it returns a generator object. `@contextmanager` wraps this generator in a small helper class whose `__enter__()` calls `next()` on the generator (running everything up to `yield`), and whose `__exit__()` resumes the generator (running everything after `yield`), by calling `next()` again (or `.throw()` if an exception needs to be delivered into the generator, covered in Part 13).

In [66]:
@contextmanager
def demo():
    print("before yield")
    yield "value"
    print("after yield")

gen_context = demo()          # nothing has printed yet - the generator hasn't started running
print("Context manager created, but not entered yet")

with gen_context as v:
    print(f"inside block, v = {v}")

Context manager created, but not entered yet
before yield
inside block, v = value
after yield


**Output:**
```
Context manager created, but not entered yet
before yield
inside block, v = value
after yield
```

## 12.6 Class-Based vs `@contextmanager`-Based - Side by Side

In [67]:
# CLASS-BASED
class Timer:
    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        import time
        print(f"Elapsed: {time.perf_counter() - self.start:.4f}s")
        return False

# @contextmanager-BASED - same behavior, far less code
from contextlib import contextmanager
import time

@contextmanager
def timer():
    start = time.perf_counter()
    yield
    print(f"Elapsed: {time.perf_counter() - start:.4f}s")

with Timer():
    sum(range(100_000))

with timer():
    sum(range(100_000))

Elapsed: 0.0012s
Elapsed: 0.0012s


**Output:**
```
Elapsed: 0.0031s
Elapsed: 0.0029s
```

## 12.7 Comparison Table

| Aspect | Class-Based | `@contextmanager`-Based |
|---|---|---|
| Boilerplate | More - requires `__init__`, `__enter__`, `__exit__` | Less - a single generator function |
| State storage | Instance attributes (`self.x`) | Local variables in the function's scope |
| Readability for simple cases | More verbose | Very concise |
| Readability for complex cases (multiple entry points, reentrant use, subclassing) | Often clearer - explicit methods, supports inheritance | Can become awkward - a single linear function body |
| Exception handling inside cleanup | Explicit `exc_type`/`exc_value`/`traceback` parameters | Requires `try/except`/`finally` around `yield` (Part 13) |
| Best suited for | Complex, reusable, or subclassable context managers | Simple, self-contained setup/cleanup logic |

## 12.8 Summary Table

| Concept | Class-Based Equivalent |
|---|---|
| Code before `yield` | `__enter__()` |
| Value passed to `yield` | `__enter__()`'s return value |
| Code after `yield` | `__exit__()` (the "no exception" path) |
| Decorator | `@contextlib.contextmanager` |
| Underlying requirement | The function must be a generator (must contain exactly one `yield` on the normal path) |

### Exercises - Section 12

**Beginner**
1. Rewrite `ManagedFile` from Part 6 as a `@contextmanager`-decorated generator function.

**Intermediate**
2. Write a `@contextmanager` function that measures and prints both wall-clock time and a simple call counter across multiple uses.

**Advanced**
3. Convert a class-based context manager with instance state (e.g. `DatabaseConnection` from Part 6) into `@contextmanager` form, and discuss which version reads more clearly for that specific use case.
# 13. `@contextmanager` with Exception Handling

## 13.1 What Happens to an Exception Raised Inside the `with` Block

When an exception occurs inside a `@contextmanager`-based context manager's `with` block, Python delivers that exception **into the generator, at the point of `yield`** - as if the `yield` statement itself had raised it. This means code after `yield` will **not** run unless it's protected by `try/finally`.

In [68]:
from contextlib import contextmanager

@contextmanager
def broken_cleanup():
    print("Setup")
    yield
    print("Cleanup - THIS WILL BE SKIPPED IF AN EXCEPTION OCCURS")

try:
    with broken_cleanup():
        raise ValueError("boom")
except ValueError as e:
    print("Caught:", e)

Setup
Caught: boom


**Output:**
```
Setup
Caught: boom
```

Notice `"Cleanup..."` never printed - the exception raised at `yield` propagated straight out of the generator function, skipping every line after `yield`.

## 13.2 The Fix: `try`/`finally` Around `yield`

In [69]:
@contextmanager
def safe_cleanup():
    print("Setup")
    try:
        yield
    finally:
        print("Cleanup - ALWAYS RUNS NOW")

try:
    with safe_cleanup():
        raise ValueError("boom")
except ValueError as e:
    print("Caught:", e)

Setup
Cleanup - ALWAYS RUNS NOW
Caught: boom


**Output:**
```
Setup
Cleanup - ALWAYS RUNS NOW
Caught: boom
```

## 13.3 Why `finally` Is Frequently Essential Here

This mirrors exactly the same lesson from Part 3 and Part 9 about manual resource cleanup: **any code after `yield` that represents genuine cleanup (closing a resource, releasing a lock, rolling back a transaction) must be wrapped in `try/finally`**, or it will be silently skipped whenever the block raises an exception - precisely the scenario where cleanup matters most.

In [70]:
@contextmanager
def acquire_lock(lock):
    lock.acquire()
    try:
        yield
    finally:
        lock.release()          # GUARANTEED to run, even if the block raises

import threading
lock = threading.Lock()

try:
    with acquire_lock(lock):
        raise RuntimeError("failure during critical section")
except RuntimeError:
    pass

print("Lock released:", not lock.locked())

Lock released: True


**Output:**
```
Lock released: True
```

## 13.4 Suppressing Exceptions in a `@contextmanager` Function

To suppress an exception (the generator-based equivalent of returning `True` from `__exit__`), catch the specific exception type around `yield` and simply **do not re-raise it**.

In [71]:
@contextmanager
def suppress_zero_division():
    try:
        yield
    except ZeroDivisionError:
        print("Suppressing ZeroDivisionError")
        # not re-raising - this suppresses the exception, equivalent to __exit__ returning True

with suppress_zero_division():
    result = 10 / 0

print("Execution continues normally")

Suppressing ZeroDivisionError
Execution continues normally


**Output:**
```
Suppressing ZeroDivisionError
Execution continues normally
```

**Precisely how this maps to the class-based protocol:** if the `except` block does **not** re-raise, `@contextmanager`'s internal machinery treats this exactly like `__exit__()` returning `True`. If the `except` block **does** re-raise (or lets a different exception propagate), that's equivalent to `__exit__()` returning `False` (or letting a different exception take over, exactly as in Part 9.4).

In [72]:
@contextmanager
def selective_suppress():
    try:
        yield
    except ZeroDivisionError:
        print("Only suppressing ZeroDivisionError specifically")

try:
    with selective_suppress():
        raise ValueError("not a ZeroDivisionError - should propagate")
except ValueError as e:
    print("Correctly NOT suppressed:", e)

Correctly NOT suppressed: not a ZeroDivisionError - should propagate


**Output:**
```
Correctly NOT suppressed: not a ZeroDivisionError
```

## 13.5 Inspecting the Exception Inside the Generator

Because the exception is delivered directly at the `yield` point, standard Python exception-handling syntax gives full access to it, just like anywhere else in the code.

In [73]:
@contextmanager
def logging_context(name):
    print(f"Entering {name}")
    try:
        yield
    except Exception as e:
        print(f"[{name}] An error occurred: {type(e).__name__}: {e}")
        raise           # re-raise after logging - do NOT silently swallow it
    finally:
        print(f"Exiting {name}")

try:
    with logging_context("data_import"):
        raise ValueError("malformed row")
except ValueError:
    pass

Entering data_import
[data_import] An error occurred: ValueError: malformed row
Exiting data_import


**Output:**
```
Entering data_import
[data_import] An error occurred: ValueError: malformed row
Exiting data_import
```

## 13.6 Summary Table

| Pattern | Effect |
|---|---|
| `yield` with no surrounding `try` | Cleanup code after `yield` is **skipped** if an exception occurs |
| `try: yield` `finally: cleanup()` | Cleanup **always** runs, exception still propagates afterward |
| `try: yield` `except SomeError: handle()` (no re-raise) | Exception of that type is **suppressed** |
| `try: yield` `except SomeError: ... raise` | Exception is logged/handled, then still propagates |

### Exercises - Section 13

**Beginner**
1. Reproduce the "skipped cleanup" bug from Section 13.1 with your own generator-based context manager, then fix it with `try/finally`.

**Intermediate**
2. Write a `@contextmanager` function that suppresses `KeyError` specifically but re-raises everything else, and test both cases.

**Advanced**
3. Write a `@contextmanager`-based database transaction helper that commits on success and rolls back on any exception, using `try/except/else` around `yield` (hint: `else` runs only if no exception occurred).
# 14. `closing()`

## 14.1 What `contextlib.closing()` Does

`closing(obj)` wraps any object that has a `.close()` method, turning it into a context manager that calls `obj.close()` automatically when the block ends - for objects that support the "closeable" pattern but don't implement the full context manager protocol themselves.

In [74]:
from contextlib import closing

class NetworkConnection:
    def __init__(self, host):
        self.host = host
        print(f"Connected to {host}")

    def close(self):
        print(f"Connection to {self.host} closed")

    def fetch(self):
        return "some data"

with closing(NetworkConnection("api.example.com")) as conn:
    print(conn.fetch())

Connected to api.example.com
some data
Connection to api.example.com closed


**Output:**
```
Connected to api.example.com
some data
Connection to api.example.com closed
```

## 14.2 When It Is Useful

Many objects, especially in older libraries or simpler custom classes, provide a `.close()` method without implementing `__enter__`/`__exit__` themselves. `closing()` is the standard, idiomatic way to make such objects usable with `with`, instead of writing a custom wrapper class just for this purpose.

In [75]:
try:
    NetworkConnection("test").__enter__()   # this object has no __enter__ at all
except AttributeError as e:
    print("Error:", e)

Connected to test
Error: 'NetworkConnection' object has no attribute '__enter__'


**Output:**
```
Connected to test
Error: 'NetworkConnection' object has no attribute '__enter__'
```

In [76]:
# closing() bridges exactly this gap
from contextlib import closing
with closing(NetworkConnection("test")) as conn:
    pass

Connected to test
Connection to test closed


**Output:**
```
Connected to test
Connection to test closed
```

## 14.3 Objects With a `.close()` Method - Practical Examples

In [77]:
from contextlib import closing
import io

# Standing in for urllib.request.urlopen(...) so this example runs without network access;
# real code would use: with closing(urlopen("https://example.com")) as response:
def fake_urlopen(url):
    return io.BytesIO(b"<html><body>Example Domain</body></html>")

# urlopen() returns an object with .close() but historically not full context-manager support
# in some contexts; closing() guarantees correct cleanup regardless
with closing(fake_urlopen("https://example.com")) as response:
    html = response.read(100)
    print(len(html), "bytes read")

40 bytes read


**Output:**
```
40 bytes read
```

In [78]:
# A custom resource class that only defines close(), not the full protocol
class ModelCache:
    def __init__(self):
        self.loaded = True
        print("Model loaded into memory")

    def close(self):
        self.loaded = False
        print("Model unloaded from memory")

with closing(ModelCache()) as cache:
    print("Cache loaded:", cache.loaded)
print("Cache still loaded:", cache.loaded)

Model loaded into memory
Cache loaded: True
Model unloaded from memory
Cache still loaded: False


**Output:**
```
Model loaded into memory
Cache loaded: True
Model unloaded from memory
Cache still loaded: False
```

## 14.4 How `closing()` Works Internally (Conceptually)

`closing()` is itself implemented as a tiny `@contextmanager` generator function, roughly equivalent to:

In [79]:
from contextlib import contextmanager

@contextmanager
def my_closing(thing):
    try:
        yield thing
    finally:
        thing.close()      # guaranteed cleanup, exactly like Part 13's lessons

class Demo:
    def close(self):
        print("Closed")

with my_closing(Demo()):
    print("Using the object")

Using the object
Closed


**Output:**
```
Using the object
Closed
```

## 14.5 Summary Table

| Aspect | Detail |
|---|---|
| Import | `from contextlib import closing` |
| Purpose | Wrap any object with a `.close()` method into a context manager |
| Returns from `__enter__` | The wrapped object itself |
| Cleanup action | Calls `wrapped_object.close()` |
| When to use | The object supports `.close()` but not the full context manager protocol |
| When NOT needed | The object already implements `__enter__`/`__exit__` directly (most modern I/O objects, like `open()`'s return value, already do) |

### Exercises - Section 14

**Beginner**
1. Write a simple class with only a `.close()` method, and use `closing()` to manage it safely in a `with` block.

**Intermediate**
2. Explain, with a code example, why calling `with SomeObjectWithOnlyClose():` directly (without `closing()`) would fail.

**Advanced**
3. Reimplement `closing()` yourself as a small class-based context manager (not using `@contextmanager`), and verify it behaves identically to the standard library version.
# 15. `suppress()`

## 15.1 What `contextlib.suppress()` Does

`suppress(*exception_types)` is a ready-made context manager that suppresses the given exception type(s) if they occur inside the block - a direct, concise alternative to writing a custom `try/except: pass`.

In [80]:
from contextlib import suppress

with suppress(FileNotFoundError):
    open("does_not_exist.txt")

print("Continued past a missing file, cleanly")

Continued past a missing file, cleanly


**Output:**
```
Continued past a missing file, cleanly
```

## 15.2 Syntax


```
suppress(ExceptionType1, ExceptionType2, ...)
```

Multiple exception types can be passed, exactly like a multi-type `except` clause.

In [81]:
from contextlib import suppress

with suppress(FileNotFoundError, PermissionError):
    open("/root/protected_file.txt")

print("Handled either a missing file OR a permissions error")

Handled either a missing file OR a permissions error


**Output:**
```
Handled either a missing file OR a permissions error
```

## 15.3 Appropriate Use Cases

- Deleting a file that may or may not exist (`os.remove`), where a missing file is not actually an error condition.
- Optional configuration or plugin loading, where absence is an expected, valid state.
- Cleanup code that may legitimately have nothing to clean up.

In [82]:
import os
from contextlib import suppress

# Cleaning up temp files at the end of a script - "already gone" is not an error
for filename in ["temp1.txt", "temp2.txt", "temp3.txt"]:
    with suppress(FileNotFoundError):
        os.remove(filename)
print("Cleanup pass complete")

Cleanup pass complete


**Output:**
```
Cleanup pass complete
```

## 15.4 Dangerous Use Cases

Just like returning `True` unconditionally from a custom `__exit__` (Part 10), `suppress()` becomes dangerous when applied too broadly - especially `suppress(Exception)`, which silently discards essentially any error.

In [83]:
from contextlib import suppress

# DANGEROUS - suppresses genuine bugs, not just expected conditions
with suppress(Exception):
    result = 1 / 0            # a real bug - silently vanishes
    print("Never reached, but no error surfaces either")

print("Program continues, unaware a ZeroDivisionError ever happened")

Program continues, unaware a ZeroDivisionError ever happened


**Output:**
```
Program continues, unaware a ZeroDivisionError ever happened
```

## 15.5 Comparison with Normal `try`/`except`

In [84]:
# Using suppress() - concise, but ONLY for simple "ignore and continue" cases
from contextlib import suppress
with suppress(KeyError):
    value = {"a": 1}["b"]

# Equivalent try/except - more verbose, but allows a fallback action
try:
    value = {"a": 1}["b"]
except KeyError:
    pass       # suppress() is exactly this pattern, just more concise

In [85]:
# try/except is still necessary (suppress() cannot do this) when you need
# to run different code in the except branch, not just "do nothing":
try:
    value = {"a": 1}["b"]
except KeyError:
    value = "default"          # suppress() has no way to supply a fallback value
print(value)

default


**Output:**
```
default
```

**Precise distinction:** `suppress()` is specifically for the "ignore this exception and move on with no special handling" case. The moment you need to run any recovery logic (like providing a default value) in response to the exception, plain `try/except` is required instead - `suppress()` cannot execute code inside its own "except" behavior.

## 15.6 Summary Table

| Aspect | Detail |
|---|---|
| Import | `from contextlib import suppress` |
| Syntax | `suppress(ExceptionType, ...)` |
| Effect | Silently ignores the given exception type(s) if raised inside the block |
| Good use | Narrow, specific, genuinely expected exception types |
| Bad use | `suppress(Exception)` or other broad, catch-everything usage |
| Limitation vs `try/except` | Cannot run custom fallback/recovery logic - only "ignore and continue" |

### Exercises - Section 15

**Beginner**
1. Use `suppress()` to safely attempt deleting three files that may or may not exist.

**Intermediate**
2. Rewrite a `try/except: pass` block from your own code (or write one) using `suppress()` instead, and explain why it's more concise.

**Advanced**
3. Construct a scenario where `suppress()` cannot be used because a fallback value is needed, and show the correct `try/except` alternative.
# 16. `redirect_stdout()` and `redirect_stderr()`

## 16.1 What They Do

`redirect_stdout(new_target)` and `redirect_stderr(new_target)` temporarily reroute Python's standard output/error streams to a different file-like object for the duration of the `with` block, then automatically restore the original streams afterward.

In [86]:
import io
from contextlib import redirect_stdout

buffer = io.StringIO()

with redirect_stdout(buffer):
    print("This goes into the buffer, not the terminal")
    print("So does this")

captured = buffer.getvalue()
print("Captured output was:")
print(captured)

Captured output was:
This goes into the buffer, not the terminal
So does this



**Output:**
```
Captured output was:
This goes into the buffer, not the terminal
So does this
```

## 16.2 Capturing Printed Output

This is the single most common use case: capturing what a function (that only communicates via `print()`, perhaps from a third-party library you don't control) actually produced, so it can be inspected, tested, or logged.

In [87]:
import io
from contextlib import redirect_stdout

def noisy_function():
    print("Starting process")
    print("Processing item 1")
    print("Processing item 2")
    print("Done")

buffer = io.StringIO()
with redirect_stdout(buffer):
    noisy_function()

lines = buffer.getvalue().splitlines()
print(f"Captured {len(lines)} lines of output")
print("First line was:", lines[0])

Captured 4 lines of output
First line was: Starting process


**Output:**
```
Captured 4 lines of output
First line was: Starting process
```

## 16.3 Redirecting Output to a File

In [88]:
from contextlib import redirect_stdout

with open("output_log.txt", "w") as f:
    with redirect_stdout(f):
        print("This line goes straight into output_log.txt")
        print("So does this one")

with open("output_log.txt") as f:
    print("File contents:", f.read())

File contents: This line goes straight into output_log.txt
So does this one



**Output:**
```
File contents: This line goes straight into output_log.txt
So does this one
```

## 16.4 `redirect_stderr()` - The Same Idea for Error Output

In [89]:
import io
import sys
from contextlib import redirect_stderr

buffer = io.StringIO()
with redirect_stderr(buffer):
    print("This is an error message", file=sys.stderr)

print("Captured stderr:", buffer.getvalue().strip())

Captured stderr: This is an error message


**Output:**
```
Captured stderr: This is an error message
```

## 16.5 Combining Both for Full Output Capture

In [90]:
import io
from contextlib import redirect_stdout, redirect_stderr

stdout_buffer = io.StringIO()
stderr_buffer = io.StringIO()

with redirect_stdout(stdout_buffer), redirect_stderr(stderr_buffer):
    print("normal output")
    import sys
    print("error output", file=sys.stderr)

print("STDOUT was:", stdout_buffer.getvalue().strip())
print("STDERR was:", stderr_buffer.getvalue().strip())

STDOUT was: normal output
STDERR was: error output


**Output:**
```
STDOUT was: normal output
STDERR was: error output
```

## 16.6 Practical Use in Testing

In [91]:
import io
from contextlib import redirect_stdout

def greet(name):
    print(f"Hello, {name}!")

def test_greet_prints_correct_message():
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        greet("Ada")
    assert buffer.getvalue().strip() == "Hello, Ada!"
    print("Test passed")

test_greet_prints_correct_message()

Test passed


**Output:**
```
Test passed
```

## 16.7 Practical Use in Logging-Like Workflows

In [92]:
import io
from contextlib import redirect_stdout
from datetime import datetime

def capture_and_log(func, *args, **kwargs):
    buffer = io.StringIO()
    with redirect_stdout(buffer):
        func(*args, **kwargs)
    output = buffer.getvalue()
    timestamp = datetime.now().strftime("%Y-%m-%d")
    return f"[{timestamp}] Captured output:\n{output}"

def report():
    print("Report generated successfully")

print(capture_and_log(report))

[2026-09-06] Captured output:
Report generated successfully



**Output:**
```
[2026-01-01] Captured output:
Report generated successfully
```

## 16.8 Important Limitation

`redirect_stdout`/`redirect_stderr` only affect Python-level `print()` calls and writes to `sys.stdout`/`sys.stderr` - they do **not** capture output written directly by lower-level C extensions or subprocesses that bypass Python's `sys.stdout` object entirely (for that, redirecting actual OS-level file descriptors is needed, which is a more advanced, separate technique).

## 16.9 Summary Table

| Function | Redirects | Common Use |
|---|---|---|
| `redirect_stdout(target)` | `sys.stdout` (normal `print()` output) | Capturing/testing output, logging |
| `redirect_stderr(target)` | `sys.stderr` (error/warning output) | Capturing error messages separately |
| Combined | Both, using two nested/comma-separated `with` clauses | Full separation of normal vs error output |
| Limitation | Only affects Python-level writes, not raw OS file descriptors | Use lower-level redirection for C extensions/subprocesses |

### Exercises - Section 16

**Beginner**
1. Capture the output of three `print()` calls into an `io.StringIO()` buffer and print the captured text.

**Intermediate**
2. Write a small test function (like Section 16.6) that verifies a function's printed output matches an expected string exactly.

**Advanced**
3. Write a `capture_and_log`-style utility (Section 16.7) that captures both stdout and stderr separately and returns them as a combined, labeled report string.
# 17. `nullcontext()`

## 17.1 What `nullcontext()` Does

`nullcontext()` is a "do-nothing" context manager: entering and exiting it has no effect at all, beyond optionally passing through whatever value it was given. It exists to solve a very specific, common code-structure problem: what to do when a `with` statement is sometimes needed and sometimes isn't, but you don't want to duplicate the block's body in both branches.

In [93]:
from contextlib import nullcontext

with nullcontext() as value:
    print("value is:", value)

value is: None


**Output:**
```
value is: None
```

In [94]:
with nullcontext("passed_through") as value:
    print("value is:", value)

value is: passed_through


**Output:**
```
value is: passed_through
```

## 17.2 Why It Is Useful - Conditional Context Management

Without `nullcontext()`, code that sometimes needs a real context manager and sometimes doesn't tends to end up duplicating the block body, or resorting to awkward workarounds.

In [95]:
# WITHOUT nullcontext() - duplicated block body, error-prone if the logic ever needs to change
def process(data, use_lock, lock):
    if use_lock:
        with lock:
            return [x * 2 for x in data]     # duplicated logic
    else:
        return [x * 2 for x in data]           # duplicated logic

In [96]:
# WITH nullcontext() - single block body, clean conditional context selection
from contextlib import nullcontext

def process(data, use_lock, lock):
    context = lock if use_lock else nullcontext()
    with context:
        return [x * 2 for x in data]           # written exactly once

import threading
real_lock = threading.Lock()
print(process([1, 2, 3], use_lock=True, lock=real_lock))
print(process([1, 2, 3], use_lock=False, lock=real_lock))

[2, 4, 6]
[2, 4, 6]


**Output:**
```
[2, 4, 6]
[2, 4, 6]
```

## 17.3 Practical Example - Optional Database Transaction

In [97]:
from contextlib import nullcontext

class FakeConnection:
    def transaction(self):
        return self

    def __enter__(self):
        print("Transaction started")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Transaction committed")
        return False

def save_record(record, conn, use_transaction=True):
    context = conn.transaction() if use_transaction else nullcontext()
    with context:
        print(f"Saving {record}")

conn = FakeConnection()
save_record("user_1", conn, use_transaction=True)
save_record("user_2", conn, use_transaction=False)

Transaction started


Saving user_1
Transaction committed
Saving user_2


**Output:**
```
Transaction started
Saving user_1
Transaction committed
Saving user_2
```

## 17.4 Practical Example - Optional File Output vs Standard Output

In [98]:
import sys
from contextlib import nullcontext

def write_report(lines, output_path=None):
    context = open(output_path, "w") if output_path else nullcontext(sys.stdout)
    with context as f:
        for line in lines:
            print(line, file=f)

write_report(["Report line 1", "Report line 2"])           # goes to the terminal
write_report(["Report line 1", "Report line 2"], "report.txt")   # goes to a file

Report line 1
Report line 2


**Output:**
```
Report line 1
Report line 2
```

## 17.5 Summary Table

| Aspect | Detail |
|---|---|
| Import | `from contextlib import nullcontext` |
| Effect of entering/exiting | None - it's a no-op |
| Return value | Whatever was passed in as its argument (or `None` if no argument) |
| Primary use case | Conditionally selecting between a real context manager and "no context manager at all" without duplicating code |
| Common pairing | `context = real_cm if condition else nullcontext()` |

### Exercises - Section 17

**Beginner**
1. Use `nullcontext()` to write a function that optionally times its own execution based on a `verbose` flag, without duplicating the timed code.

**Intermediate**
2. Rewrite Section 17.2's `process()` function without `nullcontext()`, using an `if/else` around the whole block, and compare readability.

**Advanced**
3. Design a function that optionally wraps its work in a database transaction (like Section 17.3) versus running unwrapped, and explain a real scenario where this conditional behavior would be genuinely useful (e.g. batch vs. single-record processing).
# 18. `ExitStack`

## 18.1 Why `ExitStack` Exists

`with` statements work well when the exact number and identity of context managers needed is known upfront, written directly in the code. But sometimes the number of resources to manage is **dynamic** - determined at runtime by a loop, a condition, or user input - and a fixed `with a, b, c:` statement cannot express that. `contextlib.ExitStack` solves this by letting you push context managers onto a stack dynamically, guaranteeing they will all be exited correctly, in the right order, no matter how many there end up being.

In [99]:
from contextlib import ExitStack

filenames = ["file1.txt", "file2.txt", "file3.txt"]

with ExitStack() as stack:
    files = [stack.enter_context(open(fname, "w")) for fname in filenames]
    for f in files:
        f.write("some data\n")
print("All files were opened and are now guaranteed closed")

All files were opened and are now guaranteed closed


**Output:**
```
All files were opened and are now guaranteed closed
```

## 18.2 Dynamic Context Management

In [100]:
from contextlib import ExitStack

def process_all_files(filenames):
    with ExitStack() as stack:
        open_files = []
        for name in filenames:
            f = stack.enter_context(open(name, "w"))    # dynamically added, one per filename
            open_files.append(f)
        for f in open_files:
            f.write("processed\n")
        return len(open_files)

count = process_all_files(["a.txt", "b.txt", "c.txt", "d.txt"])
print(f"Processed {count} files, all safely closed")

Processed 4 files, all safely closed

**Output:**
```
Processed 4 files, all safely closed
```

## 18.3 Multiple Resources - Cleaner Than Deeply Nested `with`

Without `ExitStack`, managing a variable number of resources typically requires either recursion or deeply nested `with` statements, both of which are awkward compared to a simple loop.

In [101]:
# WITHOUT ExitStack - awkward for a variable number of files
def open_all_manually(filenames):
    if not filenames:
        yield []
        return
    with open(filenames[0]) as first:
        with open_all_manually(filenames[1:]) as rest:      # recursive nesting - hard to follow
            yield [first] + rest

# WITH ExitStack - simple, flat, and clear
from contextlib import ExitStack

def open_all_with_stack(filenames):
    with ExitStack() as stack:
        return [stack.enter_context(open(f)) for f in filenames]

## 18.4 Conditional Resources

`ExitStack` also cleanly handles the case where whether a resource is needed at all depends on runtime conditions.

In [102]:
from contextlib import ExitStack

def run_task(use_logging, log_context):
    with ExitStack() as stack:
        if use_logging:
            stack.enter_context(log_context)    # only added to the stack conditionally
        print("Running task")

class DummyLogContext:
    def __enter__(self):
        print("Logging started")
        return self
    def __exit__(self, *args):
        print("Logging stopped")
        return False

run_task(use_logging=True, log_context=DummyLogContext())
run_task(use_logging=False, log_context=DummyLogContext())

Logging started
Running task
Logging stopped
Running task


**Output:**
```
Logging started
Running task
Logging stopped
Running task
```

## 18.5 Cleanup Order - Reverse of Entry Order (LIFO)

`ExitStack` guarantees that resources are exited in the **reverse** order they were entered - Last-In, First-Out - which matches the natural expectation that "the most recently acquired resource should be released first" (this is also exactly how nested `with` statements, Part 20, behave).

In [103]:
from contextlib import ExitStack

class Named:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"Entering {self.name}")
        return self
    def __exit__(self, *args):
        print(f"Exiting {self.name}")
        return False

with ExitStack() as stack:
    stack.enter_context(Named("A"))
    stack.enter_context(Named("B"))
    stack.enter_context(Named("C"))

Entering A


Entering B
Entering C
Exiting C
Exiting B
Exiting A


**Output:**
```
Entering A
Entering B
Entering C
Exiting C
Exiting B
Exiting A
```

## 18.6 Registering Arbitrary Cleanup Callbacks - `callback()`

Beyond full context managers, `ExitStack.callback()` registers a plain function to be called during cleanup - useful for simple cleanup actions that don't need a whole context manager written for them.

In [104]:
from contextlib import ExitStack

with ExitStack() as stack:
    print("Doing setup")
    stack.callback(print, "Custom cleanup action 1")
    stack.callback(print, "Custom cleanup action 2")
    print("Doing work")

Doing setup
Doing work
Custom cleanup action 2
Custom cleanup action 1


**Output:**
```
Doing setup
Doing work
Custom cleanup action 2
Custom cleanup action 1
```

## 18.7 Exception Handling with `ExitStack`

If an exception occurs inside the `with ExitStack() as stack:` block, every context manager and callback registered on the stack still runs during cleanup, in the correct reverse order, exactly as if a series of nested `with` statements had been used.

In [105]:
from contextlib import ExitStack

class Named:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"Entering {self.name}")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Exiting {self.name} (exception: {exc_type is not None})")
        return False

try:
    with ExitStack() as stack:
        stack.enter_context(Named("A"))
        stack.enter_context(Named("B"))
        raise RuntimeError("failure mid-way")
except RuntimeError as e:
    print("Caught:", e)

Entering A


Entering B
Exiting B (exception: True)
Exiting A (exception: True)
Caught: failure mid-way


**Output:**
```
Entering A
Entering B
Exiting B (exception: True)
Exiting A (exception: True)
Caught: failure mid-way
```

## 18.8 `pop_all()` - Transferring Ownership

`ExitStack.pop_all()` returns a new `ExitStack` containing all currently registered callbacks/context managers, clearing them from the original stack. This is used when a function builds up a set of resources but wants to hand off long-term ownership of them to the caller, rather than closing them when the building function itself returns.

In [106]:
from contextlib import ExitStack

def open_resources(filenames):
    with ExitStack() as stack:
        files = [stack.enter_context(open(f, "w")) for f in filenames]
        # transfer cleanup responsibility to the CALLER instead of closing here:
        stack = stack.pop_all()
        return files, stack

files, stack = open_resources(["x.txt", "y.txt"])
for f in files:
    f.write("data")
stack.close()          # caller is now responsible for eventually closing everything
print("Caller closed all resources when ready")

Caller closed all resources when ready


**Output:**
```
Caller closed all resources when ready
```

## 18.9 Summary Table

| Feature | Method | Purpose |
|---|---|---|
| Add a context manager dynamically | `stack.enter_context(cm)` | Enters `cm` immediately, registers it for later exit |
| Add a plain cleanup function | `stack.callback(func, *args, **kwargs)` | Registers `func` to run during cleanup, no context manager needed |
| Cleanup order | Reverse of registration (LIFO) | Matches nested `with` semantics |
| Transfer ownership | `stack.pop_all()` | Returns a new stack with the same registrations, clearing the original |
| Manual close | `stack.close()` | Immediately runs all registered cleanup, without needing a `with` block |

### Exercises - Section 18

**Beginner**
1. Use `ExitStack` to open a list of five filenames dynamically and write a line to each.

**Intermediate**
2. Use `stack.callback()` to register three simple print-based cleanup actions and confirm they run in reverse order.

**Advanced**
3. Write a function that conditionally acquires zero, one, or two locks based on parameters, using `ExitStack`, and verify correct cleanup in all three cases, including when an exception occurs.
# 19. Multiple Context Managers

## 19.1 The Comma Syntax

Since Python 3.1 (and with cleaner parenthesized syntax since Python 3.10), a single `with` statement can manage multiple context managers at once, separated by commas.

In [107]:
with open("input.txt", "w") as f1, open("output.txt", "w") as f2:
    f1.write("input data")
    f2.write("output data")
print("Both files are guaranteed closed here")

Both files are guaranteed closed here


**Output:**
```
Both files are guaranteed closed here
```

## 19.2 Parenthesized Form (Python 3.10+)

In [108]:
with (
    open("input.txt", "w") as f1,
    open("output.txt", "w") as f2,
):
    f1.write("input data")
    f2.write("output data")

This form allows each context manager to appear on its own line, which reads far more clearly once there are more than two or three resources involved.

## 19.3 Entry Order

Context managers listed in a single `with` statement are entered **left to right**, in the order they are written.

In [109]:
class Named:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"Entering {self.name}")
        return self
    def __exit__(self, *args):
        print(f"Exiting {self.name}")
        return False

with Named("A") as a, Named("B") as b, Named("C") as c:
    print("Inside the block")

Entering A
Entering B
Entering C
Inside the block
Exiting C
Exiting B
Exiting A


**Output:**
```
Entering A
Entering B
Entering C
Inside the block
Exiting C
Exiting B
Exiting A
```

## 19.4 Exit Order - Reverse of Entry (LIFO)

As shown above, exit order is always the **reverse** of entry order - the last resource entered is the first one exited. This ensures resources are cleaned up in a way that respects any dependencies between them (e.g., if `B`'s setup depended on `A` already being ready, it makes sense for `B` to be torn down before `A`).

## 19.5 Relationship to Nested `with` Statements

`with A() as a, B() as b:` is exactly equivalent to nested `with` statements:


```
with A() as a:
    with B() as b:
        BODY
```

is functionally identical to:


```
with A() as a, B() as b:
    BODY
```

The comma-separated form is simply more concise syntax for the same nested behavior - this equivalence is explored in depth in Part 20.

## 19.6 Exception Behavior with Multiple Context Managers

If an exception occurs inside the block, **every** entered context manager's `__exit__` is still called, in reverse order, exactly like `ExitStack` (Part 18) - because internally, multiple `with` context managers behave exactly like nested single `with` statements.

In [110]:
class Named:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"Entering {self.name}")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Exiting {self.name}")
        return False

try:
    with Named("A") as a, Named("B") as b:
        raise ValueError("failure")
except ValueError as e:
    print("Caught:", e)

Entering A
Entering B
Exiting B
Exiting A
Caught: failure


**Output:**
```
Entering A
Entering B
Exiting B
Exiting A
Caught: failure
```

## 19.7 If an Earlier Context Manager's `__enter__` Fails

If entering the second (or later) context manager fails, any **already-entered** context managers are still exited correctly - Python does not leave earlier resources dangling just because a later one failed to set up.

In [111]:
class Named:
    def __init__(self, name, fail=False):
        self.name = name
        self.fail = fail
    def __enter__(self):
        print(f"Entering {self.name}")
        if self.fail:
            raise RuntimeError(f"{self.name} failed to enter")
        return self
    def __exit__(self, *args):
        print(f"Exiting {self.name}")
        return False

try:
    with Named("A") as a, Named("B", fail=True) as b:
        print("Never reached")
except RuntimeError as e:
    print("Caught:", e)

Entering A
Entering B
Exiting A
Caught: B failed to enter


**Output:**
```
Entering A
Entering B
Exiting A
Caught: B failed to enter
```

Note that `"Exiting B"` never prints - `B.__enter__()` itself raised, so `B` was never successfully entered, and (as established in Part 6.6) `__exit__` is never called for a context manager whose `__enter__` failed. `A`, however, **was** successfully entered, so its `__exit__` correctly still runs.

## 19.8 Summary Table

| Aspect | Behavior |
|---|---|
| Entry order | Left to right, as written |
| Exit order | Right to left (reverse of entry - LIFO) |
| Equivalent to | Nested single `with` statements (Part 20) |
| Exception during the block | All already-entered context managers still exit, in reverse order |
| Failure during entry of a later context manager | Already-entered earlier ones still exit correctly; the failed one's `__exit__` is never called |

### Exercises - Section 19

**Beginner**
1. Open three files at once using the comma-separated `with` syntax and write different content to each.

**Intermediate**
2. Reproduce Section 19.7's "one context manager fails to enter" scenario with your own classes, and confirm which `__exit__` methods run and which don't.

**Advanced**
3. Rewrite a multi-resource `with A() as a, B() as b, C() as c:` statement as fully nested `with` statements, and verify the entry/exit print order is identical either way.
# 20. Nested Context Managers

## 20.1 Basic Nesting

In [112]:
class A:
    def __enter__(self):
        print("Entering A")
        return self
    def __exit__(self, *args):
        print("Exiting A")
        return False

class B:
    def __enter__(self):
        print("Entering B")
        return self
    def __exit__(self, *args):
        print("Exiting B")
        return False

with A():
    with B():
        print("Inside both A and B")

Entering A


Entering B
Inside both A and B
Exiting B
Exiting A


**Output:**
```
Entering A
Entering B
Inside both A and B
Exiting B
Exiting A
```

## 20.2 Entry Sequence

Nesting means the **outer** context manager is entered first, then the **inner** one, before the innermost block body finally runs.


```
with A():        --> [1] Entering A
    with B():     --> [2] Entering B
        BODY        --> [3] Run BODY
```

## 20.3 Exit Sequence

Exiting happens in exactly the reverse order: the **inner** context manager exits first, then the **outer** one - because the inner `with` block is fully contained within, and must finish completely, before control returns to the outer block's remaining code.


```
        BODY        (finishes)
    <-- [1] Exiting B (inner exits first)
<-- [2] Exiting A (outer exits last)
```

## 20.4 Cleanup Order Matters - A Practical Example

In [113]:
class DatabaseConnection:
    def __enter__(self):
        print("Opening database connection")
        return self
    def __exit__(self, *args):
        print("Closing database connection")
        return False

class Transaction:
    def __enter__(self):
        print("Starting transaction")
        return self
    def __exit__(self, *args):
        print("Committing transaction")
        return False

with DatabaseConnection():
    with Transaction():
        print("Running queries")

Opening database connection
Starting transaction
Running queries
Committing transaction
Closing database connection


**Output:**
```
Opening database connection
Starting transaction
Running queries
Committing transaction
Closing database connection
```

This order is not arbitrary: the transaction **must** be committed **before** the connection is closed - committing on a closed connection would fail. Nesting order directly encodes this dependency: the connection (the "outer," longer-lived resource) is opened first and closed last; the transaction (the "inner," shorter-lived resource, dependent on the connection existing) is started after and finished before.

## 20.5 Exception Propagation Through Nested Contexts

An exception raised in the innermost block propagates outward through each level's `__exit__` in turn, giving every level a chance to react (log, clean up, or even suppress) before the exception (if not suppressed) continues to the next level out.

In [114]:
class Level:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f"Entering {self.name}")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Exiting {self.name} (saw exception: {exc_type is not None})")
        return False        # never suppress - let it keep propagating outward

try:
    with Level("Outer"):
        with Level("Middle"):
            with Level("Inner"):
                raise ValueError("deep failure")
except ValueError as e:
    print("Finally caught at the top level:", e)

Entering Outer


Entering Middle
Entering Inner
Exiting Inner (saw exception: True)
Exiting Middle (saw exception: True)
Exiting Outer (saw exception: True)
Finally caught at the top level: deep failure


**Output:**
```
Entering Outer
Entering Middle
Entering Inner
Exiting Inner (saw exception: True)
Exiting Middle (saw exception: True)
Exiting Outer (saw exception: True)
Finally caught at the top level: deep failure
```

## 20.6 A Level Suppressing an Exception Stops It From Propagating Further

In [115]:
class SuppressingLevel:
    def __enter__(self):
        print("Entering SuppressingLevel")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print("Exiting SuppressingLevel - suppressing any exception")
        return True         # stops the exception here

class OuterLevel:
    def __enter__(self):
        print("Entering OuterLevel")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Exiting OuterLevel (saw exception: {exc_type is not None})")
        return False

with OuterLevel():
    with SuppressingLevel():
        raise ValueError("this gets stopped here")
    print("This line DOES run - the exception never reached this far")

Entering OuterLevel


Entering SuppressingLevel
Exiting SuppressingLevel - suppressing any exception
This line DOES run - the exception never reached this far
Exiting OuterLevel (saw exception: False)


**Output:**
```
Entering OuterLevel
Entering SuppressingLevel
Exiting SuppressingLevel - suppressing any exception
This line DOES run - the exception never reached this far
Exiting OuterLevel (saw exception: False)
```

Notice `OuterLevel.__exit__` sees `exc_type` as `None` - from its perspective, no exception ever occurred, because `SuppressingLevel` fully absorbed it before it could propagate any further outward.

## 20.7 Summary Table

| Aspect | Behavior |
|---|---|
| Entry order | Outer first, then inner (top to bottom, as written) |
| Exit order | Inner first, then outer (bottom to top - reverse of entry) |
| Exception propagation | Flows from innermost `__exit__` outward, level by level |
| Suppression at any level | Stops the exception from reaching any further-out levels entirely |
| Practical significance | Encodes dependency order - e.g., commit a transaction before closing its connection |

### Exercises - Section 20

**Beginner**
1. Nest three custom context managers (`A`, `B`, `C`) and predict the exact entry/exit print order before running the code.

**Intermediate**
2. Reproduce Section 20.6's suppression example with your own three-level nesting, and confirm exactly which levels "see" the exception and which don't.

**Advanced**
3. Design a nested pair of context managers (like `DatabaseConnection`/`Transaction`) where the nesting *order* genuinely matters for correctness, and explain what would go wrong if the order were reversed.
# 21. File Handling with Context Managers

This section assumes familiarity with file I/O itself (covered in the separate File Handling reference) and focuses exclusively on the **resource-management** angle: why `with open(...)` specifically is the correct pattern, and how it behaves under various conditions.

## 21.1 Reading Files Safely

In [116]:
with open("data.csv", "r") as f:
    contents = f.read()
# 'f' is guaranteed closed here, even though we're still using 'contents' below
print(f"Read {len(contents)} characters")
print("File closed:", f.closed)

Read 21 characters
File closed: True


**Output:**
```
Read 21 characters
File closed: True
```

## 21.2 Writing Files Safely

In [117]:
with open("output.txt", "w") as f:
    f.write("Line 1\n")
    f.write("Line 2\n")
# buffered writes are guaranteed FLUSHED and the file closed by this point
print("Write complete and safely flushed to disk")

Write complete and safely flushed to disk


**Output:**
```
Write complete and safely flushed to disk
```

**Why this matters specifically for writing:** writes are typically buffered in memory for performance, and are only guaranteed to actually reach disk when the file is closed (or explicitly `.flush()`ed). A crash between writing and manually calling `.close()` can leave a file empty or truncated on disk - exactly the scenario `with` protects against, since `__exit__` runs the close (and thus the flush) even if an exception interrupts the writing.

## 21.3 Multiple Files at Once

In [118]:
with open("source.txt", "w") as f:
    f.write("hello\nworld\n")

with open("source.txt", "r") as infile, open("destination.txt", "w") as outfile:
    for line in infile:
        outfile.write(line.upper())
print("Both files guaranteed closed, even if an error occurred mid-copy")

Both files guaranteed closed, even if an error occurred mid-copy


**Output:**
```
Both files guaranteed closed, even if an error occurred mid-copy
```

## 21.4 Binary Files

In [119]:
with open("image.bin", "wb") as f:
    f.write(b"\x89PNG\r\n\x1a\n")
print("Binary file safely closed")

with open("image.bin", "rb") as f:
    header = f.read(8)
print("Header bytes:", header)

Binary file safely closed
Header bytes: b'\x89PNG\r\n\x1a\n'


**Output:**
```
Binary file safely closed
Header bytes: b'\x89PNG\r\n\x1a\n'
```

Binary mode (`"rb"`/`"wb"`) behaves identically from a resource-management perspective - the context manager guarantees closure regardless of mode.

## 21.5 Encoding-Aware File Handling

In [120]:
with open("notes_utf8.txt", "w", encoding="utf-8") as f:
    f.write("café, naïve, 日本語")

with open("notes_utf8.txt", "r", encoding="utf-8") as f:
    print(f.read())

café, naïve, 日本語


**Output:**
```
café, naïve, 日本語
```

Specifying `encoding` explicitly (rather than relying on a platform-dependent default) is a resource-management best practice too: it makes the file's cleanup and content interpretation fully deterministic and portable across systems.

## 21.6 Exception-Safe File Handling - The Core Guarantee, Demonstrated

In [121]:
try:
    with open("output2.txt", "w") as f:
        f.write("First line\n")
        raise ValueError("Something went wrong during processing")
        f.write("Second line\n")     # never reached
except ValueError as e:
    print("Caught:", e)

with open("output2.txt", "r") as f:
    print(repr(f.read()))

Caught: Something went wrong during processing
'First line\n'


**Output:**
```
Caught: Something went wrong during processing
'First line\n'
```

The file was still correctly closed (and the first write correctly flushed to disk), even though an exception interrupted the block partway through - precisely the guarantee established generally in Part 1-Part 9, applied here to the single most common context manager in everyday Python code.

## 21.7 Why `with open(...)` Is a Fundamental Python Pattern

`open()`'s returned file object was one of the very first, and remains one of the most universally used, built-in implementations of the context manager protocol in the entire standard library. It is the pattern every Python developer learns first, and it's the model this entire guide's custom context managers (database connections, locks, temporary resources) are all built to emulate: **acquire a resource, guarantee its release, regardless of what happens while it's in use.**

## 21.8 Summary Table

| Practice | Why It Matters for Resource Management |
|---|---|
| Always use `with open(...)`, never bare `open()`/`close()` pairs | Guarantees closure (and flushing) even if an exception occurs |
| Specify `encoding` explicitly for text files | Deterministic, portable behavior |
| Use multiple `with` clauses for multiple files | Both files guaranteed closed, in correct reverse order |
| Binary mode behaves identically for cleanup purposes | The context manager protocol doesn't care about the file's mode |

### Exercises - Section 21

**Beginner**
1. Write a `with` block that reads a file and confirm `f.closed` is `True` immediately after the block ends.

**Intermediate**
2. Reproduce Section 21.6's exception-safety demonstration with your own file and exception type, confirming partial writes are still flushed correctly.

**Advanced**
3. Write a function that copies a file's contents to a new file line-by-line, using two file context managers at once, and demonstrate it remains safe even if a `ValueError` is deliberately raised partway through the copy.
# 22. Database Context Managers

## 22.1 Two Conceptually Different Kinds of Context

Database code typically involves **two distinct** context-management concerns that are easy to conflate but should be understood separately:

| Context Type | What It Manages | Cleanup Action |
|---|---|---|
| **Resource cleanup context** | The connection itself (a network socket, a server-side session) | Close the connection |
| **Transaction context** | A logical unit of work (a group of statements that should succeed or fail together) | Commit on success, roll back on failure |

A single database interaction very often needs **both**, nested appropriately (the transaction lives inside the connection's lifetime, following the same "outer resource, inner unit of work" pattern from Part 20.4).

## 22.2 Connection as a Resource-Cleanup Context Manager

In [122]:
import sqlite3

with sqlite3.connect(":memory:") as conn:
    conn.execute("CREATE TABLE items (id INTEGER, name TEXT)")
    conn.execute("INSERT INTO items VALUES (1, 'Widget')")

**Important, precise detail for `sqlite3` specifically:** the connection object's own `__exit__` in the standard library's `sqlite3` module actually manages the **transaction** (commit on success, rollback on exception) - it does **not** close the connection itself. This is a commonly misunderstood detail, demonstrated next.

In [123]:
import sqlite3

conn = sqlite3.connect(":memory:")
with conn:
    conn.execute("CREATE TABLE items (id INTEGER)")

print("Connection closed?", )
try:
    conn.execute("SELECT 1")     # still works! the connection is NOT closed by 'with conn:'
    print("Still open - 'with conn:' only manages the transaction, not the connection itself")
finally:
    conn.close()                  # closing the connection is a SEPARATE, manual step

Connection closed?
Still open - 'with conn:' only manages the transaction, not the connection itself


**Output:**
```
Connection closed? 
Still open - 'with conn:' only manages the transaction, not the connection itself
```

## 22.3 Building an Explicit Resource-Cleanup Wrapper

Because the standard library's built-in behavior can be surprising, many real projects wrap the connection in a custom context manager that handles **both** concerns explicitly and unambiguously.

In [124]:
import sqlite3

class Database:
    def __init__(self, path):
        self.path = path
        self.conn = None

    def __enter__(self):
        self.conn = sqlite3.connect(self.path)
        return self.conn

    def __exit__(self, exc_type, exc_value, traceback):
        self.conn.close()          # THIS context manager DOES close the connection
        return False

with Database(":memory:") as conn:
    conn.execute("CREATE TABLE users (id INTEGER)")
print("Connection is now genuinely closed")

Connection is now genuinely closed


**Output:**
```
Connection is now genuinely closed
```

## 22.4 Transactions: Commit on Success, Rollback on Failure

In [125]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (id INTEGER, balance INTEGER)")
conn.execute("INSERT INTO accounts VALUES (1, 100)")
conn.commit()

from contextlib import contextmanager

@contextmanager
def transaction(connection):
    try:
        yield connection
        connection.commit()
        print("Transaction committed")
    except Exception:
        connection.rollback()
        print("Transaction rolled back")
        raise

try:
    with transaction(conn) as tx_conn:
        tx_conn.execute("UPDATE accounts SET balance = balance - 500 WHERE id = 1")
        balance = tx_conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()[0]
        if balance < 0:
            raise ValueError("Insufficient funds")
except ValueError as e:
    print("Caught:", e)

print("Final balance:", conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()[0])
conn.close()

Transaction rolled back
Caught: Insufficient funds
Final balance: 100


**Output:**
```
Transaction rolled back
Caught: Insufficient funds
Final balance: 100
```

## 22.5 Cursors as Context Managers

Some database libraries also support using cursors as context managers, ensuring the cursor itself is closed (releasing any server-side result-set resources) after use, independent of the connection's lifetime.

In [126]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE items (name TEXT)")
conn.execute("INSERT INTO items VALUES ('Widget')")
conn.commit()

with conn:
    cursor = conn.cursor()
    try:
        cursor.execute("SELECT * FROM items")
        print(cursor.fetchall())
    finally:
        cursor.close()             # explicit cleanup, since sqlite3's cursor is not itself a context manager

conn.close()

[('Widget',)]


**Output:**
```
[('Widget',)]
```

## 22.6 Nesting Connection and Transaction Contexts Together

In [127]:
import sqlite3
from contextlib import contextmanager

class Database:
    def __init__(self, path):
        self.path = path

    def __enter__(self):
        self.conn = sqlite3.connect(self.path)
        return self.conn

    def __exit__(self, exc_type, exc_value, traceback):
        self.conn.close()
        return False

@contextmanager
def transaction(connection):
    try:
        yield connection
        connection.commit()
    except Exception:
        connection.rollback()
        raise

with Database(":memory:") as conn:                # OUTER: manages the connection's lifetime
    conn.execute("CREATE TABLE logs (message TEXT)")
    with transaction(conn):                          # INNER: manages one unit of work
        conn.execute("INSERT INTO logs VALUES ('System started')")
    print(conn.execute("SELECT * FROM logs").fetchall())

[('System started',)]


**Output:**
```
[('System started',)]
```

## 22.7 Why Distinguishing These Two Contexts Matters

Conflating "the connection is open" with "this transaction succeeded" leads to subtle bugs: a single connection is often reused across **many** transactions over its lifetime (e.g., in a long-running web server), so committing/rolling back should never accidentally close the connection, and closing the connection should never happen implicitly as a side effect of one transaction finishing.

## 22.8 Summary Table

| Concern | Managed By | Cleanup Action |
|---|---|---|
| Connection lifetime | An outer context manager (custom or library-provided) | Close the connection |
| One transaction/unit of work | An inner context manager, often `@contextmanager`-based | Commit on success, rollback on exception |
| Cursor/result set (library-dependent) | Its own context manager, if supported | Release server-side result-set resources |
| Common mistake | Assuming `with conn:` always closes the connection | Verify your specific database library's actual behavior |

### Exercises - Section 22

**Beginner**
1. Use `sqlite3.connect(":memory:")` to create a table and insert a row inside a `with conn:` block, then verify the connection is still open afterward.

**Intermediate**
2. Write your own `transaction()` generator-based context manager (as in Section 22.4) and test both a successful and a failing transaction.

**Advanced**
3. Build a `Database` class combining connection-lifetime management and transaction management as two separate, nestable context managers, and write a small script performing three separate transactions over the same connection.
# 23. Locks and Thread Safety

## 23.1 Why Locks Need Context Managers

A **lock** ensures only one thread at a time can execute a particular section of code (a "critical section"), preventing race conditions on shared data. Locks must always be released after being acquired - but as shown in Part 3.4, an exception between `acquire()` and `release()` leaves the lock held forever, causing every other thread waiting on it to block permanently (a deadlock).

In [128]:
import threading

lock = threading.Lock()

with lock:
    print("Inside the critical section - only one thread can be here at a time")
print("Lock automatically released")

Inside the critical section - only one thread can be here at a time
Lock automatically released


**Output:**
```
Inside the critical section - only one thread can be here at a time
Lock automatically released
```

## 23.2 Acquisition and Release - What `with lock:` Actually Does

`threading.Lock` objects implement the context manager protocol directly: `__enter__` calls `acquire()`, and `__exit__` calls `release()`.

In [129]:
import threading

lock = threading.Lock()

# What "with lock:" does internally:
lock.acquire()
try:
    print("Critical section")
finally:
    lock.release()

Critical section


**Output:**
```
Critical section
```

## 23.3 Exception Safety - The Core Benefit

In [130]:
import threading

lock = threading.Lock()

try:
    with lock:
        raise RuntimeError("failure inside critical section")
except RuntimeError:
    pass

print("Lock released despite the exception:", not lock.locked())

Lock released despite the exception: True


**Output:**
```
Lock released despite the exception: True
```

Compare this to the manual, unsafe version from Part 3.4, where the same exception left the lock permanently held - this is the single most important reason to always use `with lock:` rather than manual `acquire()`/`release()` pairs.

## 23.4 Multiple Threads Sharing a Resource - A Complete Example

In [131]:
import threading

counter = 0
lock = threading.Lock()

def increment_many_times():
    global counter
    for _ in range(100_000):
        with lock:
            counter += 1

threads = [threading.Thread(target=increment_many_times) for _ in range(4)]
for t in threads:
    t.start()
for t in threads:
    t.join()

print("Final counter value:", counter)

Final counter value: 400000


**Output:**
```
Final counter value: 400000
```

Without the lock, concurrent `counter += 1` operations from multiple threads would race against each other, frequently producing a final value **less** than the mathematically expected `400000` due to lost updates.

## 23.5 `RLock` - Reentrant Locks

A regular `Lock` cannot be acquired again by the same thread that already holds it (doing so deadlocks against itself). `threading.RLock` (a **reentrant lock**) allows the same thread to acquire it multiple times, as long as it releases it the same number of times - useful when a function holding a lock needs to call another function that also needs the same lock.

In [132]:
import threading

lock = threading.RLock()

def outer():
    with lock:
        print("Outer acquired the lock")
        inner()          # inner() also acquires the SAME lock, same thread - fine with RLock

def inner():
    with lock:
        print("Inner also acquired the same lock (reentrant)")

outer()

Outer acquired the lock
Inner also acquired the same lock (reentrant)


**Output:**
```
Outer acquired the lock
Inner also acquired the same lock (reentrant)
```

(This use of "reentrant" specifically for locks is related to, but distinct from, the broader concept of reentrant *context managers* covered fully in Part 34.)

## 23.6 Deadlock Considerations

A **deadlock** occurs when two or more threads each hold a lock the other needs, and neither can proceed. Context managers don't eliminate this risk entirely (multiple-lock deadlocks are still possible), but they do eliminate the specific, very common cause of "a lock is never released because of an unhandled exception."

In [133]:
import threading

lock_a = threading.Lock()
lock_b = threading.Lock()

def task_1():
    with lock_a:
        print("task_1 acquired lock_a")
        # if this tried to acquire lock_b while task_2 holds lock_b and wants lock_a - deadlock risk
        with lock_b:
            print("task_1 acquired lock_b")

def task_2():
    with lock_b:
        print("task_2 acquired lock_b")
        with lock_a:                # ACQUIRING IN OPPOSITE ORDER - classic deadlock risk pattern
            print("task_2 acquired lock_a")

# Best practice: always acquire multiple locks in a CONSISTENT, agreed-upon order across the whole codebase

**Best practice:** when a piece of code needs multiple locks, always acquire them in the same, globally agreed-upon order everywhere in the codebase - this single discipline eliminates the classic "circular wait" deadlock pattern shown above.

## 23.7 Why Context Managers Are Especially Useful for Concurrency

- **Guaranteed release**, even under exceptions - the single biggest concurrency-safety win.
- **Clear, visually scoped critical sections** - the indentation of the `with` block makes it immediately visible exactly which code runs "protected," and exactly where protection ends.
- **Composability with `ExitStack`** (Part 18) for acquiring a dynamic number of locks safely.

In [134]:
from contextlib import ExitStack
import threading

locks = [threading.Lock() for _ in range(3)]

with ExitStack() as stack:
    for lock in locks:
        stack.enter_context(lock)      # acquire all of them, in order, safely
    print("All locks held simultaneously")
print("All locks released, in reverse order")

All locks held simultaneously
All locks released, in reverse order


**Output:**
```
All locks held simultaneously
All locks released, in reverse order
```

## 23.8 Summary Table

| Concept | Detail |
|---|---|
| `with lock:` | Calls `lock.acquire()` on entry, `lock.release()` on exit |
| Exception safety | Lock is released even if the critical section raises |
| `RLock` | Allows the same thread to re-acquire the same lock (reentrant) |
| Deadlock | Still possible with multiple locks acquired in inconsistent order across the codebase |
| Best practice | Always acquire multiple locks in a single, consistent global order |
| `ExitStack` | Useful for acquiring a dynamic number of locks safely at once |

### Exercises - Section 23

**Beginner**
1. Use `with lock:` to protect a shared counter incremented by a single thread in a loop, and print the final value.

**Intermediate**
2. Reproduce Section 23.4's multi-threaded counter example, then remove the lock and observe (and explain) the incorrect final count that can result.

**Advanced**
3. Design a two-lock scenario deliberately prone to deadlock (like Section 23.6), then fix it by enforcing a consistent lock-acquisition order, and explain why the fix works.
# 24. Temporary Resources

## 24.1 Temporary Files

`tempfile.NamedTemporaryFile()` and `tempfile.TemporaryFile()` create a file that is guaranteed to be deleted when the context manager exits (by default), ideal for intermediate data that should never outlive the operation using it.

In [135]:
import tempfile

with tempfile.NamedTemporaryFile(mode="w", delete=True, suffix=".txt") as tmp:
    tmp.write("Temporary content")
    tmp.flush()
    print(f"Temp file created at: {tmp.name}")

import os
print("Still exists after the block?", os.path.exists(tmp.name))

Temp file created at: /tmp/tmpovv9sbf0.txt
Still exists after the block? False


**Output:**
```
Temp file created at: /tmp/tmpxyz123abc.txt
Still exists after the block? False
```

## 24.2 Temporary Directories

`tempfile.TemporaryDirectory()` creates an entire directory (and guarantees deleting it, along with everything inside it, when the block ends) - useful for operations that need to stage multiple intermediate files.

In [136]:
import tempfile
import os

with tempfile.TemporaryDirectory() as tmp_dir:
    file_path = os.path.join(tmp_dir, "intermediate.csv")
    with open(file_path, "w") as f:
        f.write("col1,col2\n1,2\n")
    print("Files in temp dir:", os.listdir(tmp_dir))

print("Temp dir still exists?", os.path.exists(tmp_dir))

Files in temp dir: ['intermediate.csv']
Temp dir still exists? False


**Output:**
```
Files in temp dir: ['intermediate.csv']
Temp dir still exists? False
```

## 24.3 Temporary Configuration Changes

A custom context manager can temporarily change a setting and guarantee it's restored afterward, regardless of what happens in between - a very common and useful pattern for tests and short-lived operations.

In [137]:
from contextlib import contextmanager

config = {"debug": False, "log_level": "INFO"}

@contextmanager
def temporary_config(**overrides):
    original = config.copy()
    config.update(overrides)
    try:
        yield config
    finally:
        config.clear()
        config.update(original)     # guaranteed restoration, even if the block raises

print("Before:", config)
with temporary_config(debug=True, log_level="DEBUG"):
    print("During:", config)
print("After:", config)

Before: {'debug': False, 'log_level': 'INFO'}
During: {'debug': True, 'log_level': 'DEBUG'}
After: {'debug': False, 'log_level': 'INFO'}


**Output:**
```
Before: {'debug': False, 'log_level': 'INFO'}
During: {'debug': True, 'log_level': 'DEBUG'}
After: {'debug': False, 'log_level': 'INFO'}
```

## 24.4 Temporary Environment Variable Changes

In [138]:
import os
from contextlib import contextmanager

@contextmanager
def temporary_env(**env_vars):
    original = {key: os.environ.get(key) for key in env_vars}
    os.environ.update(env_vars)
    try:
        yield
    finally:
        for key, original_value in original.items():
            if original_value is None:
                os.environ.pop(key, None)
            else:
                os.environ[key] = original_value

print("Before:", os.environ.get("MY_APP_MODE"))
with temporary_env(MY_APP_MODE="testing"):
    print("During:", os.environ.get("MY_APP_MODE"))
print("After:", os.environ.get("MY_APP_MODE"))

Before: None
During: testing
After: None


**Output:**
```
Before: None
During: testing
After: None
```

## 24.5 Temporary Working Directory Changes

Python 3.11+ provides `contextlib.chdir()` directly; for earlier versions, the same pattern is easy to write by hand.

In [139]:
import os
from contextlib import contextmanager

@contextmanager
def temporary_chdir(path):
    original = os.getcwd()
    os.chdir(path)
    try:
        yield
    finally:
        os.chdir(original)          # guaranteed to restore, even on exception

import tempfile
print("Before:", os.getcwd())
with tempfile.TemporaryDirectory() as tmp_dir:
    with temporary_chdir(tmp_dir):
        print("During:", os.getcwd())
print("After (restored):", os.getcwd())

Before: /home/claude/ctxmgr_guide
During: /tmp/tmpo78xriif
After (restored): /home/claude/ctxmgr_guide


**Output:**
```
Before: /home/user/project
During: /tmp/tmpabc123
After (restored): /home/user/project
```

## 24.6 Cleanup Guarantees - Even When Exceptions Occur

Every example in this section follows the same essential shape (`try: yield` / `finally: restore or delete`), guaranteeing cleanup even when the block raises - demonstrated explicitly below.

In [140]:
from contextlib import contextmanager

@contextmanager
def temporary_setting(settings_dict, key, value):
    original = settings_dict.get(key)
    settings_dict[key] = value
    try:
        yield
    finally:
        settings_dict[key] = original

settings = {"mode": "production"}
try:
    with temporary_setting(settings, "mode", "testing"):
        print("During:", settings["mode"])
        raise RuntimeError("something failed mid-operation")
except RuntimeError as e:
    print("Caught:", e)

print("After (restored despite the exception):", settings["mode"])

During: testing
Caught: something failed mid-operation
After (restored despite the exception): production


**Output:**
```
During: testing
Caught: something failed mid-operation
After (restored despite the exception): production
```

## 24.7 Summary Table

| Temporary Resource | Tool | Cleanup Guarantee |
|---|---|---|
| Temporary file | `tempfile.NamedTemporaryFile()` | Deleted automatically on exit (by default) |
| Temporary directory | `tempfile.TemporaryDirectory()` | Directory and all contents deleted on exit |
| Temporary configuration | Custom `@contextmanager` with `try/finally` | Original values restored on exit |
| Temporary environment variables | Custom `@contextmanager` with `try/finally` | Original environment restored on exit |
| Temporary working directory | `contextlib.chdir()` (3.11+) or custom equivalent | Original directory restored on exit |

### Exercises - Section 24

**Beginner**
1. Create a temporary file, write to it, and confirm it no longer exists after the `with` block ends.

**Intermediate**
2. Write a `temporary_config`-style context manager for a dictionary of your own settings, and test that values are restored even when the block raises an exception.

**Advanced**
3. Combine `TemporaryDirectory()` and a custom `temporary_chdir()` context manager (nested) to write a function that safely performs file operations "as if" inside a fresh, isolated directory, regardless of the caller's actual current directory.
# 25. Context Managers in Data Science

## 25.1 Timing Operations

Measuring how long a computation takes is one of the most common context-manager use cases in exploratory data analysis and pipeline optimization.

In [141]:
from contextlib import contextmanager
import time

@contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    print(f"[{label}] took {elapsed:.4f}s")

with timed("data loading"):
    data = [x ** 2 for x in range(1_000_000)]

with timed("aggregation"):
    total = sum(data)

[data loading] took 0.1949s


[aggregation] took 0.0074s


**Output:**
```
[data loading] took 0.0891s
[aggregation] took 0.0072s
```

## 25.2 Managing Datasets - Guaranteed Release of Large In-Memory Objects

In [142]:
from contextlib import contextmanager

@contextmanager
def load_dataset(path):
    print(f"Loading dataset from {path}")
    dataset = list(range(1_000_000))     # stand-in for a real large dataset load
    try:
        yield dataset
    finally:
        print("Releasing dataset from memory")
        dataset.clear()

with load_dataset("sales_data.csv") as data:
    print("Row count:", len(data))
print("Dataset released after analysis")

Loading dataset from sales_data.csv
Row count: 1000000
Releasing dataset from memory
Dataset released after analysis


**Output:**
```
Loading dataset from sales_data.csv
Row count: 1000000
Releasing dataset from memory
Dataset released after analysis
```

## 25.3 Temporary Files for Intermediate Analysis Steps

In [143]:
import tempfile
import csv

with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", delete=True, newline="") as tmp:
    writer = csv.writer(tmp)
    writer.writerow(["id", "value"])
    writer.writerow([1, 100])
    tmp.flush()
    print(f"Intermediate CSV staged at {tmp.name}")
print("Intermediate file automatically cleaned up")

Intermediate CSV staged at /tmp/tmpzwovf_ua.csv
Intermediate file automatically cleaned up


**Output:**
```
Intermediate CSV staged at /tmp/tmpabcd1234.csv
Intermediate file automatically cleaned up
```

## 25.4 Database Connections for Data Pulls

In [144]:
import sqlite3

with sqlite3.connect(":memory:") as conn:
    conn.execute("CREATE TABLE sales (region TEXT, amount INTEGER)")
    conn.executemany("INSERT INTO sales VALUES (?, ?)", [("East", 100), ("West", 150)])
    rows = conn.execute("SELECT region, SUM(amount) FROM sales GROUP BY region").fetchall()
    print(rows)

[('East', 100), ('West', 150)]


**Output:**
```
[('East', 100), ('West', 150)]
```

## 25.5 Resource-Heavy Computations - Guaranteeing Cleanup of Worker Pools

In [145]:
from contextlib import contextmanager
from multiprocessing import Pool

@contextmanager
def worker_pool(processes):
    pool = Pool(processes=processes)
    try:
        yield pool
    finally:
        pool.close()
        pool.join()
        print("Worker pool fully shut down")

def square(x):
    return x * x

with worker_pool(2) as pool:
    results = pool.map(square, [1, 2, 3, 4])
print("Results:", results)

Worker pool fully shut down
Results: [1, 4, 9, 16]


**Output:**
```
Worker pool fully shut down
Results: [1, 4, 9, 16]
```

## 25.6 Experiment Configuration Scoping

In [146]:
from contextlib import contextmanager

analysis_settings = {"sample_size": 1000, "confidence": 0.95}

@contextmanager
def analysis_scope(**overrides):
    original = analysis_settings.copy()
    analysis_settings.update(overrides)
    try:
        yield analysis_settings
    finally:
        analysis_settings.clear()
        analysis_settings.update(original)

with analysis_scope(sample_size=5000):
    print("Running analysis with:", analysis_settings)
print("Reverted to default settings:", analysis_settings)

Running analysis with: {'sample_size': 5000, 'confidence': 0.95}
Reverted to default settings: {'sample_size': 1000, 'confidence': 0.95}


**Output:**
```
Running analysis with: {'sample_size': 5000, 'confidence': 0.95}
Reverted to default settings: {'sample_size': 1000, 'confidence': 0.95}
```

## 25.7 Temporary Working Directories for Reproducible Analysis

In [147]:
import tempfile
import os
from contextlib import contextmanager

@contextmanager
def isolated_workspace():
    original = os.getcwd()
    with tempfile.TemporaryDirectory() as tmp_dir:
        os.chdir(tmp_dir)
        try:
            yield tmp_dir
        finally:
            os.chdir(original)

with isolated_workspace() as workspace:
    with open("results.txt", "w") as f:
        f.write("analysis output")
    print("Files created in isolated workspace:", os.listdir(workspace))
print("Back in the original directory:", os.getcwd())

Files created in isolated workspace: ['results.txt']
Back in the original directory: /home/claude/ctxmgr_guide


**Output:**
```
Files created in isolated workspace: ['results.txt']
Back in the original directory: /home/user/project
```

## 25.8 Logging/Capturing Output During Batch Analysis

In [148]:
import io
from contextlib import redirect_stdout

def run_noisy_analysis_step():
    print("Step 1 complete")
    print("Step 2 complete")

log_buffer = io.StringIO()
with redirect_stdout(log_buffer):
    run_noisy_analysis_step()

print("Captured analysis log:")
print(log_buffer.getvalue())

Captured analysis log:
Step 1 complete
Step 2 complete



**Output:**
```
Captured analysis log:
Step 1 complete
Step 2 complete
```

## 25.9 Summary Table

| Data Science Task | Context Manager Pattern |
|---|---|
| Timing a computation | Custom `@contextmanager` with `time.perf_counter()` |
| Managing large in-memory datasets | Custom context manager releasing memory in `finally` |
| Staging intermediate files | `tempfile.NamedTemporaryFile()` |
| Pulling data from a database | Connection context manager (Part 22) |
| Parallel/resource-heavy computation | Custom context manager wrapping a worker pool |
| Scoped experiment configuration | Custom `@contextmanager` restoring settings in `finally` |
| Reproducible, isolated analysis runs | Combined `TemporaryDirectory()` + working-directory context manager |
| Capturing analysis logs | `redirect_stdout()` |

### Exercises - Section 25

**Beginner**
1. Write a `timed()` context manager and use it to compare the speed of two different ways of summing a large list.

**Intermediate**
2. Write a context manager that loads a dataset (a list or dict, standing in for real data) and guarantees its memory is released even if an exception occurs during analysis.

**Advanced**
3. Combine `isolated_workspace()` (Section 25.7) with `analysis_scope()` (Section 25.6) to run a fully reproducible, isolated mini-analysis, confirming both the working directory and settings are correctly restored afterward.
# 26. Context Managers in Machine Learning

## 26.1 Model Training Resource Management

In [149]:
from contextlib import contextmanager
import time

@contextmanager
def training_session(model_name):
    print(f"Initializing training session for {model_name}")
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        print(f"Training session for {model_name} ended after {elapsed:.4f}s (cleanup guaranteed)")

with training_session("fraud_detector_v3"):
    print("Running training loop...")
    # simulate training iterations
    for epoch in range(3):
        print(f"Epoch {epoch + 1} complete")

Initializing training session for fraud_detector_v3
Running training loop...
Epoch 1 complete
Epoch 2 complete
Epoch 3 complete
Training session for fraud_detector_v3 ended after 0.0000s (cleanup guaranteed)


**Output:**
```
Initializing training session for fraud_detector_v3
Running training loop...
Epoch 1 complete
Epoch 2 complete
Epoch 3 complete
Training session for fraud_detector_v3 ended after 0.0002s (cleanup guaranteed)
```

## 26.2 Database Connections for Feature/Label Retrieval

In [150]:
import sqlite3

with sqlite3.connect(":memory:") as conn:
    conn.execute("CREATE TABLE training_data (feature REAL, label INTEGER)")
    conn.executemany("INSERT INTO training_data VALUES (?, ?)", [(0.5, 1), (0.2, 0), (0.8, 1)])
    rows = conn.execute("SELECT feature, label FROM training_data").fetchall()
    print("Training rows fetched:", rows)

Training rows fetched: [(0.5, 1), (0.2, 0), (0.8, 1)]


**Output:**
```
Training rows fetched: [(0.5, 1), (0.2, 0), (0.8, 1)]
```

## 26.3 Temporary Model Checkpoint Files

In [151]:
import tempfile
import os

with tempfile.NamedTemporaryFile(suffix=".ckpt", delete=True) as ckpt_file:
    ckpt_file.write(b"fake_model_weights_bytes")
    ckpt_file.flush()
    print(f"Checkpoint temporarily saved at {ckpt_file.name}")
    print("Checkpoint size:", os.path.getsize(ckpt_file.name), "bytes")
print("Temporary checkpoint automatically cleaned up")

Checkpoint temporarily saved at /tmp/tmpj75mov0c.ckpt
Checkpoint size: 24 bytes
Temporary checkpoint automatically cleaned up


**Output:**
```
Checkpoint temporarily saved at /tmp/tmpabc123.ckpt
Checkpoint size: 24 bytes
Temporary checkpoint automatically cleaned up
```

## 26.4 Experiment Environments - Reproducible, Scoped Hyperparameters

In [152]:
from contextlib import contextmanager

active_hyperparameters = {"learning_rate": 0.01, "batch_size": 32}

@contextmanager
def experiment(**hyperparams):
    original = active_hyperparameters.copy()
    active_hyperparameters.update(hyperparams)
    print(f"Starting experiment with: {active_hyperparameters}")
    try:
        yield active_hyperparameters
    finally:
        active_hyperparameters.clear()
        active_hyperparameters.update(original)
        print("Hyperparameters reverted to baseline")

with experiment(learning_rate=0.001, batch_size=64):
    print("Training with experimental hyperparameters")

print("Back to baseline:", active_hyperparameters)

Starting experiment with: {'learning_rate': 0.001, 'batch_size': 64}
Training with experimental hyperparameters
Hyperparameters reverted to baseline
Back to baseline: {'learning_rate': 0.01, 'batch_size': 32}


**Output:**
```
Starting experiment with: {'learning_rate': 0.001, 'batch_size': 64}
Training with experimental hyperparameters
Hyperparameters reverted to baseline
Back to baseline: {'learning_rate': 0.01, 'batch_size': 32}
```

## 26.5 GPU/Resource Management Concepts

Actual GPU memory management is typically handled by deep learning framework internals (e.g., PyTorch/TensorFlow device contexts), but the same conceptual pattern applies directly: a context manager acquires a scarce resource (a GPU device, a fixed-size memory pool) and guarantees its release, even if training raises an exception partway through - for example, a conceptual device-scoping context manager:

In [153]:
from contextlib import contextmanager

class FakeGPUAllocator:
    def __init__(self):
        self.allocated = 0

    def allocate(self, mb):
        self.allocated += mb
        print(f"Allocated {mb}MB (total: {self.allocated}MB)")

    def free(self, mb):
        self.allocated -= mb
        print(f"Freed {mb}MB (total: {self.allocated}MB)")

@contextmanager
def gpu_memory(allocator, mb):
    allocator.allocate(mb)
    try:
        yield
    finally:
        allocator.free(mb)      # guaranteed release, even if training crashes mid-way

gpu = FakeGPUAllocator()
try:
    with gpu_memory(gpu, 2048):
        raise RuntimeError("Training crashed unexpectedly")
except RuntimeError as e:
    print("Caught:", e)

print("Final allocated memory:", gpu.allocated, "MB")

Allocated 2048MB (total: 2048MB)
Freed 2048MB (total: 0MB)
Caught: Training crashed unexpectedly
Final allocated memory: 0 MB


**Output:**
```
Allocated 2048MB (total: 2048MB)
Freed 2048MB (total: 0MB)
Caught: Training crashed unexpectedly
Final allocated memory: 0 MB
```

## 26.6 Logging Training Progress

In [154]:
import io
from contextlib import redirect_stdout

def train_model():
    for epoch in range(3):
        print(f"Epoch {epoch+1}: loss=0.{9-epoch}23")

log = io.StringIO()
with redirect_stdout(log):
    train_model()

with open("training_log.txt", "w") as f:
    f.write(log.getvalue())

print("Training log saved:")
print(log.getvalue())

Training log saved:
Epoch 1: loss=0.923
Epoch 2: loss=0.823
Epoch 3: loss=0.723



**Output:**
```
Training log saved:
Epoch 1: loss=0.923
Epoch 2: loss=0.823
Epoch 3: loss=0.723
```

## 26.7 Model Evaluation Scoping

In [155]:
from contextlib import contextmanager

@contextmanager
def evaluation_mode(model_state):
    was_training = model_state.get("training", True)
    model_state["training"] = False           # switch to eval mode
    print("Switched to evaluation mode")
    try:
        yield
    finally:
        model_state["training"] = was_training  # ALWAYS restore, regardless of eval outcome
        print("Restored to previous mode")

state = {"training": True}
with evaluation_mode(state):
    print("Running evaluation, training =", state["training"])
print("After evaluation, training =", state["training"])

Switched to evaluation mode
Running evaluation, training = False
Restored to previous mode
After evaluation, training = True


**Output:**
```
Switched to evaluation mode
Running evaluation, training = False
Restored to previous mode
After evaluation, training = True
```

## 26.8 Why Context Managers Improve Reliability in ML Workflows

- Training jobs often run for hours; a crash partway through should never leave GPU memory, database connections, or file handles leaked, since that could affect other jobs sharing the same machine.
- Experiment configuration scoping (Section 26.4) ensures hyperparameter sweeps never accidentally "leak" settings from one experiment into the next.
- Guaranteed mode-switching (Section 26.7) prevents a model from being accidentally left in evaluation mode after a crash during a metrics computation, which would silently corrupt subsequent training.

## 26.9 Summary Table

| ML Task | Context Manager Pattern |
|---|---|
| Training session timing/lifecycle | Custom `@contextmanager` wrapping the training loop |
| Feature/label retrieval | Database connection context manager |
| Temporary checkpoints | `tempfile.NamedTemporaryFile()` |
| Reproducible experiments | Custom context manager scoping hyperparameters |
| GPU/resource allocation | Custom context manager guaranteeing release on exit |
| Logging training progress | `redirect_stdout()` |
| Train/eval mode switching | Custom context manager restoring state in `finally` |

### Exercises - Section 26

**Beginner**
1. Write a `training_session()`-style context manager that prints start/end messages and total elapsed time.

**Intermediate**
2. Write an `experiment()` context manager (like Section 26.4) and run three different hyperparameter configurations, confirming settings always revert correctly.

**Advanced**
3. Design a context manager combining GPU memory allocation (Section 26.5) and evaluation-mode switching (Section 26.7) for a single evaluation step, ensuring both resources are correctly released/restored even if evaluation raises an exception.
# 27. Context Managers in AI Engineering

## 27.1 API Clients - Guaranteed Session Cleanup

Production AI applications commonly call external APIs (LLM providers, vector databases, embedding services). Wrapping the client connection in a context manager guarantees the underlying HTTP session is always closed, even if a request fails.

In [156]:
from contextlib import contextmanager

class FakeAPIClient:
    def __init__(self, api_key):
        self.api_key = api_key
        self.session_open = False

    def connect(self):
        self.session_open = True
        print("API session opened")

    def close(self):
        self.session_open = False
        print("API session closed")

    def generate(self, prompt):
        if not self.session_open:
            raise RuntimeError("Session not open")
        return f"Response to: {prompt}"

@contextmanager
def api_client(api_key):
    client = FakeAPIClient(api_key)
    client.connect()
    try:
        yield client
    finally:
        client.close()

with api_client("sk-fake-key") as client:
    print(client.generate("Summarize this document"))

API session opened
Response to: Summarize this document
API session closed


**Output:**
```
API session opened
Response to: Summarize this document
API session closed
```

## 27.2 Database Sessions in AI Applications

In [157]:
import sqlite3

with sqlite3.connect(":memory:") as conn:
    conn.execute("CREATE TABLE embeddings (id INTEGER, vector TEXT)")
    conn.execute("INSERT INTO embeddings VALUES (1, '[0.1, 0.2, 0.3]')")
    result = conn.execute("SELECT vector FROM embeddings WHERE id = 1").fetchone()
    print("Retrieved embedding:", result[0])

Retrieved embedding: [0.1, 0.2, 0.3]


**Output:**
```
Retrieved embedding: [0.1, 0.2, 0.3]
```

## 27.3 Temporary Resources in AI Pipelines

In [158]:
import tempfile

with tempfile.TemporaryDirectory() as tmp_dir:
    import os
    intermediate_path = os.path.join(tmp_dir, "chunked_input.txt")
    with open(intermediate_path, "w") as f:
        f.write("Chunked document content for embedding")
    print("Processing intermediate file:", os.path.exists(intermediate_path))
print("Pipeline temp files automatically cleaned up")

Processing intermediate file: True
Pipeline temp files automatically cleaned up


**Output:**
```
Processing intermediate file: True
Pipeline temp files automatically cleaned up
```

## 27.4 Full AI Pipeline Wrapping Multiple Resources

In [159]:
from contextlib import ExitStack, contextmanager

@contextmanager
def vector_db_connection():
    print("Connecting to vector database")
    yield "vector_db_handle"
    print("Closing vector database connection")

@contextmanager
def llm_client():
    print("Initializing LLM client")
    yield "llm_client_handle"
    print("Shutting down LLM client")

def run_rag_pipeline(query):
    with ExitStack() as stack:
        db = stack.enter_context(vector_db_connection())
        llm = stack.enter_context(llm_client())
        print(f"Retrieving context from {db} for query: {query}")
        print(f"Generating response using {llm}")
        return "Generated answer"

result = run_rag_pipeline("What is a context manager?")
print("Result:", result)

Connecting to vector database
Initializing LLM client
Retrieving context from vector_db_handle for query: What is a context manager?
Generating response using llm_client_handle
Shutting down LLM client
Closing vector database connection
Result: Generated answer


**Output:**
```
Connecting to vector database
Initializing LLM client
Retrieving context from vector_db_handle for query: What is a context manager?
Generating response using llm_client_handle
Shutting down LLM client
Closing vector database connection
```

## 27.5 Model Inference Resource Management

In [160]:
from contextlib import contextmanager

@contextmanager
def loaded_model(model_name):
    print(f"Loading {model_name} into memory")
    model = {"name": model_name, "loaded": True}
    try:
        yield model
    finally:
        model["loaded"] = False
        print(f"Unloading {model_name} from memory")

with loaded_model("sentiment-classifier-v2") as model:
    print(f"Running inference with {model['name']}")

Loading sentiment-classifier-v2 into memory
Running inference with sentiment-classifier-v2
Unloading sentiment-classifier-v2 from memory


**Output:**
```
Loading sentiment-classifier-v2 into memory
Running inference with sentiment-classifier-v2
Unloading sentiment-classifier-v2 from memory
```

## 27.6 Request Sessions (e.g., `requests.Session()`)

In [161]:
# Conceptual example using the popular 'requests' library's own context-manager support
import contextlib

class FakeSession:
    def __enter__(self):
        print("Session opened (connection pooling enabled)")
        return self
    def __exit__(self, *args):
        print("Session closed (connections released)")
        return False
    def get(self, url):
        return f"GET {url} -> 200 OK"

with FakeSession() as session:
    print(session.get("/api/v1/predict"))
    print(session.get("/api/v1/status"))

Session opened (connection pooling enabled)
GET /api/v1/predict -> 200 OK
GET /api/v1/status -> 200 OK
Session closed (connections released)


**Output:**
```
Session opened (connection pooling enabled)
GET /api/v1/predict -> 200 OK
GET /api/v1/status -> 200 OK
Session closed (connections released)
```

Using a single session (rather than creating a new connection per request) across multiple API calls is a significant performance optimization in production AI services, since it reuses underlying TCP connections instead of re-establishing them each time - and wrapping the session in a context manager guarantees it's cleaned up correctly even if one of the requests fails.

## 27.7 Logging in Production AI Pipelines

In [162]:
from contextlib import contextmanager
import time

@contextmanager
def request_logger(request_id):
    start = time.perf_counter()
    print(f"[{request_id}] Request started")
    try:
        yield
    except Exception as e:
        print(f"[{request_id}] Request failed: {e}")
        raise
    finally:
        elapsed = time.perf_counter() - start
        print(f"[{request_id}] Request finished in {elapsed:.4f}s")

with request_logger("req-001"):
    print("Processing inference request")

[req-001] Request started
Processing inference request
[req-001] Request finished in 0.0001s


**Output:**
```
[req-001] Request started
Processing inference request
[req-001] Request finished in 0.0001s
```

## 27.8 Resource Cleanup in Production AI Applications - Why It Matters More Here

Production AI services typically run as long-lived processes handling thousands of requests. Every unclosed API session, database connection, or unreleased model resource compounds over the service's uptime - precisely the "gradual degradation" failure pattern described in Part 3.7, but with the added complexity of external API rate limits and connection-pool quotas that can be exhausted far faster than local file descriptors.

## 27.9 Summary Table

| AI Engineering Concern | Context Manager Pattern |
|---|---|
| External API client lifecycle | Custom context manager wrapping connect/close |
| Vector DB / database sessions | Connection context manager (Part 22) |
| Pipeline intermediate files | `tempfile.TemporaryDirectory()` |
| Multi-resource pipelines | `ExitStack` dynamically combining several contexts |
| Model loading/unloading for inference | Custom context manager freeing memory in `finally` |
| HTTP request sessions | Session object's own context manager support |
| Per-request logging | Custom `@contextmanager` with `try/except/finally` |

### Exercises - Section 27

**Beginner**
1. Write an `api_client()`-style context manager (like Section 27.1) and use it to make two "requests" inside one `with` block.

**Intermediate**
2. Combine a database connection context manager and an API client context manager using `ExitStack`, and write a small function using both together.

**Advanced**
3. Design a `request_logger()`-style context manager that also captures and logs any exception that occurs during the request, without suppressing it, and test it with both a successful and a failing "request."
# 28. Asynchronous Context Managers

## 28.1 Why Asynchronous Resources Need Special Handling

Some setup/cleanup operations are themselves asynchronous - for example, opening a network connection using `async`/`await`-based I/O, where the connection or disconnection process itself involves awaiting other coroutines. The regular `__enter__`/`__exit__` protocol is fully synchronous and has no way to `await` anything inside it. Python therefore provides a parallel, asynchronous version of the protocol.

## 28.2 `async with`

`async with` is the asynchronous counterpart of `with`, used inside `async def` functions, for context managers whose setup/cleanup methods are themselves coroutines.

In [163]:
import asyncio

class AsyncResource:
    async def __aenter__(self):
        print("Async setup starting")
        await asyncio.sleep(0.1)      # simulates an async operation, e.g. a network handshake
        print("Async setup complete")
        return self

    async def __aexit__(self, exc_type, exc_value, traceback):
        print("Async cleanup starting")
        await asyncio.sleep(0.1)
        print("Async cleanup complete")
        return False

async def main():
    async with AsyncResource() as resource:
        print("Using the resource")

asyncio.run(main())

Async setup starting


Async setup complete
Using the resource
Async cleanup starting


Async cleanup complete


**Output:**
```
Async setup starting
Async setup complete
Using the resource
Async cleanup starting
Async cleanup complete
```

## 28.3 `__aenter__` and `__aexit__`

These are the asynchronous equivalents of `__enter__`/`__exit__`: both must be defined as `async def` methods, and both are `await`-ed automatically by the `async with` statement.

| Sync Protocol | Async Protocol |
|---|---|
| `__enter__(self)` | `async def __aenter__(self)` |
| `__exit__(self, exc_type, exc_value, traceback)` | `async def __aexit__(self, exc_type, exc_value, traceback)` |
| `with` | `async with` |
| Must appear inside | Any function | An `async def` function |

## 28.4 Why `async def` and `await` Are Required Here

`__aenter__` and `__aexit__` are coroutine functions (defined with `async def`), meaning calling them produces a coroutine object that must be `await`-ed to actually run and get a result - this is exactly why `async with` (not plain `with`) is required: only `async with` knows to `await` these methods correctly, and it can only be used inside another `async def` function, since `await` itself is only valid there.

In [164]:
class AsyncDatabaseConnection:
    async def __aenter__(self):
        await asyncio.sleep(0.05)     # simulates an async network handshake
        print("Async DB connection established")
        return self

    async def __aexit__(self, exc_type, exc_value, traceback):
        await asyncio.sleep(0.05)
        print("Async DB connection closed")
        return False

    async def query(self, sql):
        await asyncio.sleep(0.02)      # simulates an async query round-trip
        return f"Result of: {sql}"

async def main():
    async with AsyncDatabaseConnection() as db:
        result = await db.query("SELECT * FROM users")
        print(result)

asyncio.run(main())

Async DB connection established
Result of: SELECT * FROM users


Async DB connection closed


**Output:**
```
Async DB connection established
Result of: SELECT * FROM users
Async DB connection closed
```

## 28.5 Exception Handling in Async Context Managers

Exception behavior mirrors the synchronous protocol exactly: `__aexit__` receives the same `(exc_type, exc_value, traceback)` triple, and its return value suppresses or propagates the exception in exactly the same way as `__exit__`.

In [165]:
class AsyncResource:
    async def __aenter__(self):
        return self

    async def __aexit__(self, exc_type, exc_value, traceback):
        print(f"Async cleanup, exception occurred: {exc_type is not None}")
        return False       # never suppress

async def main():
    try:
        async with AsyncResource():
            raise ValueError("async failure")
    except ValueError as e:
        print("Caught:", e)

asyncio.run(main())

Async cleanup, exception occurred: True
Caught: async failure


**Output:**
```
Async cleanup, exception occurred: True
Caught: async failure
```

## 28.6 Async Suppression - `__aexit__` Returning `True`

In [166]:
class AsyncSuppressor:
    async def __aenter__(self):
        return self

    async def __aexit__(self, exc_type, exc_value, traceback):
        if exc_type is ValueError:
            print("Suppressing ValueError asynchronously")
            return True
        return False

async def main():
    async with AsyncSuppressor():
        raise ValueError("this gets suppressed")
    print("Execution continues normally")

asyncio.run(main())

Suppressing ValueError asynchronously
Execution continues normally


**Output:**
```
Suppressing ValueError asynchronously
Execution continues normally
```

## 28.7 Multiple and Nested Async Context Managers

Both the comma syntax and nesting work identically to the synchronous case, just with `async with`.

In [167]:
class Named:
    def __init__(self, name):
        self.name = name
    async def __aenter__(self):
        print(f"Entering {self.name}")
        return self
    async def __aexit__(self, *args):
        print(f"Exiting {self.name}")
        return False

async def main():
    async with Named("A") as a, Named("B") as b:
        print("Inside both")

asyncio.run(main())

Entering A
Entering B
Inside both
Exiting B
Exiting A


**Output:**
```
Entering A
Entering B
Inside both
Exiting B
Exiting A
```

## 28.8 A Practical Example - Async HTTP Client Session

In [168]:
class AsyncHTTPSession:
    async def __aenter__(self):
        await asyncio.sleep(0.02)
        print("HTTP session opened")
        return self

    async def __aexit__(self, exc_type, exc_value, traceback):
        await asyncio.sleep(0.02)
        print("HTTP session closed")
        return False

    async def get(self, url):
        await asyncio.sleep(0.01)
        return f"200 OK from {url}"

async def fetch_data():
    async with AsyncHTTPSession() as session:
        result = await session.get("https://api.example.com/data")
        print(result)

asyncio.run(fetch_data())

HTTP session opened
200 OK from https://api.example.com/data


HTTP session closed


**Output:**
```
HTTP session opened
200 OK from https://api.example.com/data
HTTP session closed
```

## 28.9 Summary Table

| Concept | Synchronous | Asynchronous |
|---|---|---|
| Statement | `with` | `async with` |
| Entry method | `__enter__(self)` | `async def __aenter__(self)` |
| Exit method | `__exit__(self, exc_type, exc_value, tb)` | `async def __aexit__(self, exc_type, exc_value, tb)` |
| Where usable | Any function | Only inside `async def` functions |
| Exception suppression | Return truthy from `__exit__` | Return truthy from `__aexit__` |
| Multiple/nested contexts | Comma syntax / nested `with` | Comma syntax / nested `async with` |

### Exercises - Section 28

**Beginner**
1. Write a minimal `async` context manager with print statements in `__aenter__` and `__aexit__`, and run it using `asyncio.run()`.

**Intermediate**
2. Reproduce Section 28.5's exception-handling example with your own async context manager and exception type.

**Advanced**
3. Build an `AsyncHTTPSession`-style context manager (Section 28.8) and use it inside a loop to make three simulated requests within a single session, timing the total duration.
# 29. `@asynccontextmanager`

## 29.1 Creating an Async Context Manager from an Async Generator

`contextlib.asynccontextmanager` is the asynchronous counterpart of `@contextmanager` (Part 12): it turns an `async def` generator function (one using `yield` exactly once) into a full async context manager, without writing a class with `__aenter__`/`__aexit__`.

In [169]:
from contextlib import asynccontextmanager
import asyncio

@asynccontextmanager
async def async_resource():
    print("Async setup")
    await asyncio.sleep(0.05)
    yield "async_value"
    await asyncio.sleep(0.05)
    print("Async cleanup")

async def main():
    async with async_resource() as value:
        print(f"Using {value}")

asyncio.run(main())

Async setup


Using async_value
Async cleanup


**Output:**
```
Async setup
Using async_value
Async cleanup
```

## 29.2 Exception Handling - `try`/`finally` Around `yield`, Just Like the Sync Version

Exactly the same lesson from Part 13 applies here: cleanup code after `yield` is skipped if the block raises an exception, unless it's protected by `try/finally`.

In [170]:
from contextlib import asynccontextmanager
import asyncio

@asynccontextmanager
async def safe_async_resource():
    print("Setup")
    try:
        yield
    finally:
        await asyncio.sleep(0.01)
        print("Cleanup - guaranteed, even on exception")

async def main():
    try:
        async with safe_async_resource():
            raise ValueError("async failure")
    except ValueError as e:
        print("Caught:", e)

asyncio.run(main())

Setup
Cleanup - guaranteed, even on exception
Caught: async failure


**Output:**
```
Setup
Cleanup - guaranteed, even on exception
Caught: async failure
```

## 29.3 Suppressing Exceptions in `@asynccontextmanager`

Identical pattern to the synchronous version: catch the specific exception around `yield` and don't re-raise it.

In [171]:
from contextlib import asynccontextmanager

@asynccontextmanager
async def suppress_value_errors():
    try:
        yield
    except ValueError as e:
        print(f"Suppressing: {e}")

async def main():
    async with suppress_value_errors():
        raise ValueError("this disappears")
    print("Execution continues normally")

asyncio.run(main())

Suppressing: this disappears
Execution continues normally


**Output:**
```
Suppressing: this disappears
Execution continues normally
```

## 29.4 Comparison: `@contextmanager` vs `@asynccontextmanager`

In [172]:
# SYNCHRONOUS version
from contextlib import contextmanager
import time

@contextmanager
def timer():
    start = time.perf_counter()
    yield
    print(f"Elapsed: {time.perf_counter() - start:.4f}s")

# ASYNCHRONOUS version - identical shape, async setup/cleanup
from contextlib import asynccontextmanager
import asyncio

@asynccontextmanager
async def async_timer():
    start = time.perf_counter()
    yield
    await asyncio.sleep(0)          # a placeholder for a genuinely async cleanup step
    print(f"Elapsed: {time.perf_counter() - start:.4f}s")

with timer():
    sum(range(100_000))

async def main():
    async with async_timer():
        sum(range(100_000))

asyncio.run(main())

Elapsed: 0.0012s
Elapsed: 0.0012s


**Output:**
```
Elapsed: 0.0028s
Elapsed: 0.0031s
```

## 29.5 Comparison Table

| Aspect | `@contextmanager` | `@asynccontextmanager` |
|---|---|---|
| Import | `from contextlib import contextmanager` | `from contextlib import asynccontextmanager` |
| Decorates | A regular generator function (`def` with `yield`) | An async generator function (`async def` with `yield`) |
| Used with | `with` | `async with` |
| Can `await` inside setup/cleanup | No | Yes |
| Exception handling | `try`/`finally` (or `except`) around `yield` | Identical - `try`/`finally` (or `except`) around `yield` |
| Typical use case | Simple synchronous setup/cleanup | Setup/cleanup involving async I/O (network, async DB drivers, `asyncio` primitives) |

## 29.6 A Realistic Async Example - Managing an Async Database Pool Connection

In [173]:
from contextlib import asynccontextmanager
import asyncio

class FakeAsyncPool:
    async def acquire(self):
        await asyncio.sleep(0.02)
        print("Connection acquired from pool")
        return "connection_handle"

    async def release(self, conn):
        await asyncio.sleep(0.02)
        print("Connection released back to pool")

pool = FakeAsyncPool()

@asynccontextmanager
async def pooled_connection():
    conn = await pool.acquire()
    try:
        yield conn
    finally:
        await pool.release(conn)

async def run_query():
    async with pooled_connection() as conn:
        print(f"Running query using {conn}")

asyncio.run(run_query())

Connection acquired from pool
Running query using connection_handle


Connection released back to pool


**Output:**
```
Connection acquired from pool
Running query using connection_handle
Connection released back to pool
```

## 29.7 Summary Table

| Concept | Detail |
|---|---|
| Decorator | `@asynccontextmanager` |
| Function type required | `async def` generator (one `yield`) |
| Statement used to enter | `async with` |
| Setup | Code before `yield` |
| Value bound by `as` | The value passed to `yield` |
| Cleanup | Code after `yield`, wrapped in `try/finally` for safety |
| Suppression | `except SpecificError:` around `yield`, without re-raising |

### Exercises - Section 29

**Beginner**
1. Rewrite Section 28's `AsyncHTTPSession` class-based context manager as an `@asynccontextmanager` generator function.

**Intermediate**
2. Write an `@asynccontextmanager` function that suppresses `TimeoutError` specifically, and test it against both a `TimeoutError` and a different exception type.

**Advanced**
3. Build a `pooled_connection()`-style async context manager (Section 29.6) that also logs how long the connection was held, using `time.perf_counter()` around the `yield`.
# 30. Context Managers in Web Applications

## 30.1 Application Startup/Shutdown - FastAPI's Lifespan Context Manager

FastAPI uses an `@asynccontextmanager` function directly as its application "lifespan" handler: code before `yield` runs once at startup, code after `yield` runs once at shutdown, guaranteeing cleanup even if the application terminates due to an error.

In [174]:
from contextlib import asynccontextmanager

# Conceptual illustration of FastAPI's lifespan pattern (not a runnable web server here)
@asynccontextmanager
async def lifespan(app):
    print("Application starting up - opening database pool")
    db_pool = "fake_db_pool"
    yield {"db_pool": db_pool}
    print("Application shutting down - closing database pool")

# FastAPI would be constructed as: app = FastAPI(lifespan=lifespan)
# and internally, FastAPI enters/exits this context manager around the app's entire running lifetime

This is a direct, large-scale application of the exact same setup/cleanup guarantee covered throughout this handbook - applied to an entire web application's lifetime instead of a single function call.

## 30.2 Request-Scoped Resources - Database Sessions Per Request

A very common web-framework pattern: acquire a fresh database session for each incoming request, and guarantee it's closed when the request finishes, regardless of whether the request succeeded or raised an exception.

In [175]:
from contextlib import contextmanager

class FakeDBSession:
    def __init__(self):
        print("Session opened for this request")
    def close(self):
        print("Session closed at end of request")
    def query(self, sql):
        return f"Result of: {sql}"

@contextmanager
def get_db_session():
    session = FakeDBSession()
    try:
        yield session
    finally:
        session.close()

def handle_request(request_path):
    with get_db_session() as db:
        print(db.query(f"SELECT * FROM data WHERE path = '{request_path}'"))

handle_request("/users/1")

Session opened for this request
Result of: SELECT * FROM data WHERE path = '/users/1'
Session closed at end of request


**Output:**
```
Session opened for this request
Result of: SELECT * FROM data WHERE path = '/users/1'
Session closed at end of request
```

Many web frameworks (Flask via extensions, FastAPI via dependency injection with `yield`) build exactly this "one context-managed resource per request" pattern directly into their request-handling machinery.

## 30.3 FastAPI-Style Dependencies Using `yield`

FastAPI's dependency injection system directly supports functions that `yield` a resource - internally, this is the same generator-based context-manager pattern from Part 12, applied to per-request dependency setup/teardown.

In [176]:
# Conceptual illustration of a FastAPI-style yield dependency
def get_db():
    db = FakeDBSession()
    try:
        yield db          # the request handler receives 'db' here
    finally:
        db.close()          # runs automatically after the request completes

# Usage in a route (conceptual):
# @app.get("/items")
# def read_items(db = Depends(get_db)):
#     return db.query("SELECT * FROM items")

## 30.4 API Client Sessions in Backend Services

In [177]:
from contextlib import contextmanager

class ExternalAPIClient:
    def __enter__(self):
        print("Connecting to external payment API")
        return self
    def __exit__(self, *args):
        print("Disconnecting from external payment API")
        return False
    def charge(self, amount):
        return f"Charged ${amount}"

def process_payment(amount):
    with ExternalAPIClient() as client:
        return client.charge(amount)

print(process_payment(49.99))

Connecting to external payment API
Disconnecting from external payment API
Charged $49.99


**Output:**
```
Connecting to external payment API
Charged $49.99
Disconnecting from external payment API
```

## 30.5 Flask-Style Application Context (Conceptual)

Flask uses its own application and request context objects, which are also fundamentally context-manager-based, guaranteeing resources tied to a request (like a database connection stashed on the request-local `g` object) are cleaned up automatically at the end of each request via `teardown` hooks - conceptually the same "acquire, use, guaranteed release" pattern, just integrated into the framework's own request lifecycle rather than an explicit `with` block in each view function.

## 30.6 Why This Matters at Scale

Web applications handle many concurrent requests; a resource leak that would be a minor annoyance in a single script becomes a severe production issue at scale - a single unclosed database session per request, multiplied across thousands of requests per minute, can exhaust a connection pool within minutes and take down an entire service. Context managers, applied consistently at the request-handling layer, are one of the most effective structural defenses against this class of failure.

## 30.7 Summary Table

| Web Application Concern | Context Manager Pattern |
|---|---|
| App-wide startup/shutdown | `@asynccontextmanager`-based "lifespan" handler (FastAPI) |
| Per-request database session | `@contextmanager` or `yield`-based dependency, scoped to one request |
| External API client sessions | Class-based or `@contextmanager`-based session wrapper |
| Framework-managed contexts | Flask's app/request context, FastAPI's `Depends(..., yield ...)` |
| Why it matters at scale | A per-request leak becomes a pool-exhausting outage under load |

### Exercises - Section 30

**Beginner**
1. Write a `get_db_session()`-style context manager (Section 30.2) and use it inside a function simulating handling two separate "requests."

**Intermediate**
2. Write a conceptual `lifespan()` async context manager (Section 30.1) that opens and closes both a database pool and an API client, using `ExitStack` or `AsyncExitStack` internally.

**Advanced**
3. Explain, in your own words, why a per-request resource leak is a more severe problem in a web application than in a one-off script, referencing Section 30.6.
# 31. Context Managers and Decorators

## 31.1 The Conceptual Difference

**Decorators** (covered in depth in the separate Decorators reference) wrap **behavior around a function call** - they modify what happens each time a specific function is invoked. **Context managers** wrap **behavior around a block of code's lifetime** - they modify what happens around a scoped region of code, regardless of which function(s) that code happens to live in.

| Aspect | Decorator | Context Manager |
|---|---|---|
| Wraps | A function definition | A block of code (any block, anywhere) |
| Applied via | `@decorator` above a function | `with cm():` around a block |
| Reused by | Calling the decorated function | Writing another `with` block |
| Scope of effect | Every call to that specific function | Exactly the one block it wraps |

In [178]:
# Decorator - wraps FUNCTION CALLS
def logged(func):
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        result = func(*args, **kwargs)
        print(f"{func.__name__} finished")
        return result
    return wrapper

@logged
def add(a, b):
    return a + b

print(add(2, 3))

Calling add
add finished
5


**Output:**
```
Calling add
add finished
5
```

In [179]:
# Context manager - wraps a BLOCK OF CODE, not tied to any one function
from contextlib import contextmanager

@contextmanager
def logged_block(label):
    print(f"Starting {label}")
    yield
    print(f"Finished {label}")

with logged_block("addition"):
    result = 2 + 3
print(result)

Starting addition
Finished addition
5


**Output:**
```
Starting addition
Finished addition
5
```

## 31.2 This Is a Useful Distinction, Not an Absolute Rule

The clean "function calls vs. code blocks" distinction is genuinely useful for understanding *why* each tool exists, but it is not an ironclad boundary - several important nuances blur the line, covered next.

## 31.3 `contextlib.ContextDecorator` - A Context Manager That's Also a Decorator

Any class-based context manager can be made usable as a **decorator too**, by inheriting from `contextlib.ContextDecorator`. When used as a decorator, it wraps the *entire function body* in its own `with` context automatically, every time the function is called.

In [180]:
from contextlib import ContextDecorator

class Timed(ContextDecorator):
    def __enter__(self):
        import time
        self.start = time.perf_counter()
        return self
    def __exit__(self, *args):
        import time
        print(f"Elapsed: {time.perf_counter() - self.start:.4f}s")
        return False

# Used as a context manager (the "normal" way):
with Timed():
    sum(range(100_000))

# Used as a DECORATOR (thanks to inheriting from ContextDecorator):
@Timed()
def compute():
    return sum(range(100_000))

compute()

Elapsed: 0.0012s
Elapsed: 0.0013s


4999950000

**Output:**
```
Elapsed: 0.0025s
Elapsed: 0.0026s
```

## 31.4 `@contextmanager`-Decorated Functions Are Also Usable as Decorators

Because of `ContextDecorator`'s design, any `@contextmanager`-decorated generator function can *also* be directly used as a function decorator, with zero extra code - this is one of the more surprising and useful pieces of overlap between the two tools.

In [181]:
from contextlib import contextmanager
import time

@contextmanager
def timed(label):
    start = time.perf_counter()
    yield
    print(f"[{label}] took {time.perf_counter() - start:.4f}s")

# Used as a context manager:
with timed("manual block"):
    sum(range(500_000))

# Used as a DECORATOR - notice timed(...) is called, then applied with @
@timed("decorated function")
def compute_sum():
    return sum(range(500_000))

compute_sum()

[manual block] took 0.0068s
[decorated function] took 0.0060s


124999750000

**Output:**
```
[manual block] took 0.0141s
[decorated function] took 0.0139s
```

## 31.5 When to Use Each

| Situation | Prefer |
|---|---|
| The same setup/cleanup should wrap **every call** to one specific, named function | Decorator (or a `ContextDecorator`) |
| The setup/cleanup should wrap **one specific block**, possibly spanning multiple statements, and possibly reused in different places with different surrounding code | Context manager (`with`) |
| You want the flexibility to use the *same* logic **both** ways, depending on the call site | `ContextDecorator` or `@contextmanager` (which supports both automatically) |

## 31.6 Situations Where Both Are Used Together

In [182]:
from contextlib import contextmanager
import time

@contextmanager
def timed(label):
    start = time.perf_counter()
    try:
        yield
    finally:
        print(f"[{label}] {time.perf_counter() - start:.4f}s")

def retry(times):                     # a genuine decorator, unrelated to context managers
    def decorator(func):
        def wrapper(*args, **kwargs):
            for attempt in range(times):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"Attempt {attempt + 1} failed: {e}")
            raise RuntimeError("All retries failed")
        return wrapper
    return decorator

@retry(times=2)
def flaky_operation():
    with timed("flaky_operation body"):
        raise ValueError("simulated failure")

try:
    flaky_operation()
except RuntimeError as e:
    print("Final failure:", e)

[flaky_operation body] 0.0000s
Attempt 1 failed: simulated failure
[flaky_operation body] 0.0000s
Attempt 2 failed: simulated failure
Final failure: All retries failed


**Output:**
```
[flaky_operation body] 0.0001s
Attempt 1 failed: simulated failure
[flaky_operation body] 0.0001s
Attempt 2 failed: simulated failure
Final failure: All retries failed
```

Here, a **decorator** (`@retry`) manages retry behavior across function *calls*, while a **context manager** (`timed`) manages timing around a specific *block* inside the function body - each tool doing the job it's conceptually best suited for, composed together cleanly.

## 31.7 Summary Table

| Concept | Decorator | Context Manager | Both (`ContextDecorator` / `@contextmanager`) |
|---|---|---|---|
| Primary unit wrapped | Function | Code block | Either, depending on usage |
| Syntax | `@decorator` | `with cm():` | Both `@cm()` and `with cm():` work |
| Typical use | Logging, retries, caching, access control around a function | Resource setup/cleanup around a scoped block | Timing, logging that makes sense either way |

### Exercises - Section 31

**Beginner**
1. Write a `@contextmanager`-based function and use it both as a `with` block and as a decorator on a small function.

**Intermediate**
2. Write a `ContextDecorator` subclass that logs entry/exit messages, and apply it to two different functions.

**Advanced**
3. Combine a genuine decorator (e.g., a simple `@retry` decorator) with a `@contextmanager`-based timing block inside the decorated function, as in Section 31.6, and explain why each tool was the right choice for its specific job.
# 32. Context Managers and OOP

This section connects context managers to object-oriented programming concepts (covered fully in the separate OOP reference) without repeating that material - the focus here is specifically on how the context manager protocol *is* OOP, in action.

## 32.1 Context Managers Are Just Classes with Two Special Methods

Every class-based context manager in this guide has been an ordinary Python class, made usable with `with` purely by implementing two specific dunder (special) methods: `__enter__` and `__exit__`. There is no separate "context manager" type in Python - it is simply a **protocol**: any class satisfying it automatically works with `with`, exactly as any class defining `__len__` automatically works with `len()`.

In [183]:
class AnyClassCanBeAContextManager:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        return False

with AnyClassCanBeAContextManager():
    print("Works - no special base class or registration needed")

Works - no special base class or registration needed


**Output:**
```
Works - no special base class or registration needed
```

## 32.2 Encapsulation Applied to Resource Management

A well-designed context manager class **encapsulates** everything about a resource's lifecycle inside itself: the resource handle is typically stored as a "protected" instance attribute, and external code interacts with it only through the object's public interface, never needing to know internally how or when setup/cleanup actually happens.

In [184]:
class DatabaseConnection:
    def __init__(self, host):
        self._host = host        # protected, per OOP naming conventions
        self._conn = None

    def __enter__(self):
        self._conn = f"connection_to_{self._host}"
        print(f"Connected: {self._conn}")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Disconnected: {self._conn}")
        self._conn = None
        return False

    def query(self, sql):
        return f"Result of '{sql}' via {self._conn}"

with DatabaseConnection("prod-db") as db:
    print(db.query("SELECT 1"))

Connected: connection_to_prod-db
Result of 'SELECT 1' via connection_to_prod-db
Disconnected: connection_to_prod-db


**Output:**
```
Connected: connection_to_prod-db
Result of 'SELECT 1' via connection_to_prod-db
Disconnected: connection_to_prod-db
None
```

## 32.3 Object Lifecycle vs Context Manager Lifecycle - A Key Distinction

An object's general lifecycle (`__new__`, `__init__`, usage, eventual `__del__`/garbage collection) is entirely separate from a context manager's `__enter__`/`__exit__` lifecycle. A single object instance can be **reused** across multiple `with` blocks (entered and exited more than once) without being recreated each time - this distinction becomes especially important in Part 33-34's discussion of reusable versus single-use context managers.

In [185]:
class ReusableTimer:
    def __enter__(self):
        import time
        self._start = time.perf_counter()
        return self
    def __exit__(self, *args):
        import time
        print(f"Block took {time.perf_counter() - self._start:.4f}s")
        return False

timer = ReusableTimer()          # ONE object, created ONCE

with timer:                       # entered and exited once
    sum(range(100_000))

with timer:                        # the SAME object, entered and exited AGAIN
    sum(range(200_000))

Block took 0.0013s
Block took 0.0026s


**Output:**
```
Block took 0.0025s
Block took 0.0051s
```

## 32.4 Inheritance with Context Managers

Context manager classes can be subclassed just like any other class, letting a subclass extend or override setup/cleanup behavior while reusing the parent's core logic (via `super()`, as covered in the OOP reference).

In [186]:
class BasicConnection:
    def __enter__(self):
        print("Basic connection opened")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print("Basic connection closed")
        return False

class LoggingConnection(BasicConnection):
    def __enter__(self):
        print("Logging: about to connect")
        return super().__enter__()          # reuse the parent's setup logic

    def __exit__(self, exc_type, exc_value, traceback):
        print("Logging: about to disconnect")
        return super().__exit__(exc_type, exc_value, traceback)

with LoggingConnection():
    print("Using the connection")

Logging: about to connect
Basic connection opened
Using the connection
Logging: about to disconnect
Basic connection closed


**Output:**
```
Logging: about to connect
Basic connection opened
Using the connection
Logging: about to disconnect
Basic connection closed
```

## 32.5 Composition - A Context Manager Composed of Other Context Managers

Just as OOP favors composition for combining independent behaviors (rather than deep inheritance chains), a context manager class can internally **compose** other context managers, delegating to them rather than reimplementing their logic.

In [187]:
class TransactionalSession:
    def __init__(self, connection_cm, transaction_cm):
        self._connection_cm = connection_cm     # composed - not inherited
        self._transaction_cm = transaction_cm

    def __enter__(self):
        self._conn = self._connection_cm.__enter__()
        self._transaction_cm.__enter__()
        return self._conn

    def __exit__(self, exc_type, exc_value, traceback):
        self._transaction_cm.__exit__(exc_type, exc_value, traceback)
        return self._connection_cm.__exit__(exc_type, exc_value, traceback)

(In practice, `ExitStack` from Part 18 is almost always a cleaner way to compose multiple context managers than manually calling `__enter__`/`__exit__` like this - this example exists purely to make the OOP composition connection explicit.)

## 32.6 Summary Table

| OOP Concept | How It Applies to Context Managers |
|---|---|
| Classes and special methods | `__enter__`/`__exit__` are just two more dunder methods, like `__len__` or `__str__` |
| Encapsulation | Resource handles are typically stored as protected instance attributes |
| Object lifecycle vs context lifecycle | A single object can be entered/exited multiple times, independent of its own construction/destruction |
| Inheritance | Subclasses can extend setup/cleanup behavior via `super()` |
| Composition | A context manager can be built from other context managers, favoring composition over inheritance where appropriate |

### Exercises - Section 32

**Beginner**
1. Write a simple context manager class, then create one instance and use it in two separate `with` blocks, confirming it works both times.

**Intermediate**
2. Write a subclass of a context manager (like `LoggingConnection` above) that adds behavior to both `__enter__` and `__exit__` using `super()`.

**Advanced**
3. Design a composed context manager (like `TransactionalSession`) that internally uses `ExitStack` rather than manual `__enter__`/`__exit__` calls, and explain why this is generally the safer, more idiomatic approach.
# 33. Advanced Context Manager Design

## 33.1 Reusable Context Managers

A **reusable** context manager is one whose *same instance* can be entered and exited more than once (across separate, non-overlapping `with` blocks), as already shown in Part 32.3. Not every context manager is safely reusable - it depends entirely on whether `__enter__`/`__exit__` correctly reset any internal state each time.

In [188]:
class ReusableCounter:
    def __init__(self):
        self.uses = 0

    def __enter__(self):
        self.uses += 1
        print(f"Entering (use #{self.uses})")
        return self

    def __exit__(self, *args):
        print(f"Exiting (use #{self.uses})")
        return False

counter = ReusableCounter()
with counter:
    pass
with counter:          # reused safely - state (self.uses) correctly accumulates
    pass

Entering (use #1)
Exiting (use #1)
Entering (use #2)
Exiting (use #2)


**Output:**
```
Entering (use #1)
Exiting (use #1)
Entering (use #2)
Exiting (use #2)
```

## 33.2 Stateful Context Managers

A **stateful** context manager retains and uses information across its own `__enter__`/`__exit__` calls, and often across multiple uses. State must be managed carefully, especially for reusable context managers, to avoid old state from a previous use leaking incorrectly into a new one.

In [189]:
class StatefulTransaction:
    def __init__(self):
        self.operations = []

    def __enter__(self):
        self.operations = []          # IMPORTANT: reset state on each new entry
        return self

    def add(self, op):
        self.operations.append(op)

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            print(f"Committing: {self.operations}")
        else:
            print(f"Rolling back: {self.operations}")
        return False

tx = StatefulTransaction()
with tx:
    tx.add("INSERT row 1")
    tx.add("INSERT row 2")

with tx:                      # reused - state correctly starts fresh, not carrying over old operations
    tx.add("UPDATE row 3")

Committing: ['INSERT row 1', 'INSERT row 2']
Committing: ['UPDATE row 3']


**Output:**
```
Committing: ['INSERT row 1', 'INSERT row 2']
Committing: ['UPDATE row 3']
```

**The mistake this avoids:** forgetting to reset `self.operations` in `__enter__` would cause the second use to incorrectly include leftover operations from the first use.

## 33.3 Context Manager Factories

A **context manager factory** is a plain function that returns a *new* context manager instance each time it's called, configured according to its arguments - useful for parameterizing context managers cleanly without requiring callers to know the underlying class name.

In [190]:
def timer(label="operation"):
    from contextlib import contextmanager
    import time

    @contextmanager
    def _timer():
        start = time.perf_counter()
        yield
        print(f"[{label}] {time.perf_counter() - start:.4f}s")

    return _timer()

with timer("data load"):
    sum(range(100_000))

with timer("aggregation"):
    sum(range(500_000))

[data load] 0.0013s


[aggregation] 0.0078s


**Output:**
```
[data load] 0.0025s
[aggregation] 0.0128s
```

## 33.4 Parameterized Context Managers

Most real-world context managers accept parameters through `__init__` (class-based) or through the outer function's arguments (`@contextmanager`-based), configuring their behavior per use - every custom context manager class in this guide (`DatabaseConnection(host)`, `SuppressSpecific(*types)`) is already an example of this pattern.

In [191]:
from contextlib import contextmanager

@contextmanager
def retryable_operation(max_attempts=3):
    attempt = 0
    while True:
        attempt += 1
        try:
            yield attempt
            break
        except Exception as e:
            if attempt >= max_attempts:
                print(f"Giving up after {attempt} attempts")
                raise
            print(f"Attempt {attempt} failed: {e}, retrying")

# NOTE: this simplified example illustrates the parameterization concept;
# a production version would need more careful control flow around the yield/loop interaction.

## 33.5 Dynamic Context Management - Choosing Contexts at Runtime

`ExitStack` (Part 18) is the primary tool for genuinely dynamic context management (an unknown number of resources, decided at runtime), but simpler cases can also select between different pre-defined context managers based on a runtime condition.

In [192]:
from contextlib import nullcontext

def get_lock_context(use_lock, lock):
    return lock if use_lock else nullcontext()

import threading
lock = threading.Lock()

with get_lock_context(True, lock):
    print("Locked execution")

with get_lock_context(False, lock):
    print("Unlocked execution")

Locked execution
Unlocked execution


**Output:**
```
Locked execution
Unlocked execution
```

## 33.6 Composable Context Managers

Context managers can be designed to be **composed** - combined together to build more complex behavior from simpler, independent pieces - most cleanly using `ExitStack` (Part 18) rather than manual nesting or nested nesting or inheritance.

In [193]:
from contextlib import contextmanager, ExitStack

@contextmanager
def logging_context(label):
    print(f"[{label}] start")
    yield
    print(f"[{label}] end")

@contextmanager
def timing_context(label):
    import time
    start = time.perf_counter()
    yield
    print(f"[{label}] took {time.perf_counter() - start:.4f}s")

def run_with_full_observability(label, work_func):
    with ExitStack() as stack:
        stack.enter_context(logging_context(label))
        stack.enter_context(timing_context(label))
        return work_func()

result = run_with_full_observability("computation", lambda: sum(range(100_000)))
print("Result:", result)

[computation] start
[computation] took 0.0013s
[computation] end
Result: 4999950000


**Output:**
```
[computation] start
[computation] took 0.0026s
[computation] end
Result: 4999950000
```

## 33.7 Important Design Considerations

- **Always reset relevant internal state in `__enter__`** if the context manager is meant to be reusable (Section 33.2).
- **Document reusability and reentrancy explicitly** (Part 34) - never assume callers will guess correctly whether an instance can be reused or nested.
- **Prefer composition (via `ExitStack` or explicit delegation) over deep inheritance** when combining multiple independent pieces of setup/cleanup behavior.
- **Keep each context manager focused on one resource or one concern** - a context manager trying to manage five unrelated resources at once is a sign it should be split into several, composed together.

## 33.8 Summary Table

| Design Concept | Key Idea |
|---|---|
| Reusable | Same instance, entered/exited more than once, in separate (non-overlapping) blocks |
| Stateful | Retains data across `__enter__`/`__exit__`; must reset appropriately if reusable |
| Factory | A function producing a fresh, configured context manager instance per call |
| Parameterized | Configured via `__init__` arguments or an outer function's parameters |
| Dynamic | Chosen or combined at runtime, often via `ExitStack` |
| Composable | Built from smaller, independent context managers, ideally via `ExitStack` |

### Exercises - Section 33

**Beginner**
1. Write a reusable context manager (like `ReusableCounter`) and use the same instance in three separate `with` blocks.

**Intermediate**
2. Write a context manager factory function (like `timer(label)`) that returns a differently configured context manager depending on its argument.

**Advanced**
3. Build a composable observability wrapper (like Section 33.6) combining at least three independent `@contextmanager` functions via `ExitStack`, and test it around two different pieces of work.
# 34. Reentrant Context Managers

## 34.1 What Reentrancy Means

A context manager is **reentrant** if the *same instance* can be safely entered again **while it is already active** - that is, nested inside its own `with` block - without breaking. This is a stronger, more specific property than merely being "reusable" (Part 33.1), which only guarantees safe reuse across *separate, non-overlapping* `with` blocks.

In [194]:
import threading

lock = threading.RLock()          # Reentrant Lock - specifically designed for this

with lock:
    print("Outer acquisition")
    with lock:                      # the SAME lock instance, entered again WHILE already held
        print("Inner (reentrant) acquisition - safe with RLock")
    print("Inner released, outer still held")
print("Outer released")

Outer acquisition
Inner (reentrant) acquisition - safe with RLock
Inner released, outer still held
Outer released


**Output:**
```
Outer acquisition
Inner (reentrant) acquisition - safe with RLock
Inner released, outer still held
Outer released
```

## 34.2 Reusable vs Reentrant vs Single-Use - The Precise Distinctions

| Property | Definition | Example |
|---|---|---|
| **Single-use** | Can be entered exactly once, ever; a second use is broken or raises an error | A generator-based `@contextmanager` context manager re-entered a second time (Section 34.4) |
| **Reusable** | Can be entered multiple times, but only in **separate, sequential, non-overlapping** `with` blocks | `ReusableCounter` from Part 33.1 |
| **Reentrant** | Can be entered again **while already active** (nested inside its own block) | `threading.RLock()` |

Every reentrant context manager is necessarily also reusable (if it can be nested inside itself, it can certainly also be used sequentially), but the reverse is not true - many perfectly reusable context managers are **not** safely reentrant.

## 34.3 A Reusable but NOT Reentrant Example

In [195]:
import threading

plain_lock = threading.Lock()          # a REGULAR Lock - reusable, but NOT reentrant

with plain_lock:
    print("Outer acquisition succeeded")
    # Attempting to re-enter the SAME plain Lock here would DEADLOCK the current thread
    # (demonstrating this directly would hang the example, so it is described rather than run)
print("Outer released")

# plain_lock IS reusable across separate blocks:
with plain_lock:
    print("Reused safely in a separate, later block")

Outer acquisition succeeded


Outer released
Reused safely in a separate, later block


**Output:**
```
Outer acquisition succeeded
Outer released
Reused safely in a separate, later block
```

## 34.4 Generator-Based (`@contextmanager`) Context Managers Are Single-Use Per Call

Each call to a `@contextmanager`-decorated function produces a **new** generator object, which can only be driven through `__enter__`/`__exit__` **once** - attempting to reuse that *same generator object* a second time raises an error, because a generator cannot be restarted after finishing.

In [196]:
from contextlib import contextmanager

@contextmanager
def once_only():
    print("Setup")
    yield
    print("Cleanup")

cm = once_only()           # ONE generator-backed context manager object

with cm:
    print("First use")

try:
    with cm:                  # attempting to reuse the SAME already-exhausted generator object
        print("Second use")
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Setup
First use
Cleanup
Error: AttributeError: '_GeneratorContextManager' object has no attribute 'args'


**Output:**
```
Setup
First use
Cleanup
Error: AttributeError: '_GeneratorContextManager' object has no attribute 'args'
```

(The exact exception type/message is a CPython implementation detail that has changed across Python versions - older versions raised `RuntimeError: generator didn't yield` instead. Either way, the underlying lesson is identical: the same generator-backed context manager object cannot be driven through the protocol twice.)

**The correct fix:** call the factory function again to get a **fresh** context manager instance for each use, rather than reusing the same one.

In [197]:
with once_only():          # a NEW call, a NEW generator, works fine
    print("First use (fresh instance)")

with once_only():           # ANOTHER new call, ANOTHER new generator, also works fine
    print("Second use (fresh instance)")

Setup
First use (fresh instance)
Cleanup
Setup
Second use (fresh instance)
Cleanup


**Output:**
```
Setup
First use (fresh instance)
Cleanup
Setup
Second use (fresh instance)
Cleanup
```

## 34.5 Why Not Every Context Manager Should Be Assumed Reentrant

Assuming reentrancy without verifying it is a common source of subtle bugs - particularly with recursive functions or nested helper calls that might, unexpectedly, re-enter the same resource's context.

In [198]:
class NonReentrantResource:
    def __init__(self):
        self._active = False

    def __enter__(self):
        if self._active:
            raise RuntimeError("Cannot re-enter - this context manager is not reentrant")
        self._active = True
        print("Entered")
        return self

    def __exit__(self, *args):
        self._active = False
        print("Exited")
        return False

resource = NonReentrantResource()

try:
    with resource:
        with resource:            # re-entering the SAME instance while already active
            pass
except RuntimeError as e:
    print("Error:", e)

Entered


Exited
Error: Cannot re-enter - this context manager is not reentrant


**Output:**
```
Entered
Error: Cannot re-enter - this context manager is not reentrant
```

This explicit `self._active` guard is a common, deliberate defensive pattern: rather than silently misbehaving if re-entered, the context manager fails loudly and immediately, making the bug obvious rather than allowing corrupted state to propagate quietly.

## 34.6 Designing for Reentrancy When It's Genuinely Needed

If a context manager genuinely needs to support nested self-re-entry (e.g., a logging-indentation context manager that legitimately nests), design it explicitly with a counter or stack, similar to how `RLock` internally tracks an acquisition count per thread.

In [199]:
class ReentrantIndentLogger:
    def __init__(self):
        self.depth = 0

    def __enter__(self):
        print("  " * self.depth + "Entering")
        self.depth += 1
        return self

    def __exit__(self, *args):
        self.depth -= 1
        print("  " * self.depth + "Exiting")
        return False

logger = ReentrantIndentLogger()
with logger:
    print("  " * logger.depth + "Doing outer work")
    with logger:                     # deliberately designed to support this
        print("  " * logger.depth + "Doing inner work")

Entering
  Doing outer work
  Entering
    Doing inner work
  Exiting
Exiting


**Output:**
```
Entering
  Doing outer work
  Entering
    Doing inner work
  Exiting
Exiting
```

## 34.7 Summary Table

| Property | Can Reuse Across Separate Blocks? | Can Nest Inside Itself? | Example |
|---|---|---|---|
| Single-use | No | No | A `@contextmanager` generator object, used a second time |
| Reusable (not reentrant) | Yes | No | `threading.Lock()`, most simple custom classes without an explicit reentrancy guard |
| Reentrant | Yes | Yes | `threading.RLock()`, a deliberately-designed indent-tracking logger |

### Exercises - Section 34

**Beginner**
1. Attempt to reuse the same `@contextmanager`-produced object twice (like Section 34.4) and observe the resulting error firsthand.

**Intermediate**
2. Add an explicit `self._active` reentrancy guard (like Section 34.5) to a context manager class of your own design, and confirm it raises a clear error on accidental self-nesting.

**Advanced**
3. Design a deliberately reentrant context manager (like `ReentrantIndentLogger`) for a different use case of your choosing, and explain what internal state makes the reentrancy safe.
# 35. Context Variables (`contextvars`)

## 35.1 What Context-Local State Means

**Context-local state** is data that is scoped to a particular logical "context" of execution - most importantly, in `asyncio`-based code, scoped to a specific **task**, rather than to the whole program (like a global variable) or to a specific **thread** (like thread-local storage). This distinction matters enormously in async code, where many concurrent tasks run interleaved on a single thread - thread-local storage would incorrectly treat all of them as sharing the same "thread" state.

## 35.2 `ContextVar` - Declaring a Context Variable

In [200]:
from contextvars import ContextVar

request_id: ContextVar[str] = ContextVar("request_id", default="unset")

print(request_id.get())          # reads the current value in this context

unset


**Output:**
```
unset
```

## 35.3 `set()` and `get()`

In [201]:
from contextvars import ContextVar

request_id = ContextVar("request_id", default="unset")

print("Before:", request_id.get())
token = request_id.set("req-12345")
print("After set:", request_id.get())

Before: unset
After set: req-12345


**Output:**
```
Before: unset
After set: req-12345
```

## 35.4 Tokens and `reset()`

`.set()` returns a **token**, which can be passed to `.reset()` to restore the variable to its value *before* that specific `.set()` call - precisely the same "temporarily change, then restore" pattern used throughout Part 24, but built into the language for context-local state specifically.

In [202]:
from contextvars import ContextVar

request_id = ContextVar("request_id", default="unset")

token = request_id.set("req-001")
print("During:", request_id.get())

request_id.reset(token)
print("After reset:", request_id.get())

During: req-001
After reset: unset


**Output:**
```
During: req-001
After reset: unset
```

## 35.5 Using `ContextVar` as a Context Manager Pattern

Combining `ContextVar` with `@contextmanager` produces exactly the "scoped, auto-restoring" behavior that fits naturally into this handbook's core theme.

In [203]:
from contextvars import ContextVar
from contextlib import contextmanager

request_id: ContextVar[str] = ContextVar("request_id", default="unset")

@contextmanager
def request_context(rid):
    token = request_id.set(rid)
    try:
        yield
    finally:
        request_id.reset(token)     # guaranteed restoration, same principle as Part 24

def log(message):
    print(f"[{request_id.get()}] {message}")

log("before any request")
with request_context("req-abc"):
    log("handling request")
log("after the request")

[unset] before any request
[req-abc] handling request
[unset] after the request


**Output:**
```
[unset] before any request
[req-abc] handling request
[unset] after the request
```

## 35.6 Why `contextvars` Is Different from Ordinary Global Variables

An ordinary global (module-level) variable is shared by the **entire process** - every thread and every async task sees the exact same value at all times, with no built-in scoping mechanism. A `ContextVar` set inside one async task is invisible to other concurrently running tasks, even though they may all be executing "at the same time" on the same thread.

In [204]:
import asyncio
from contextvars import ContextVar

user_id: ContextVar[str] = ContextVar("user_id", default="anonymous")

async def handle_request(uid):
    user_id.set(uid)
    await asyncio.sleep(0.05)          # simulates concurrent work interleaving with other tasks
    print(f"Handling request for user: {user_id.get()}")

async def main():
    await asyncio.gather(
        handle_request("alice"),
        handle_request("bob"),
    )

asyncio.run(main())

Handling request for user: alice
Handling request for user: bob


**Output:**
```
Handling request for user: alice
Handling request for user: bob
```

Even though both `handle_request` calls run concurrently and interleave their `await` points on the same thread, each one correctly sees only its **own** `user_id` value - this would not be true of a plain global variable, which both tasks would share and overwrite, likely causing one request to see the other's user ID.

## 35.7 Why `contextvars` Is Different from Thread-Local Storage

`threading.local()` scopes data per **thread** - every async task sharing the same thread (which is the normal situation in `asyncio`, since a single thread typically runs an entire event loop with many concurrent tasks) would incorrectly share the same thread-local value. `contextvars` was specifically introduced to solve this exact gap, correctly scoping data per **task** (technically, per `asyncio` "Context," which is automatically copied for each new task), not per thread.

In [205]:
# Conceptual comparison (not directly runnable as a side-by-side demo, since it requires
# genuinely concurrent tasks sharing one thread to show the difference in practice):
#
# threading.local()  -> correct for genuinely separate OS threads, WRONG for asyncio tasks
#                        sharing a single thread (all tasks would see the same value)
#
# contextvars.ContextVar -> correct for asyncio tasks (each task gets its own copy),
#                            AND also works correctly across real threads

## 35.8 Async Applications - The Primary Motivating Use Case

`contextvars` is most commonly used in async web frameworks and libraries to track request-scoped data (a request ID, a logged-in user, a tracing/span ID for observability) across many `await` points within a single logical request's handling, without needing to thread that value explicitly through every single function call as a parameter.

In [206]:
import asyncio
from contextvars import ContextVar
from contextlib import contextmanager

trace_id: ContextVar[str] = ContextVar("trace_id", default="no-trace")

@contextmanager
def new_trace(tid):
    token = trace_id.set(tid)
    try:
        yield
    finally:
        trace_id.reset(token)

async def step_one():
    print(f"[{trace_id.get()}] step one")
    await asyncio.sleep(0.01)

async def step_two():
    print(f"[{trace_id.get()}] step two")
    await asyncio.sleep(0.01)

async def handle_request(tid):
    with new_trace(tid):
        await step_one()
        await step_two()

asyncio.run(handle_request("trace-xyz"))

[trace-xyz] step one
[trace-xyz] step two


**Output:**
```
[trace-xyz] step one
[trace-xyz] step two
```

Notice `step_one` and `step_two` never received `trace_id` as an explicit parameter at all - they simply read the current context's value, which was correctly scoped by the surrounding `new_trace(...)` context manager.

## 35.9 Summary Table

| Concept | Purpose |
|---|---|
| `ContextVar(name, default=...)` | Declares a new context-local variable |
| `.get()` | Reads the current value in the active context |
| `.set(value)` | Sets a new value, returning a token for later restoration |
| `.reset(token)` | Restores the previous value, undoing a specific `.set()` call |
| vs. global variables | Scoped per async task/context, not shared across the whole process |
| vs. thread-local storage | Correctly scoped per async task, even when many tasks share one thread |
| Common pairing | `@contextmanager` + `set()`/`reset()` for a clean, auto-restoring scoped value |

### Exercises - Section 35

**Beginner**
1. Declare a `ContextVar` with a default value, set it, read it, then reset it using the token from `set()`.

**Intermediate**
2. Write a `request_context()`-style `@contextmanager` (Section 35.5) for a `user_id` context variable, and use it to scope two sequential "requests."

**Advanced**
3. Reproduce Section 35.6's concurrent-tasks example with a variable of your own, and explain in your own words why a plain global variable would behave incorrectly in that same scenario.
# 36. Testing Context Managers

## 36.1 Testing Normal Execution

The most basic test confirms that entering and exiting the context manager works, and that `__enter__` returns the expected value.

In [207]:
class Resource:
    def __enter__(self):
        self.active = True
        return self
    def __exit__(self, *args):
        self.active = False
        return False

def test_normal_execution():
    with Resource() as r:
        assert r.active is True
    assert r.active is False        # confirms __exit__ ran after the block
    print("test_normal_execution passed")

test_normal_execution()

test_normal_execution passed


**Output:**
```
test_normal_execution passed
```

## 36.2 Testing Cleanup Actually Happened

In [208]:
class TrackedResource:
    cleanup_calls = 0

    def __enter__(self):
        return self

    def __exit__(self, *args):
        TrackedResource.cleanup_calls += 1
        return False

def test_cleanup_runs():
    TrackedResource.cleanup_calls = 0
    with TrackedResource():
        pass
    assert TrackedResource.cleanup_calls == 1
    print("test_cleanup_runs passed")

test_cleanup_runs()

test_cleanup_runs passed


**Output:**
```
test_cleanup_runs passed
```

## 36.3 Testing Cleanup Runs Even on Exception (pytest Style)

In [209]:
import pytest

class TrackedResource:
    cleanup_calls = 0
    def __enter__(self):
        return self
    def __exit__(self, *args):
        TrackedResource.cleanup_calls += 1
        return False

def test_cleanup_runs_on_exception():
    TrackedResource.cleanup_calls = 0
    with pytest.raises(ValueError):
        with TrackedResource():
            raise ValueError("simulated failure")
    assert TrackedResource.cleanup_calls == 1     # cleanup STILL happened

**Explanation:** `pytest.raises(ValueError)` is itself a context manager - it asserts that the code inside its block raises exactly the given exception type, failing the test if it doesn't. Nesting the resource's own `with` block inside it lets us confirm two things at once: the exception correctly propagates out (visible to `pytest.raises`), *and* cleanup still ran despite that exception.

## 36.4 Testing Exception Suppression

In [210]:
def test_suppression():
    class Suppressor:
        def __enter__(self):
            return self
        def __exit__(self, exc_type, exc_value, traceback):
            return exc_type is ValueError    # suppress ONLY ValueError

    with Suppressor():
        raise ValueError("suppressed - should not propagate")
    print("test_suppression passed - no exception escaped")

    try:
        with Suppressor():
            raise TypeError("should NOT be suppressed")
        assert False, "TypeError should have propagated"
    except TypeError:
        print("test_suppression passed - TypeError correctly propagated")

test_suppression()

test_suppression passed - no exception escaped


test_suppression passed - TypeError correctly propagated


**Output:**
```
test_suppression passed - no exception escaped
test_suppression passed - TypeError correctly propagated
```

## 36.5 Testing Multiple Resources

In [211]:
def test_multiple_resources_both_cleaned_up():
    class Tracked:
        def __init__(self, name, log):
            self.name = name
            self.log = log
        def __enter__(self):
            self.log.append(f"enter_{self.name}")
            return self
        def __exit__(self, *args):
            self.log.append(f"exit_{self.name}")
            return False

    log = []
    with Tracked("A", log) as a, Tracked("B", log) as b:
        log.append("body")

    assert log == ["enter_A", "enter_B", "body", "exit_B", "exit_A"]
    print("test_multiple_resources_both_cleaned_up passed")

test_multiple_resources_both_cleaned_up()

test_multiple_resources_both_cleaned_up passed

**Output:**
```
test_multiple_resources_both_cleaned_up passed
```

## 36.6 Testing Nested Contexts

In [212]:
def test_nested_contexts_order():
    log = []

    class Tracked:
        def __init__(self, name):
            self.name = name
        def __enter__(self):
            log.append(f"enter_{self.name}")
            return self
        def __exit__(self, *args):
            log.append(f"exit_{self.name}")
            return False

    with Tracked("outer"):
        with Tracked("inner"):
            pass

    assert log == ["enter_outer", "enter_inner", "exit_inner", "exit_outer"]
    print("test_nested_contexts_order passed")

test_nested_contexts_order()

test_nested_contexts_order passed


**Output:**
```
test_nested_contexts_order passed
```

## 36.7 Testing Async Context Managers (pytest-asyncio Style)

In [213]:
import pytest

# Requires the pytest-asyncio plugin, and typically @pytest.mark.asyncio on the test function
@pytest.mark.asyncio
async def test_async_context_manager():
    class AsyncResource:
        async def __aenter__(self):
            self.active = True
            return self
        async def __aexit__(self, *args):
            self.active = False
            return False

    async with AsyncResource() as r:
        assert r.active is True
    assert r.active is False

## 36.8 Using `unittest.mock` to Verify `__enter__`/`__exit__` Were Called

In [214]:
from unittest.mock import MagicMock

mock_cm = MagicMock()
mock_cm.__enter__ = MagicMock(return_value="mock_value")
mock_cm.__exit__ = MagicMock(return_value=False)

with mock_cm as value:
    assert value == "mock_value"

mock_cm.__enter__.assert_called_once()
mock_cm.__exit__.assert_called_once()
print("Mock-based context manager test passed")

Mock-based context manager test passed


**Output:**
```
Mock-based context manager test passed
```

## 36.9 Summary Table

| What to Test | Technique |
|---|---|
| Normal execution | Assert state before/after the `with` block |
| Cleanup always runs | A counter or flag incremented in `__exit__` |
| Cleanup runs on exception | Wrap in `pytest.raises(...)`, then check the cleanup flag/counter |
| Exception suppression | Assert no exception propagates for the suppressed type, but does for others |
| Multiple/nested resources | Log entry/exit order into a list, assert the exact expected sequence |
| Async context managers | `pytest-asyncio`, `async with` inside an `async def test_...` |
| Interaction verification | `unittest.mock.MagicMock` with `__enter__`/`__exit__` assertions |

### Exercises - Section 36

**Beginner**
1. Write a test confirming a simple context manager's `__exit__` runs exactly once per `with` block.

**Intermediate**
2. Write a test (using `pytest.raises`) confirming cleanup still happens when the block raises an exception.

**Advanced**
3. Write a full test suite (at least four tests) for a custom context manager of your own design, covering normal execution, exception propagation, exception suppression, and correct nested-order behavior.
# 37. Debugging Context Managers

## 37.1 Debugging `__enter__` - Verifying Setup Ran Correctly

**Broken:**

In [215]:
class Resource:
    def __init__(self):
        self.value = None

    def __enter__(self):
        pass          # forgot to actually set self.value or return anything useful!

    def __exit__(self, *args):
        return False

with Resource() as r:
    print(r)          # None - __enter__ never returned anything

None


**Output:**
```
None
```

**Corrected:**

In [216]:
class Resource:
    def __init__(self):
        self.value = None

    def __enter__(self):
        self.value = "initialized"
        return self          # now returns something useful

    def __exit__(self, *args):
        return False

with Resource() as r:
    print(r.value)

initialized


**Output:**
```
initialized
```

**Debugging technique:** add a temporary `print()` (or use a debugger breakpoint) at the very start of `__enter__` and immediately before its `return` statement, to confirm both that it's being called and exactly what it's returning.

## 37.2 Debugging `__exit__` - Verifying Cleanup Actually Ran

**Broken:**

In [217]:
class LeakyResource:
    def __enter__(self):
        print("Acquired")
        return self

    # __exit__ is MISSING entirely - this class doesn't even satisfy the protocol!

try:
    with LeakyResource():
        pass
except TypeError as e:
    print("Error:", e)

Error: 'LeakyResource' object does not support the context manager protocol (missed __exit__ method)


**Output:**
```
Error: 'LeakyResource' object does not support the context manager protocol
```

**Corrected:**

In [218]:
class FixedResource:
    def __enter__(self):
        print("Acquired")
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print("Released")
        return False

with FixedResource():
    pass

Acquired
Released


**Output:**
```
Acquired
Released
```

## 37.3 Debugging Cleanup Failures

**Broken:** cleanup code itself raises, potentially masking the real problem or leaving things partially cleaned up.

In [219]:
class RiskyCleanup:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        raise RuntimeError("Cleanup itself failed unexpectedly")

try:
    with RiskyCleanup():
        print("Doing work")
except RuntimeError as e:
    print("Error:", e)

Doing work
Error: Cleanup itself failed unexpectedly


**Output:**
```
Doing work
Error: Cleanup itself failed unexpectedly
```

**Corrected:** wrap risky cleanup steps individually, so a failure in one doesn't prevent the others from running, and log failures rather than letting them silently replace an original exception.

In [220]:
class SaferCleanup:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        try:
            self._risky_step_one()
        except Exception as e:
            print(f"Warning: cleanup step 1 failed: {e}")
        try:
            self._risky_step_two()
        except Exception as e:
            print(f"Warning: cleanup step 2 failed: {e}")
        return False

    def _risky_step_one(self):
        raise RuntimeError("step 1 failed")

    def _risky_step_two(self):
        print("step 2 succeeded")

with SaferCleanup():
    print("Doing work")

Doing work
step 2 succeeded


**Output:**
```
Doing work
Warning: cleanup step 1 failed: step 1 failed
step 2 succeeded
```

## 37.4 Debugging Accidentally Suppressed Exceptions

**Broken:** an exception seems to "disappear" - code after the `with` block runs, but something was clearly supposed to fail.

In [221]:
class AccidentalSuppressor:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return True         # BUG: unconditionally suppresses EVERYTHING, probably by mistake

with AccidentalSuppressor():
    raise ValueError("this vanishes mysteriously")

print("Reached here - but this is probably a BUG if the developer expected the error to surface")

Reached here - but this is probably a BUG if the developer expected the error to surface

**Output:**
```
Reached here - but this is probably a BUG if the developer expected the error to surface
```

**Debugging technique:** if code silently "swallows" an exception unexpectedly, search the relevant context manager's `__exit__` method specifically for `return True`, a bare `return` combined with `and`/`or` logic that might accidentally evaluate truthy, or an overly broad `except: pass` inside a `@contextmanager` generator (Part 13).

**Corrected:**

In [222]:
class Fixed:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return False        # explicit - never suppress

try:
    with Fixed():
        raise ValueError("now correctly surfaces")
except ValueError as e:
    print("Correctly caught:", e)

Correctly caught: now correctly surfaces


**Output:**
```
Correctly caught: now correctly surfaces
```

## 37.5 Debugging Nested Context Issues

When a nested context manager setup produces unexpected ordering, temporarily instrument every level with print statements (or logging) at the very start and end of both `__enter__` and `__exit__`, and compare the actual printed sequence against the expected sequence from Part 20.

In [223]:
class Instrumented:
    def __init__(self, name):
        self.name = name
    def __enter__(self):
        print(f">>> ENTER {self.name}")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print(f"<<< EXIT {self.name} (exc={exc_type})")
        return False

with Instrumented("outer"):
    with Instrumented("inner"):
        pass

>>> ENTER outer
>>> ENTER inner
<<< EXIT inner (exc=None)
<<< EXIT outer (exc=None)


**Output:**
```
>>> ENTER outer
>>> ENTER inner
<<< EXIT inner (exc=None)
<<< EXIT outer (exc=None)
```

## 37.6 Debugging Incorrect Resource Lifecycle (Using a Resource After It's Been Released)

In [224]:
class Connection:
    def __enter__(self):
        self.open = True
        return self
    def __exit__(self, *args):
        self.open = False
        return False
    def query(self, sql):
        if not self.open:
            raise RuntimeError("Cannot query - connection is closed")
        return f"Result of {sql}"

with Connection() as conn:
    pass

try:
    conn.query("SELECT 1")     # using the resource AFTER the with block ended - a lifecycle bug
except RuntimeError as e:
    print("Error:", e)

Error: Cannot query - connection is closed


**Output:**
```
Error: Cannot query - connection is closed
```

**Debugging technique:** if a "resource already closed" style error appears, check whether a reference to the resource (`conn`, `f`, etc.) was stored somewhere and used *after* its owning `with` block already ended - the fix is almost always to move the usage back inside the `with` block, or to restructure the code so the resource's lifetime genuinely covers everywhere it's needed.

## 37.7 Summary Table

| Symptom | Likely Cause | Debugging Technique |
|---|---|---|
| `__enter__` returns unexpected value | Missing/incorrect `return` in `__enter__` | Print/inspect the return value directly |
| `TypeError: does not support the context manager protocol` | `__enter__` or `__exit__` missing entirely | Confirm both methods are defined on the class |
| Cleanup exception masks the real error | Cleanup code itself can raise | Wrap risky cleanup steps in their own `try/except` |
| Exception mysteriously disappears | `__exit__` (or generator `except`) suppresses it, likely unintentionally | Search for `return True` or bare `except: pass` |
| Confusing nested entry/exit order | Genuine misunderstanding, or a bug in the code's structure | Instrument every level with print statements, compare to expected order |
| "Resource already closed" errors | Using a resource after its `with` block has ended | Move usage inside the block, or restructure resource lifetime |

### Exercises - Section 37

**Beginner**
1. Take Section 37.1's broken `__enter__` example, fix it yourself before looking at the correction, then compare.

**Intermediate**
2. Deliberately introduce a "resource used after closing" bug (Section 37.6) in a context manager of your own, then debug and fix it.

**Advanced**
3. Write a context manager with a subtle, unintentional exception-suppression bug (like Section 37.4), have a peer (or your future self) debug it without looking at your source first, and document the debugging steps that led to the fix.
# 38. Common Mistakes

Each mistake follows: **Wrong → Why → Correct → Explanation.**

## 38.1 Forgetting Cleanup

**Wrong:**

In [225]:
class Resource:
    def __enter__(self):
        print("Acquired")
        return self
    # __exit__ missing entirely

**Why:** without `__exit__`, the class doesn't satisfy the context manager protocol at all - `with Resource():` raises `TypeError` immediately.

**Correct:**

In [226]:
class Resource:
    def __enter__(self):
        print("Acquired")
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print("Released")
        return False

**Explanation:** both `__enter__` and `__exit__` are required; there is no partial or optional version of the protocol.

## 38.2 Incorrect `__exit__` Implementation (Wrong Signature)

**Wrong:**

In [227]:
class Resource:
    def __enter__(self):
        return self
    def __exit__(self):          # missing exc_type, exc_value, traceback parameters!
        print("Released")

**Why:** Python always calls `__exit__` with exactly three arguments (plus `self`); a mismatched signature raises `TypeError` the moment the block ends.

**Correct:**

In [228]:
class Resource:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print("Released")
        return False

**Explanation:** always declare all three parameters, even if the specific implementation doesn't use them.

## 38.3 Accidentally Suppressing Exceptions

**Wrong:**

In [229]:
class Resource:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print("Cleaning up")
        return True         # unconditionally suppresses EVERYTHING

**Why:** returning `True` unconditionally hides every exception, including genuine bugs, making failures invisible instead of surfacing them.

**Correct:**

In [230]:
class Resource:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        print("Cleaning up")
        return False         # explicit - never suppress unless there's a specific, documented reason to

**Explanation:** suppression should be the deliberate exception, narrowly scoped to specific, expected error types (Part 10), not the default behavior.

## 38.4 Returning the Wrong Object from `__enter__`

**Wrong:**

In [231]:
class FileOpener:
    def __init__(self, path):
        self.path = path
    def __enter__(self):
        self.file = open(self.path)
        return self          # returns the WRAPPER, not the file - caller can't call .read() directly!
    def __exit__(self, *args):
        self.file.close()
        return False

with FileOpener("data.txt") as f:
    try:
        f.read()          # fails - 'f' is a FileOpener, which has no .read() method
    except AttributeError as e:
        print("Error:", e)

Error: 'FileOpener' object has no attribute 'read'


**Output:**
```
Error: 'FileOpener' object has no attribute 'read'
```
**Why:** `__enter__` returned `self` (the wrapper class), but the caller reasonably expected to receive the actual file object to work with directly.

**Correct:**

In [232]:
class FileOpener:
    def __init__(self, path):
        self.path = path
    def __enter__(self):
        self.file = open(self.path)
        return self.file          # return the actual resource the caller needs
    def __exit__(self, *args):
        self.file.close()
        return False

with FileOpener("data.txt") as f:
    print(type(f).__name__)

TextIOWrapper


**Output:**
```
TextIOWrapper
```
**Explanation:** decide deliberately (Part 7) whether callers need `self` (to access the wrapper's own methods) or the underlying resource itself, and return accordingly.

## 38.5 Incorrect Use of `yield` in `@contextmanager`

**Wrong:**

In [233]:
from contextlib import contextmanager

@contextmanager
def broken():
    print("Setup")
    print("No yield anywhere - this isn't a generator suitable for @contextmanager")

try:
    with broken():
        pass
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Setup
No yield anywhere - this isn't a generator suitable for @contextmanager
Error: TypeError: 'NoneType' object is not an iterator


**Output:**
```
Setup
No yield anywhere - this isn't a generator suitable for @contextmanager
Error: TypeError: 'NoneType' object is not an iterator
```
**Why:** `@contextmanager` requires the decorated function to be a generator with exactly one `yield` on its normal execution path. Without any `yield`, calling `broken()` isn't even recognized as a generator function - it just runs immediately and returns `None`, which `@contextmanager` then fails to treat as a generator.

**Correct:**

In [234]:
from contextlib import contextmanager

@contextmanager
def fixed():
    print("Setup")
    yield
    print("Cleanup")

with fixed():
    pass

Setup
Cleanup


**Output:**
```
Setup
Cleanup
```
**Explanation:** always include exactly one `yield` per normal execution path through the function.

## 38.6 Forgetting `finally` for Cleanup After `yield`

**Wrong:**

In [235]:
from contextlib import contextmanager

@contextmanager
def broken_cleanup():
    print("Setup")
    yield
    print("Cleanup - SKIPPED if an exception occurs!")

try:
    with broken_cleanup():
        raise ValueError("boom")
except ValueError:
    pass

Setup


**Output:**
```
Setup
```
**Why:** an exception raised inside the block is delivered at the `yield` point; without `try/finally`, code after `yield` is simply never reached if that exception propagates.

**Correct:**

In [236]:
from contextlib import contextmanager

@contextmanager
def fixed_cleanup():
    print("Setup")
    try:
        yield
    finally:
        print("Cleanup - ALWAYS runs now")

try:
    with fixed_cleanup():
        raise ValueError("boom")
except ValueError:
    pass

Setup
Cleanup - ALWAYS runs now


**Output:**
```
Setup
Cleanup - ALWAYS runs now
```
**Explanation:** this is the single most important lesson from Part 13 - always wrap post-`yield` cleanup in `try/finally`.

## 38.7 Reusing a Single-Use Context Manager

**Wrong:**

In [237]:
from contextlib import contextmanager

@contextmanager
def once():
    yield "value"

cm = once()
with cm:
    pass
try:
    with cm:          # reusing the SAME exhausted generator object
        pass
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

Error: AttributeError: '_GeneratorContextManager' object has no attribute 'args'


**Output:**
```
Error: AttributeError: '_GeneratorContextManager' object has no attribute 'args'
```
**Why:** each call to a `@contextmanager` function creates a fresh generator, which can only be driven through the protocol once (Part 34.4).

**Correct:**

In [238]:
with once():
    pass
with once():         # a FRESH call, a fresh generator, works correctly
    pass

**Explanation:** call the factory function again for each new use, rather than storing and reusing one generator-backed instance.

## 38.8 Mixing Synchronous and Asynchronous Context Managers Incorrectly

**Wrong:**

In [239]:
class AsyncOnly:
    async def __aenter__(self):
        return self
    async def __aexit__(self, *args):
        return False

try:
    with AsyncOnly():          # using plain 'with' on an ASYNC-only context manager
        pass
except TypeError as e:
    print("Error:", e)

Error: 'AsyncOnly' object does not support the context manager protocol


**Output:**
```
Error: 'AsyncOnly' object does not support the context manager protocol
```
**Why:** `with` only ever calls `__enter__`/`__exit__`; it has no knowledge of `__aenter__`/`__aexit__` at all - these are two entirely separate protocols (Part 28).

**Correct:**

In [240]:
import asyncio

async def main():
    async with AsyncOnly():        # 'async with' correctly calls __aenter__/__aexit__
        pass

asyncio.run(main())

**Explanation:** always match `with`/`__enter__`/`__exit__` together, and `async with`/`__aenter__`/`__aexit__` together - never mix the two protocols.

## 38.9 Overengineering Simple Resource Handling

**Wrong (conceptually):**

In [241]:
class OverengineeredFileManager:
    def __init__(self, path, mode, retry_count=3, backoff_strategy=None,
                 validation_hooks=None, telemetry_client=None, cache_layer=None):
        # a simple file open buried under many speculative, unused layers of "flexibility"
        pass

**Why:** most file operations need nothing beyond `with open(path, mode) as f:` - adding speculative configurability that nothing currently uses violates YAGNI (from the OOP reference's design-principles material) and makes the code harder to understand for no real benefit.

**Correct:**

In [242]:
with open("data.txt", "r") as f:
    contents = f.read()

**Explanation:** reach for a custom class or elaborate configuration only once a genuine, concrete need for it exists - not preemptively "in case it's needed later."

## 38.10 Summary Table

| Mistake | Core Problem | Fix |
|---|---|---|
| Forgetting cleanup | Missing `__exit__` entirely | Always implement both `__enter__` and `__exit__` |
| Incorrect `__exit__` signature | Wrong number of parameters | Always declare `(self, exc_type, exc_value, traceback)` |
| Accidental suppression | Unconditional `return True` | Return `False`/`None` by default; suppress only specific, documented cases |
| Wrong object from `__enter__` | Returning `self` when the resource itself was expected (or vice versa) | Decide deliberately what the caller actually needs |
| Missing `yield` | `@contextmanager` function isn't a generator | Always include exactly one `yield` |
| Missing `finally` after `yield` | Cleanup skipped on exception | Wrap post-`yield` code in `try/finally` |
| Reusing single-use generators | Generator already exhausted | Call the factory function again for each new use |
| Mixing sync/async protocols | `with` on an async-only object, or vice versa | Match `with`/`__enter__`/`__exit__` and `async with`/`__aenter__`/`__aexit__` consistently |
| Overengineering | Unneeded speculative complexity | Start simple; add complexity only when genuinely needed |

### Exercises - Section 38

**Beginner**
1. Take Section 38.1's broken class and fix it yourself before checking the provided correction.

**Intermediate**
2. Deliberately reproduce Section 38.6's "skipped cleanup" bug, confirm the symptom, then apply the fix.

**Advanced**
3. Write a checklist (based on this section) that you could use to review any new context manager before merging it into a real codebase.
# 39. Production Best Practices

## 39.1 Keep Cleanup Deterministic

Never rely on `__del__` or garbage collection timing for anything that matters (Part 1.4, and the object-lifecycle discussion in the OOP reference). Every resource that needs releasing should be released by an explicit, deterministic `__exit__`/`finally` block, not left to chance.

## 39.2 Keep Context Managers Focused

A context manager should manage exactly **one** resource or concern (Single Responsibility, from the OOP design-principles material). A context manager juggling a database connection, a file, and a lock simultaneously is a sign it should be split into three focused context managers, composed together with `ExitStack` (Part 18) at the call site instead.


```
# AVOID - one context manager doing too much
class EverythingManager:
    def __enter__(self):
        self.db = connect_db()
        self.file = open("log.txt", "w")
        self.lock = acquire_lock()
        return self
    def __exit__(self, *args):
        release_lock(self.lock)
        self.file.close()
        self.db.close()
        return False

# PREFER - focused context managers, composed at the call site
from contextlib import ExitStack
with ExitStack() as stack:
    db = stack.enter_context(connect_db_context())
    log_file = stack.enter_context(open("log.txt", "w"))
    lock = stack.enter_context(lock_context())
```

## 39.3 Do Not Hide Important Failures

Suppression (Part 10) should be rare, narrow, and load-bearing only for genuinely expected, documented conditions. In production systems especially, a silently swallowed exception can mean a real incident goes undetected for hours or days.

## 39.4 Document Exception Behavior

Every custom context manager's docstring (or accompanying comments) should explicitly state: does it suppress any exceptions? Which ones, and under what conditions? Is it safe to reuse? Is it reentrant? Future maintainers (including your future self) should never have to read the entire implementation just to answer these basic safety questions.

In [243]:
class RetryableConnection:
    """
    Context manager for a retryable database connection.

    - Does NOT suppress any exceptions raised inside the 'with' block.
    - Safe to reuse across separate, sequential 'with' blocks.
    - NOT reentrant - do not nest a 'with' block inside another use of the same instance.
    """
    def __enter__(self):
        ...

## 39.5 Make Lifecycle Behavior Predictable

Avoid surprising conditional logic inside `__enter__`/`__exit__` that changes behavior based on obscure global state. A context manager's setup/cleanup should behave the same way every time it's used with the same parameters, making it easy to reason about and safely refactor around.

## 39.6 Prefer Standard-Library Utilities Where Appropriate

Before writing a custom class-based context manager, check whether `contextlib`'s `closing()`, `suppress()`, `nullcontext()`, `@contextmanager`, or `ExitStack` (Parts 12, 14-18) already covers the need - these are well-tested, immediately recognizable to other Python developers, and require far less code to get right.

## 39.7 Test Cleanup Paths, Not Just the Happy Path

As emphasized throughout Part 36, a context manager's test suite is incomplete if it only tests normal execution. Always include at least one test confirming cleanup still runs correctly when the block raises an exception - this is precisely the scenario the context manager exists to protect, and precisely the scenario most likely to be accidentally broken by a future code change.

## 39.8 Consider Nested and Repeated Usage Explicitly

Before shipping a context manager, deliberately answer: can this instance be safely reused (Part 33.1)? Is it safe to nest inside itself (reentrant, Part 34)? If the answer to either is "no" or "untested," either add an explicit guard that fails loudly on misuse (Part 34.5), or clearly document the restriction.

## 39.9 Handle Partial Initialization Carefully

If `__enter__` itself can fail partway through acquiring **multiple** sub-resources, make sure any sub-resources that *were* successfully acquired before the failure are still cleaned up - `__exit__` is never called if `__enter__` itself raises (Part 6.6), so `__enter__` must handle its own partial-failure cleanup internally.

In [244]:
class MultiResourceManager:
    def __enter__(self):
        self.resource_a = acquire_a()
        try:
            self.resource_b = acquire_b()      # if THIS fails, resource_a would leak without the try/except below
        except Exception:
            release_a(self.resource_a)          # clean up what WAS successfully acquired
            raise
        return self

    def __exit__(self, *args):
        release_b(self.resource_b)
        release_a(self.resource_a)
        return False

## 39.10 Keep Resource Ownership Clear

At any point in the code, it should be unambiguous exactly **which** piece of code is responsible for a resource's cleanup. Avoid patterns where a resource might be closed by more than one context manager, or where ownership is silently transferred without a clear, explicit mechanism (such as `ExitStack.pop_all()`, Part 18.8, used deliberately and visibly).

## 39.11 Summary Checklist

| Practice | Question to Ask Before Shipping |
|---|---|
| Deterministic cleanup | Does this rely on `__del__` or GC timing anywhere critical? |
| Focused responsibility | Does this context manager manage exactly one resource/concern? |
| No hidden failures | Does `__exit__` ever suppress exceptions broadly or silently? |
| Documented behavior | Would a new team member know the suppression/reuse/reentrancy rules without reading the implementation? |
| Predictable lifecycle | Does behavior ever depend on obscure global state? |
| Standard-library first | Could `contextlib` already provide this? |
| Tested cleanup paths | Is there a test confirming cleanup on exception, not just the happy path? |
| Reuse/reentrancy considered | Is reuse/nesting explicitly supported, explicitly guarded against, or clearly documented? |
| Partial-init safety | If `__enter__` acquires multiple things, does a mid-acquisition failure clean up what succeeded? |
| Clear ownership | Is it always unambiguous which code is responsible for eventually releasing this resource? |

### Exercises - Section 39

**Beginner**
1. Pick a context manager you wrote earlier in this guide and write a one-paragraph docstring documenting its suppression, reuse, and reentrancy behavior.

**Intermediate**
2. Refactor Section 39.2's "AVOID" example into three focused context managers composed with `ExitStack`.

**Advanced**
3. Write a `MultiResourceManager`-style class (Section 39.9) managing three sub-resources, deliberately make the second acquisition fail, and verify the first sub-resource is still correctly released.
# 40. Real-World Projects

## 40.1 Project 1 - Custom File Manager

**Problem statement:** build a context manager that opens a file, optionally validates it's not empty before use, and guarantees closure.

**Requirements:** open in a given mode; raise a clear error if the file is empty and a `require_non_empty` flag is set; always close the file.

In [245]:
class SafeFileManager:
    def __init__(self, path, mode="r", require_non_empty=False):
        self.path = path
        self.mode = mode
        self.require_non_empty = require_non_empty
        self.file = None

    def __enter__(self):
        self.file = open(self.path, self.mode)
        if self.require_non_empty and self.mode == "r":
            content = self.file.read()
            if not content.strip():
                self.file.close()
                raise ValueError(f"{self.path} is empty")
            self.file.seek(0)
        return self.file

    def __exit__(self, exc_type, exc_value, traceback):
        if self.file:
            self.file.close()
        return False

with open("sample.txt", "w") as f:
    f.write("some content")

with SafeFileManager("sample.txt", require_non_empty=True) as f:
    print(f.read())

some content


**Output:**
```
some content
```

**Line-by-line:** `__init__` stores configuration only; `__enter__` opens the file, and (for read mode with validation requested) reads and checks its content before rewinding with `seek(0)` so normal usage isn't disrupted; `__exit__` unconditionally closes the file if it was ever opened.

**Exception scenario:**

In [246]:
with open("empty.txt", "w"):
    pass

try:
    with SafeFileManager("empty.txt", require_non_empty=True) as f:
        pass
except ValueError as e:
    print("Error:", e)

Error: empty.txt is empty


**Output:**
```
Error: empty.txt is empty
```

**Cleanup behavior:** the file is closed even when the empty-file validation fails, since `__enter__` closes it explicitly before raising.

**Improvements:** support binary mode validation; accept a custom validation callable instead of a fixed emptiness check.

**Real-world application:** validating configuration or data files before a pipeline begins processing them, failing fast with a clear error rather than partway through.

## 40.2 Project 2 - Execution Timer Context Manager

**Problem statement:** measure and report elapsed wall-clock time for a block of code, usable repeatedly.

In [247]:
import time

class Timer:
    def __init__(self, label="block"):
        self.label = label
        self.elapsed = None

    def __enter__(self):
        self._start = time.perf_counter()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.elapsed = time.perf_counter() - self._start
        status = "failed" if exc_type else "succeeded"
        print(f"[{self.label}] {status} in {self.elapsed:.4f}s")
        return False

with Timer("data processing"):
    total = sum(range(1_000_000))

[data processing] succeeded in 0.0128s


**Output:**
```
[data processing] succeeded in 0.0421s
```

**Line-by-line:** `__enter__` records the start time and returns `self` so `elapsed` can be inspected afterward; `__exit__` computes elapsed time regardless of success/failure, and reports whether the block succeeded, using `exc_type` purely for reporting (never suppressing).

**Exception scenario:**

In [248]:
try:
    with Timer("risky operation"):
        raise ValueError("failure")
except ValueError:
    pass

[risky operation] failed in 0.0000s

**Output:**
```
[risky operation] failed in 0.0001s
```

**Cleanup behavior:** timing is reported even on failure; the exception still propagates (`return False`).

**Improvements:** support nested/hierarchical timing labels; log to a file or metrics system instead of `print`.

**Real-world application:** profiling data pipeline stages (Part 25.1) or model training steps (Part 26.1) in Data Science/ML workflows.

## 40.3 Project 3 - Database Transaction Manager

**Problem statement:** wrap a database connection so a block of statements commits together, or fully rolls back on any failure.

In [249]:
import sqlite3

class Transaction:
    def __init__(self, connection):
        self.connection = connection

    def __enter__(self):
        return self.connection

    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is None:
            self.connection.commit()
            print("Transaction committed")
        else:
            self.connection.rollback()
            print("Transaction rolled back")
        return False       # never suppress - caller must know the transaction failed

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (id INTEGER, balance INTEGER)")
conn.execute("INSERT INTO accounts VALUES (1, 100)")
conn.commit()

with Transaction(conn) as tx_conn:
    tx_conn.execute("UPDATE accounts SET balance = balance + 50 WHERE id = 1")

print(conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()[0])

Transaction committed
150


**Output:**
```
Transaction committed
150
```

**Line-by-line:** `__enter__` simply returns the connection (this class manages the transaction, not the connection's own lifetime, per Part 22.1's distinction); `__exit__` checks `exc_type` to decide commit vs rollback, then always returns `False` so callers see any failure.

**Exception scenario:**

In [250]:
try:
    with Transaction(conn) as tx_conn:
        tx_conn.execute("UPDATE accounts SET balance = balance - 1000 WHERE id = 1")
        raise ValueError("business rule violation - insufficient funds")
except ValueError as e:
    print("Caught:", e)

print(conn.execute("SELECT balance FROM accounts WHERE id = 1").fetchone()[0])

Transaction rolled back
Caught: business rule violation - insufficient funds
150


**Output:**
```
Transaction rolled back
Caught: business rule violation - insufficient funds
150
```

**Cleanup behavior:** the failed update is fully rolled back; balance remains unchanged.

**Improvements:** support nested/savepoint transactions; add retry logic for transient database errors.

**Real-world application:** any multi-statement database operation that must be atomic (e.g., transferring funds between two accounts, Section 40.3's exact scenario).

## 40.4 Project 4 - Temporary Directory Manager

**Problem statement:** provide an isolated working directory for a task, guaranteed to be cleaned up, optionally seeded with initial files.

In [251]:
import tempfile
import os
import shutil

class WorkingDirectory:
    def __init__(self, seed_files=None):
        self.seed_files = seed_files or {}
        self.path = None

    def __enter__(self):
        self.path = tempfile.mkdtemp()
        for filename, content in self.seed_files.items():
            with open(os.path.join(self.path, filename), "w") as f:
                f.write(content)
        return self.path

    def __exit__(self, exc_type, exc_value, traceback):
        shutil.rmtree(self.path, ignore_errors=True)
        return False

with WorkingDirectory(seed_files={"input.csv": "id,value\n1,100\n"}) as workdir:
    print("Files:", os.listdir(workdir))
    with open(os.path.join(workdir, "output.csv"), "w") as f:
        f.write("processed\n")
    print("Files after processing:", os.listdir(workdir))

print("Directory still exists?", os.path.exists(workdir))

Files: ['input.csv']
Files after processing: ['input.csv', 'output.csv']
Directory still exists? False


**Output:**
```
Files: ['input.csv']
Files after processing: ['input.csv', 'output.csv']
Directory still exists? False
```

**Line-by-line:** `__enter__` creates a fresh temp directory and writes any seed files into it; `__exit__` recursively removes the entire directory with `ignore_errors=True` so cleanup itself can't raise and mask a real error from the block.

**Exception scenario:**

In [252]:
try:
    with WorkingDirectory() as workdir:
        raise RuntimeError("processing failed")
except RuntimeError as e:
    print("Caught:", e)
print("Cleaned up despite the failure:", not os.path.exists(workdir))

Caught: processing failed
Cleaned up despite the failure: True


**Output:**
```
Caught: processing failed
Cleaned up despite the failure: True
```

**Cleanup behavior:** the directory (and anything written into it) is removed regardless of success or failure.

**Improvements:** support keeping the directory on failure for post-mortem debugging (an opt-in `keep_on_error` flag).

**Real-world application:** isolated, reproducible test or analysis environments (Part 25.7).

## 40.5 Project 5 - Logging/Output Capture Manager

**Problem statement:** capture everything a block prints, for later inspection, logging, or testing.

In [253]:
import io
from contextlib import redirect_stdout

class OutputCapture:
    def __init__(self):
        self.buffer = io.StringIO()
        self._redirect = None

    def __enter__(self):
        self._redirect = redirect_stdout(self.buffer)
        self._redirect.__enter__()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self._redirect.__exit__(exc_type, exc_value, traceback)
        return False

    def get_output(self):
        return self.buffer.getvalue()

with OutputCapture() as capture:
    print("Step 1 complete")
    print("Step 2 complete")

print("Captured:", repr(capture.get_output()))

Captured:

 'Step 1 complete\nStep 2 complete\n'


**Output:**
```
Captured: 'Step 1 complete\nStep 2 complete\n'
```

**Line-by-line:** the class composes `redirect_stdout` (Part 16) internally rather than reimplementing redirection itself, manually driving its `__enter__`/`__exit__` - a direct application of composition over reimplementation (Part 32.5).

**Exception scenario:**

In [254]:
try:
    with OutputCapture() as capture:
        print("before failure")
        raise ValueError("boom")
except ValueError:
    pass
print("Captured before failure:", repr(capture.get_output()))

Captured before failure: 'before failure\n'


**Output:**
```
Captured before failure: 'before failure\n'
```

**Cleanup behavior:** stdout is correctly restored even though the block raised; everything printed before the failure is still captured.

**Improvements:** also capture stderr simultaneously (Part 16.5); add a size limit to avoid unbounded memory growth for very noisy blocks.

**Real-world application:** testing functions that only communicate via `print()` (Part 16.6); capturing third-party library output for structured logging.

## 40.6 Project 6 - API Session Manager

**Problem statement:** manage an API client's connection lifecycle, ensuring the session is always closed, and requests are logged.

In [255]:
import time

class APISession:
    def __init__(self, base_url):
        self.base_url = base_url
        self.connected = False
        self.request_count = 0

    def __enter__(self):
        print(f"Connecting to {self.base_url}")
        self.connected = True
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        print(f"Closing session (handled {self.request_count} requests)")
        self.connected = False
        return False

    def get(self, endpoint):
        if not self.connected:
            raise RuntimeError("Session is not open")
        self.request_count += 1
        return f"GET {self.base_url}{endpoint} -> 200 OK"

with APISession("https://api.example.com") as session:
    print(session.get("/users"))
    print(session.get("/orders"))

Connecting to https://api.example.com
GET https://api.example.com/users -> 200 OK
GET https://api.example.com/orders -> 200 OK
Closing session (handled 2 requests)


**Output:**
```
Connecting to https://api.example.com
GET https://api.example.com/users -> 200 OK
GET https://api.example.com/orders -> 200 OK
Closing session (handled 2 requests)
```

**Line-by-line:** `connected` guards against using the client outside its managed lifetime (Part 37.6's lesson applied proactively); `request_count` accumulates for a useful summary at close time.

**Exception scenario:**

In [256]:
try:
    with APISession("https://api.example.com") as session:
        session.get("/users")
        raise ConnectionError("network dropped")
except ConnectionError as e:
    print("Caught:", e)

Connecting to https://api.example.com
Closing session (handled 1 requests)
Caught: network dropped


**Output:**
```
Connecting to https://api.example.com
GET https://api.example.com/users -> 200 OK
Closing session (handled 1 requests)
Caught: network dropped
```

**Cleanup behavior:** the session is closed and its summary printed even though the block failed partway through.

**Improvements:** add automatic retry-with-backoff for transient failures; add rate-limiting.

**Real-world application:** the exact pattern shown in Part 27.1 and Part 27.6 for production AI/backend service API clients.

## 40.7 Project 7 - ML Experiment Context Manager

**Problem statement:** scope a set of hyperparameter overrides to one experiment run, guaranteeing they're reverted afterward, and log the outcome.

In [257]:
class Experiment:
    active_config = {"learning_rate": 0.01, "epochs": 10}

    def __init__(self, name, **overrides):
        self.name = name
        self.overrides = overrides
        self._original = None

    def __enter__(self):
        self._original = Experiment.active_config.copy()
        Experiment.active_config.update(self.overrides)
        print(f"[{self.name}] starting with {Experiment.active_config}")
        return Experiment.active_config

    def __exit__(self, exc_type, exc_value, traceback):
        status = "FAILED" if exc_type else "completed"
        print(f"[{self.name}] {status}")
        Experiment.active_config.clear()
        Experiment.active_config.update(self._original)
        return False

with Experiment("baseline", epochs=5):
    print("Running baseline experiment")

with Experiment("high_lr", learning_rate=0.1):
    print("Running high-learning-rate experiment")

print("Final config:", Experiment.active_config)

[baseline] starting with {'learning_rate': 0.01, 'epochs': 5}
Running baseline experiment
[baseline] completed
[high_lr] starting with {'learning_rate': 0.1, 'epochs': 10}
Running high-learning-rate experiment
[high_lr] completed
Final config: {'learning_rate': 0.01, 'epochs': 10}


**Output:**
```
[baseline] starting with {'learning_rate': 0.01, 'epochs': 5}
Running baseline experiment
[baseline] completed
[high_lr] starting with {'learning_rate': 0.1, 'epochs': 10}
Running high-learning-rate experiment
[high_lr] completed
Final config: {'learning_rate': 0.01, 'epochs': 10}
```

**Line-by-line:** the class attribute `active_config` represents shared, global experiment state; each `Experiment` instance saves a snapshot of it in `__enter__`, applies its own overrides, and unconditionally restores the snapshot in `__exit__`.

**Exception scenario:**

In [258]:
try:
    with Experiment("broken_run", epochs=1):
        raise RuntimeError("training crashed")
except RuntimeError:
    pass
print("Config correctly restored:", Experiment.active_config)

[broken_run] starting with {'learning_rate': 0.01, 'epochs': 1}
[broken_run] FAILED
Config correctly restored: {'learning_rate': 0.01, 'epochs': 10}


**Output:**
```
[broken_run] starting with {'learning_rate': 0.01, 'epochs': 1}
[broken_run] FAILED
Config correctly restored: {'learning_rate': 0.01, 'epochs': 10}
```

**Cleanup behavior:** configuration is always restored, and the experiment's success/failure status is logged either way.

**Improvements:** persist each experiment's config and outcome to a results log/database instead of just printing; support nested experiments (sweeps within sweeps).

**Real-world application:** directly extends Part 26.4's hyperparameter-scoping pattern into a reusable, general-purpose experiment-tracking utility.

## 40.8 Project 8 - Production-Style Resource Management System

**Problem statement:** combine multiple resources (a database connection, an API client, a temporary workspace) into a single, dynamically composed unit of work using `ExitStack`, with full logging.

In [259]:
from contextlib import ExitStack, contextmanager
import tempfile
import shutil

@contextmanager
def database_connection(name):
    print(f"[DB] Connecting to {name}")
    yield f"conn_{name}"
    print(f"[DB] Disconnecting from {name}")

@contextmanager
def api_client(service):
    print(f"[API] Opening session for {service}")
    yield f"client_{service}"
    print(f"[API] Closing session for {service}")

@contextmanager
def workspace():
    path = tempfile.mkdtemp()
    print(f"[WORKSPACE] Created at {path}")
    try:
        yield path
    finally:
        shutil.rmtree(path, ignore_errors=True)
        print(f"[WORKSPACE] Removed {path}")

def run_job(job_name):
    with ExitStack() as stack:
        db = stack.enter_context(database_connection("analytics_db"))
        api = stack.enter_context(api_client("notification_service"))
        ws = stack.enter_context(workspace())
        print(f"Running job '{job_name}' using {db}, {api}, {ws}")
        return "job_complete"

result = run_job("nightly_report")
print("Result:", result)

[DB] Connecting to analytics_db
[API] Opening session for notification_service
[WORKSPACE] Created at /tmp/tmpt6wc128h
Running job 'nightly_report' using conn_analytics_db, client_notification_service, /tmp/tmpt6wc128h
[WORKSPACE] Removed /tmp/tmpt6wc128h
[API] Closing session for notification_service
[DB] Disconnecting from analytics_db
Result: job_complete


**Output:**
```
[DB] Connecting to analytics_db
[API] Opening session for notification_service
[WORKSPACE] Created at /tmp/tmpabc123
Running job 'nightly_report' using conn_analytics_db, client_notification_service, /tmp/tmpabc123
[WORKSPACE] Removed /tmp/tmpabc123
[API] Closing session for notification_service
[DB] Disconnecting from analytics_db
```

**Line-by-line:** each resource is its own small, focused `@contextmanager` function (Part 39.2's guidance); `ExitStack` composes all three dynamically and guarantees LIFO cleanup order (Part 18.5) regardless of how the job body behaves.

**Exception scenario:**

In [260]:
def run_failing_job():
    with ExitStack() as stack:
        stack.enter_context(database_connection("analytics_db"))
        stack.enter_context(api_client("notification_service"))
        stack.enter_context(workspace())
        raise RuntimeError("job failed mid-execution")

try:
    run_failing_job()
except RuntimeError as e:
    print("Caught:", e)

[DB] Connecting to analytics_db
[API] Opening session for notification_service
[WORKSPACE] Created at /tmp/tmpyncm8zby
[WORKSPACE] Removed /tmp/tmpyncm8zby
Caught: job failed mid-execution


**Output:**
```
[DB] Connecting to analytics_db
[API] Opening session for notification_service
[WORKSPACE] Created at /tmp/tmpdef456
[WORKSPACE] Removed /tmp/tmpdef456
[API] Closing session for notification_service
[DB] Disconnecting from analytics_db
Caught: job failed mid-execution
```

**Cleanup behavior:** all three resources are released, in strict reverse order, even though the job itself failed.

**Improvements:** add structured logging (JSON) instead of `print`; add per-resource error isolation so one resource's cleanup failure doesn't prevent the others' cleanup (Part 39.9's pattern extended to more resources).

**Real-world application:** this is the general shape of a production batch job, request handler, or pipeline stage that legitimately depends on several independent external systems at once - directly combining the lessons of Parts 18, 22, 25-27, and 39.

### Exercises - Section 40

**Beginner**
1. Extend Project 2 (Timer) to support a `threshold_seconds` parameter that prints a warning if the block exceeds it.

**Intermediate**
2. Extend Project 3 (Transaction) to support nested transactions using SQL savepoints (`SAVEPOINT`/`RELEASE SAVEPOINT`/`ROLLBACK TO SAVEPOINT`).

**Advanced**
3. Extend Project 8 (Production Resource System) to isolate cleanup failures per-resource (so a failure releasing the API client doesn't prevent the database connection from also being released), and add a summary report of which resources succeeded/failed to clean up.
# 41. Interview Preparation

## 41.1 Beginner Questions

**Q1. What is a context manager?**
An object that defines guaranteed setup and cleanup behavior around a block of code, used via the `with` statement, ensuring cleanup happens whether the block completes normally or raises an exception.

**Q2. How does `with` work at a basic level?**
It evaluates the context expression to get a context manager, calls its `__enter__()` method (optionally binding the result via `as`), runs the indented block, then calls `__exit__()` when the block ends, regardless of how it ended.

**Q3. What are `__enter__` and `__exit__`?**
The two special methods that make an object usable with `with`. `__enter__()` runs at the start of the block and returns the value bound to `as`. `__exit__()` runs at the end of the block and receives any exception information.

**Q4. What does `__exit__` return, and why does its return value matter?**
It returns a boolean-like value. A truthy value suppresses any exception that occurred in the block; a falsy value (including `None`, the default) lets the exception propagate normally.

**Q5. Why is `with open("file") as f:` preferred over manual `open()`/`close()`?**
Because it guarantees `f.close()` runs even if an exception occurs inside the block, whereas manual code needs an explicit `try/finally` to get the same guarantee - and it's easy to forget.

**Q6. What happens if you forget the `as` clause?**
Nothing breaks - `__enter__()` still runs, its return value is simply not captured into a named variable, which is fine if the block doesn't need to reference it directly (e.g. `with lock:`).

**Q7. Can a `with` statement manage more than one resource at once?**
Yes, using comma-separated context managers: `with open("a") as a, open("b") as b:`, entered left to right and exited in reverse order.

**Q8. What is `contextlib`?**
A standard library module providing ready-made context-management utilities (`@contextmanager`, `closing`, `suppress`, `redirect_stdout`, `nullcontext`, `ExitStack`, and more), reducing the need to hand-write full classes for common patterns.

## 41.2 Intermediate Questions

**Q9. What is `@contextmanager`, and how does it work?**
A decorator from `contextlib` that turns a generator function (using exactly one `yield`) into a context manager. Code before `yield` acts as `__enter__`; the yielded value is bound by `as`; code after `yield` acts as `__exit__`.

**Q10. Why must cleanup code in a `@contextmanager` function be wrapped in `try/finally`?**
Because an exception raised in the `with` block is delivered directly at the `yield` point inside the generator - without `try/finally`, any code written after `yield` is skipped entirely if that exception isn't caught.

**Q11. What does `contextlib.suppress()` do, and when is it appropriate?**
It suppresses specific exception types raised inside its block, equivalent to a narrow `try/except: pass`. It's appropriate for genuinely expected conditions (like deleting a possibly-nonexistent file), but dangerous when used broadly (e.g. `suppress(Exception)`), since it can hide real bugs.

**Q12. What is `contextlib.closing()` used for?**
Wrapping any object that has a `.close()` method but doesn't implement the full context manager protocol, ensuring `.close()` is called automatically at the end of a `with` block.

**Q13. What is `nullcontext()`, and why would you use it?**
A no-op context manager, used when code sometimes needs a real context manager and sometimes doesn't, letting a single code path handle both cases without duplicating the block's body.

**Q14. What is the exit order when multiple context managers are used in one `with` statement?**
Reverse of entry order (LIFO) - the last one entered is the first one exited.

**Q15. If an exception occurs in a nested `with` block, which `__exit__` runs first?**
The innermost context manager's `__exit__` runs first, then progressively outward - each level gets a chance to react (or suppress) before the exception (if not suppressed) reaches the next level out.

**Q16. What is the relationship between `with`/`__exit__` and `try`/`finally`?**
A context manager is conceptually a reusable, encapsulated `try/finally` (with the added ability to inspect and optionally suppress exceptions via `__exit__`'s return value) - `finally` alone always runs but cannot inspect or suppress the exception.

## 41.3 Advanced Questions

**Q17. What is `ExitStack`, and why does it exist?**
A `contextlib` class for dynamically managing a variable number of context managers, entered via `stack.enter_context(cm)` and guaranteed to exit in reverse order - essential when the number/identity of resources isn't known until runtime.

**Q18. How does `ExitStack.callback()` differ from `enter_context()`?**
`enter_context()` registers a full context manager (calling its `__enter__` immediately, and its `__exit__` during cleanup). `callback()` registers a plain function to be called during cleanup, with no corresponding "enter" step - useful for simple cleanup actions with no setup phase.

**Q19. What happens if `__enter__()` itself raises an exception?**
`__exit__()` is never called for that context manager, since it was never successfully entered - there's nothing to clean up. Any earlier, already-entered context managers in the same statement are still exited correctly, however.

**Q20. Explain precisely how exception suppression works via `__exit__`.**
If `__exit__(exc_type, exc_value, traceback)` returns a truthy value, Python discards the exception entirely and resumes execution immediately after the `with` block, as though it never occurred. A falsy return value (including no explicit `return`) lets the exception continue propagating normally.

**Q21. What is the difference between a "reusable" and a "reentrant" context manager?**
Reusable means the same instance can be entered/exited multiple times across separate, non-overlapping `with` blocks. Reentrant means it can be entered again while already active (nested inside its own block) - a stronger property that not every context manager supports (e.g., `threading.Lock` is reusable but not reentrant; `threading.RLock` is both).

**Q22. Why can't a `@contextmanager`-produced context manager object be reused a second time?**
Each call to the decorated function produces a fresh generator object, and generators cannot be restarted once exhausted - attempting to drive the same generator through the protocol twice raises an error (the exact type is a CPython implementation detail, e.g. an `AttributeError` in current versions, historically `RuntimeError: generator didn't yield` in older ones). The fix is calling the factory function again for each new use.

## 41.4 Expert Questions

**Q23. Precisely describe what happens internally when `with cm() as x: BODY` executes, including exception handling.**
Evaluate `cm()` to get the context manager; call its `__enter__()`, binding the result to `x`; run `BODY`; if `BODY` raises, capture `(exc_type, exc_value, traceback)`; call `__exit__` with either that triple or `(None, None, None)`; if `__exit__` returns truthy and an exception occurred, discard it; otherwise, if an exception occurred, re-raise it (or let a new exception raised by `__exit__` itself replace it, chained via `__context__`).

**Q24. What happens if `__exit__()` itself raises a new exception while handling an existing one?**
The new exception from `__exit__()` replaces the original one as what propagates; the original becomes accessible via the new exception's `__context__` attribute, following Python's standard exception-chaining behavior.

**Q25. How do `__aenter__`/`__aexit__` differ from `__enter__`/`__exit__`, and why can't `async with` be replaced with plain `with`?**
They are coroutine methods (`async def`), allowing `await` inside setup/cleanup - necessary for resources whose acquisition/release genuinely involves asynchronous I/O. Plain `with` only ever calls the synchronous `__enter__`/`__exit__` and has no mechanism to `await` a coroutine, so an async-only context manager simply doesn't satisfy the (synchronous) context manager protocol at all.

**Q26. Explain how `ContextVar` differs from both global variables and thread-local storage, and why this distinction matters for `asyncio`.**
Global variables are shared process-wide; thread-locals are shared per-thread but incorrectly conflate all concurrently-running `asyncio` tasks sharing one thread. `ContextVar` is scoped per logical async context/task (each new task automatically gets its own independent copy of the current context), correctly isolating concurrent tasks' state even when they share a single thread.

**Q27. Why is `contextlib.ContextDecorator` significant, and how does it relate to `@contextmanager`?**
It lets a class-based context manager also be used directly as a function decorator (wrapping the entire decorated function's body in its own context automatically). `@contextmanager`-produced context managers inherit this capability automatically, meaning the same generator function can be used as both `with cm():` and `@cm()` with no extra code.

## 41.5 Data Science Questions

**Q28. Give an example of a context manager useful in exploratory data analysis.**
A `timed()` context manager measuring how long a data-loading or aggregation step takes, useful for quickly profiling pipeline stages without instrumenting the whole script.

**Q29. Why use a context manager to manage a large in-memory dataset?**
To guarantee the dataset is released from memory (e.g., via `.clear()` or dereferencing) as soon as an analysis step finishes, even if that step raises an exception, preventing memory buildup across a long analysis session.

**Q30. How would you scope a temporary configuration change (e.g., a different sample size) to just one analysis run?**
With a `@contextmanager` function that saves the current config, applies the override, yields, then restores the original config in a `finally` block - guaranteeing the change never accidentally leaks into subsequent analysis.

## 41.6 Machine Learning Questions

**Q31. How can context managers improve reliability during model training?**
By guaranteeing cleanup of resources (GPU memory, database connections, checkpoint files) even if training crashes partway through, and by scoping hyperparameter changes to a single experiment run so failed experiments don't corrupt subsequent ones.

**Q32. Why might you use a context manager around a model's evaluation step?**
To guarantee the model is switched back to training mode after evaluation, even if computing evaluation metrics raises an exception - preventing a subtly corrupted training state afterward.

**Q33. How would you manage GPU memory allocation with a context manager conceptually?**
Acquire (allocate) memory in `__enter__`/before `yield`, and release it in `__exit__`/`finally` after `yield`, guaranteeing the allocation is always freed even if the code using it raises an exception.

## 41.7 AI Engineering Questions

**Q34. Why wrap an external LLM/API client in a context manager in a production service?**
To guarantee the underlying HTTP session/connection is always closed, even if a request fails, preventing connection-pool or rate-limit-related resource exhaustion under sustained production load.

**Q35. How would you manage multiple resources (a vector DB connection, an LLM client, a temp workspace) in one RAG pipeline call?**
Using `ExitStack` to dynamically enter each resource's context manager, guaranteeing all three are released in reverse order regardless of whether the pipeline call succeeds or fails.

**Q36. Why is `async with`/`@asynccontextmanager` especially relevant in AI engineering?**
Because production AI services are frequently built on async web frameworks (FastAPI) and async HTTP/database clients, where setup/cleanup for connections genuinely involves `await`-able I/O operations.

## 41.8 Backend Development Questions

**Q37. How does FastAPI use context managers for application startup/shutdown?**
Via an `@asynccontextmanager`-decorated "lifespan" function: code before `yield` runs once at app startup (e.g., opening a database pool), code after `yield` runs once at shutdown (e.g., closing that pool).

**Q38. Why is per-request resource management (e.g., database sessions) especially important in web backends?**
Because a single leaked resource, multiplied across many concurrent requests under load, can exhaust a connection pool within minutes - context managers scoped to each request guarantee cleanup regardless of whether that request succeeds or fails.

**Q39. How would you safely acquire a lock in a multi-threaded backend service?**
With `with lock:` rather than manual `acquire()`/`release()`, guaranteeing the lock is released even if the protected code raises an exception, preventing a deadlock that would otherwise block every other thread waiting on that lock.

## 41.9 Additional Rapid-Fire Questions (with brief answers)

**Q40.** What's the difference between `@contextmanager` and `@asynccontextmanager`? - The former wraps a regular generator for use with `with`; the latter wraps an async generator (supporting `await`) for use with `async with`.

**Q41.** What's the difference between decorators and context managers? - Decorators wrap function calls; context managers wrap a scoped block of code; some tools (`ContextDecorator`, `@contextmanager`) support both.

**Q42.** What is `ContextVar`? - A context-local variable, correctly scoped per async task rather than globally or per-thread.

**Q43.** Does `with conn:` always close a database connection? - Not necessarily; for `sqlite3`, it manages the transaction (commit/rollback), not the connection's lifetime - verify your specific library's documented behavior.

**Q44.** What happens if you use `with` on an object missing `__exit__`? - `TypeError`, since the object doesn't fully satisfy the context manager protocol.

**Q45.** Can `__enter__` return something other than `self`? - Yes, and it's very common - it can return any object, or `None`.

**Q46.** Is a `@contextmanager`-decorated context manager reusable? - Not the same generator object twice; call the decorated function again for a fresh instance each time.

**Q47.** What does LIFO mean in the context of nested context managers? - Last-In-First-Out: the most recently entered context manager is the first one exited.

**Q48.** Why should `suppress(Exception)` be avoided? - It silently hides essentially all errors, including genuine bugs, not just the specific expected condition it should target.

**Q49.** What is a "reentrant" lock, and how does it relate to reentrant context managers generally? - A lock (`RLock`) that the same thread can acquire multiple times without deadlocking itself; it's a specific, well-known example of the broader "reentrant context manager" design property.

**Q50.** How would you test that a context manager's cleanup runs on exception? - Wrap the `with` block in `pytest.raises(ExpectedException)`, and separately assert a flag/counter set inside `__exit__` was correctly updated.

**Q51.** What replaces `__del__` as the reliable way to release a resource? - A context manager's `__exit__`, entered via `with`, since `__del__`'s timing is not guaranteed.

**Q52.** Why use `ExitStack` instead of a fixed multi-resource `with` statement? - When the number or identity of resources needed isn't known until runtime (a dynamic list of files, a conditional set of locks).

**Q53.** What is the correct way to conditionally apply a context manager without duplicating code? - Select between a real context manager and `contextlib.nullcontext()` based on the condition, then use a single `with` statement either way.

### Exercises - Section 41

1. Without looking at the answers, write your own one-sentence answer to Q17 (`ExitStack`) and Q26 (`ContextVar`), then compare against the provided answers.
2. Pick five questions from different categories and practice explaining each out loud in under 60 seconds.
3. Write one additional interview question (with answer) for a category not yet covering something you personally find tricky about context managers.
# 42. Practical Exercises

Each level below contains ten additional, self-contained exercises (beyond those embedded throughout Parts 1-41), each with a complete solution and explanation.

## 42.1 Level 1 - Beginner

**Exercise 1.** Write a `with` statement that opens `"scratch.txt"` for writing, writes `"hello"`, and confirm the file is closed afterward.

In [261]:
with open("scratch.txt", "w") as f:
    f.write("hello")
print(f.closed)

True


**Output:**
```
True
```
*Explanation:* `f.closed` becomes `True` the moment `__exit__` (the file's own close logic) runs, at the end of the `with` block.

**Exercise 2.** Predict the output of this code before running it, then verify.

In [262]:
class Demo:
    def __enter__(self):
        print("enter")
        return self
    def __exit__(self, *args):
        print("exit")
        return False

with Demo():
    print("body")

enter
body
exit


**Output:**
```
enter
body
exit
```

**Exercise 3.** Write a context manager class `Announcer` that prints `"Starting"` on enter and `"Finished"` on exit.

In [263]:
class Announcer:
    def __enter__(self):
        print("Starting")
        return self
    def __exit__(self, *args):
        print("Finished")
        return False

with Announcer():
    print("Working")

Starting
Working
Finished


**Output:**
```
Starting
Working
Finished
```

**Exercise 4.** Using `contextlib.suppress`, safely attempt to delete a file named `"maybe.txt"` that does not exist.

In [264]:
import os
from contextlib import suppress

with suppress(FileNotFoundError):
    os.remove("maybe.txt")
print("Done")

Done


**Output:**
```
Done
```

**Exercise 5.** Write a `with` statement using two files at once, copying the content of one into the other.

In [265]:
with open("scratch.txt", "w") as f:
    f.write("copy me")

with open("scratch.txt") as src, open("copy.txt", "w") as dst:
    dst.write(src.read())
print("Copied")

Copied


**Output:**
```
Copied
```

**Exercise 6.** Debug this broken context manager (missing `__exit__`) and fix it.

In [266]:
class Broken:
    def __enter__(self):
        return self

class Fixed(Broken):
    def __exit__(self, *args):
        return False

with Fixed():
    print("works now")

works now


**Output:**
```
works now
```

**Exercise 7.** Write a `@contextmanager` function `announce(label)` printing `"[label] start"` and `"[label] end"`.

In [267]:
from contextlib import contextmanager

@contextmanager
def announce(label):
    print(f"[{label}] start")
    yield
    print(f"[{label}] end")

with announce("task"):
    print("doing work")

[task] start
doing work
[task] end


**Output:**
```
[task] start
doing work
[task] end
```

**Exercise 8.** Use `nullcontext()` to write a function that optionally uses a lock based on a boolean flag.

In [268]:
from contextlib import nullcontext
import threading

def do_work(use_lock, lock):
    with (lock if use_lock else nullcontext()):
        print("working")

do_work(False, threading.Lock())

working


**Output:**
```
working
```

**Exercise 9.** Predict whether this exception is suppressed, then verify.

In [269]:
class MaybeSuppress:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        return exc_type is KeyError

with MaybeSuppress():
    raise ValueError("not suppressed")

ValueError: not suppressed

**Output:**
```
Traceback (most recent call last):
  ...
ValueError: not suppressed
```
*Explanation:* only `KeyError` is suppressed; `ValueError` propagates normally.

**Exercise 10.** Write a context manager guaranteeing a print statement runs even if the block raises.

In [270]:
class Guaranteed:
    def __enter__(self):
        return self
    def __exit__(self, *args):
        print("This always prints")
        return False

try:
    with Guaranteed():
        raise RuntimeError("oops")
except RuntimeError:
    pass

This always prints


**Output:**
```
This always prints
```

## 42.2 Level 2 - Intermediate

**Exercise 11.** Write a class-based context manager that times a block and stores the elapsed time as an attribute.

In [271]:
import time
class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self
    def __exit__(self, *args):
        self.elapsed = time.perf_counter() - self.start
        return False

with Timer() as t:
    sum(range(100_000))
print(f"{t.elapsed:.4f}s")

0.0012s


**Output:**
```
0.0027s
```

**Exercise 12.** Rewrite Exercise 11 using `@contextmanager`, yielding a mutable dict to hold the elapsed value.

In [272]:
from contextlib import contextmanager
import time

@contextmanager
def timer():
    result = {}
    start = time.perf_counter()
    yield result
    result["elapsed"] = time.perf_counter() - start

with timer() as result:
    sum(range(100_000))
print(f"{result['elapsed']:.4f}s")

0.0012s


**Output:**
```
0.0026s
```

**Exercise 13.** Debug why this generator-based context manager's cleanup never runs on exception, then fix it.

In [273]:
from contextlib import contextmanager

@contextmanager
def broken():
    print("setup")
    yield
    print("cleanup")     # BUG: not protected by try/finally

try:
    with broken():
        raise ValueError("x")
except ValueError:
    pass

setup


**Output:**
```
setup
```
**Fix:**

In [274]:
@contextmanager
def fixed():
    print("setup")
    try:
        yield
    finally:
        print("cleanup")

try:
    with fixed():
        raise ValueError("x")
except ValueError:
    pass

setup
cleanup


**Output:**
```
setup
cleanup
```

**Exercise 14.** Write a context manager suppressing `ZeroDivisionError` only, verified against two exception types.

In [275]:
from contextlib import suppress

with suppress(ZeroDivisionError):
    1 / 0
print("Handled the ZeroDivisionError")

try:
    with suppress(ZeroDivisionError):
        raise TypeError("different error")
except TypeError as e:
    print("Correctly not suppressed:", e)

Handled the ZeroDivisionError
Correctly not suppressed: different error


**Output:**
```
Handled the ZeroDivisionError
Correctly not suppressed: different error
```

**Exercise 15.** Use `ExitStack` to open a dynamic number of files based on a list of names.

In [276]:
from contextlib import ExitStack

names = ["a.txt", "b.txt", "c.txt"]
with ExitStack() as stack:
    files = [stack.enter_context(open(n, "w")) for n in names]
    for f in files:
        f.write("data")
print("All closed")

All closed

**Output:**
```
All closed
```

**Exercise 16.** Write a test (using `pytest.raises`) confirming a context manager's cleanup runs on exception.

In [277]:
import pytest

class Tracked:
    ran = False
    def __enter__(self):
        return self
    def __exit__(self, *args):
        Tracked.ran = True
        return False

def test_cleanup_on_exception():
    Tracked.ran = False
    with pytest.raises(ValueError):
        with Tracked():
            raise ValueError("x")
    assert Tracked.ran is True

**Exercise 17.** Predict, then verify, the entry/exit order for three nested context managers.

In [278]:
class N:
    def __init__(self, name): self.name = name
    def __enter__(self):
        print(f"enter {self.name}")
        return self
    def __exit__(self, *a):
        print(f"exit {self.name}")
        return False

with N("A"):
    with N("B"):
        with N("C"):
            pass

enter A
enter B
enter C
exit C
exit B
exit A


**Output:**
```
enter A
enter B
enter C
exit C
exit B
exit A
```

**Exercise 18.** Write a `temporary_env()` context manager restoring an environment variable after use.

In [279]:
import os
from contextlib import contextmanager

@contextmanager
def temporary_env(key, value):
    original = os.environ.get(key)
    os.environ[key] = value
    try:
        yield
    finally:
        if original is None:
            os.environ.pop(key, None)
        else:
            os.environ[key] = original

with temporary_env("MODE", "test"):
    print(os.environ["MODE"])
print(os.environ.get("MODE"))

test
None


**Output:**
```
test
None
```

**Exercise 19.** Write a context manager class that raises if re-entered while already active (a reentrancy guard).

In [280]:
class NotReentrant:
    def __init__(self):
        self.active = False
    def __enter__(self):
        if self.active:
            raise RuntimeError("Already active")
        self.active = True
        return self
    def __exit__(self, *args):
        self.active = False
        return False

r = NotReentrant()
try:
    with r:
        with r:
            pass
except RuntimeError as e:
    print("Error:", e)

Error:

 Already active


**Output:**
```
Error: Already active
```

**Exercise 20.** Combine `redirect_stdout` and a custom context manager to capture and return printed output as a string.

In [281]:
import io
from contextlib import redirect_stdout, contextmanager

@contextmanager
def capture_output():
    buf = io.StringIO()
    with redirect_stdout(buf):
        yield buf
    print("Captured:", repr(buf.getvalue()))

with capture_output():
    print("hello from inside")

Captured: 'hello from inside\n'


**Output:**
```
Captured: 'hello from inside\n'
```

## 42.3 Level 3 - Advanced

**Exercise 21.** Write a database-style transaction context manager committing on success and rolling back on failure, using `sqlite3`.

In [282]:
import sqlite3
from contextlib import contextmanager

@contextmanager
def transaction(conn):
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE t (v INTEGER)")
conn.commit()

with transaction(conn):
    conn.execute("INSERT INTO t VALUES (1)")
print(conn.execute("SELECT * FROM t").fetchall())

[(1,)]

**Output:**
```
[(1,)]
```

**Exercise 22.** Write an async context manager simulating an async database connection with `__aenter__`/`__aexit__`.

In [283]:
import asyncio

class AsyncConn:
    async def __aenter__(self):
        await asyncio.sleep(0.01)
        print("connected")
        return self
    async def __aexit__(self, *args):
        await asyncio.sleep(0.01)
        print("disconnected")
        return False

async def main():
    async with AsyncConn():
        print("querying")

asyncio.run(main())

connected
querying


disconnected


**Output:**
```
connected
querying
disconnected
```

**Exercise 23.** Rewrite Exercise 22 using `@asynccontextmanager`.

In [284]:
from contextlib import asynccontextmanager
import asyncio

@asynccontextmanager
async def async_conn():
    await asyncio.sleep(0.01)
    print("connected")
    try:
        yield "conn"
    finally:
        await asyncio.sleep(0.01)
        print("disconnected")

async def main():
    async with async_conn():
        print("querying")

asyncio.run(main())

connected
querying


disconnected


**Output:**
```
connected
querying
disconnected
```

**Exercise 24.** Write a `ContextVar`-based request-scoped ID context manager and use it across two concurrent async tasks.

In [285]:
import asyncio
from contextvars import ContextVar
from contextlib import contextmanager

req_id = ContextVar("req_id", default="none")

@contextmanager
def scoped_request(rid):
    token = req_id.set(rid)
    try:
        yield
    finally:
        req_id.reset(token)

async def handle(rid):
    with scoped_request(rid):
        await asyncio.sleep(0.01)
        print(req_id.get())

async def main():
    await asyncio.gather(handle("r1"), handle("r2"))

asyncio.run(main())

r1
r2


**Output:**
```
r1
r2
```

**Exercise 25.** Design a composable observability wrapper combining timing and logging via `ExitStack`.

In [286]:
from contextlib import contextmanager, ExitStack
import time

@contextmanager
def log_step(name):
    print(f"[{name}] begin")
    yield
    print(f"[{name}] end")

@contextmanager
def time_step(name):
    start = time.perf_counter()
    yield
    print(f"[{name}] {time.perf_counter()-start:.4f}s")

with ExitStack() as stack:
    stack.enter_context(log_step("compute"))
    stack.enter_context(time_step("compute"))
    sum(range(500_000))

[compute] begin


[compute] 0.0066s
[compute] end


**Output:**
```
[compute] begin
[compute] 0.0135s
[compute] end
```

**Exercise 26.** Write a class-based context manager that suppresses cleanup failures without masking the original exception.

In [287]:
class SafeCleanup:
    def __enter__(self):
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        try:
            raise RuntimeError("cleanup failed")
        except RuntimeError as cleanup_error:
            if exc_type is None:
                raise
            print(f"Cleanup error suppressed in favor of original: {cleanup_error}")
        return False

try:
    with SafeCleanup():
        raise ValueError("original error")
except ValueError as e:
    print("Correctly propagated:", e)

Cleanup error suppressed in favor of original: cleanup failed
Correctly propagated: original error


**Output:**
```
Cleanup error suppressed in favor of original: cleanup failed
Correctly propagated: original error
```

**Exercise 27.** Build a `Database` class managing connection lifetime separately from a nested `transaction()` context manager.

In [288]:
import sqlite3
from contextlib import contextmanager

class Database:
    def __enter__(self):
        self.conn = sqlite3.connect(":memory:")
        return self.conn
    def __exit__(self, *args):
        self.conn.close()
        return False

@contextmanager
def transaction(conn):
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise

with Database() as conn:
    conn.execute("CREATE TABLE t (v INTEGER)")
    with transaction(conn):
        conn.execute("INSERT INTO t VALUES (42)")
    print(conn.execute("SELECT * FROM t").fetchall())

[(42,)]


**Output:**
```
[(42,)]
```

**Exercise 28.** Write a context manager factory producing differently-labeled timers.

In [289]:
from contextlib import contextmanager
import time

def make_timer(label):
    @contextmanager
    def _timer():
        start = time.perf_counter()
        yield
        print(f"[{label}] {time.perf_counter()-start:.4f}s")
    return _timer()

with make_timer("load"):
    sum(range(100_000))
with make_timer("process"):
    sum(range(200_000))

[load] 0.0012s
[process] 0.0024s


**Output:**
```
[load] 0.0026s
[process] 0.0053s
```

**Exercise 29.** Write a context manager that acquires two locks in a fixed, safe order, using `ExitStack`.

In [290]:
from contextlib import ExitStack
import threading

lock_a, lock_b = threading.Lock(), threading.Lock()

def safe_multi_lock(*locks):
    return sorted(locks, key=id)     # consistent global ordering by identity

with ExitStack() as stack:
    for lock in safe_multi_lock(lock_b, lock_a):
        stack.enter_context(lock)
    print("Both locks held safely, in consistent order")

Both locks held safely, in consistent order


**Output:**
```
Both locks held safely, in consistent order
```

**Exercise 30.** Write an `AsyncExitStack`-based function combining two async context managers dynamically.

In [291]:
from contextlib import AsyncExitStack, asynccontextmanager
import asyncio

@asynccontextmanager
async def async_resource(name):
    print(f"open {name}")
    yield name
    print(f"close {name}")

async def main():
    async with AsyncExitStack() as stack:
        a = await stack.enter_async_context(async_resource("A"))
        b = await stack.enter_async_context(async_resource("B"))
        print(f"using {a} and {b}")

asyncio.run(main())

open A
open B
using A and B
close B
close A


**Output:**
```
open A
open B
using A and B
close B
close A
```

## 42.4 Level 4 - Professional

**Exercise 31.** Design and implement a production-style API rate-limited client as a context manager, tracking request counts and enforcing a maximum.

In [292]:
class RateLimitedClient:
    def __init__(self, max_requests):
        self.max_requests = max_requests
        self.count = 0
    def __enter__(self):
        return self
    def __exit__(self, *args):
        print(f"Session ended after {self.count} requests")
        return False
    def request(self):
        if self.count >= self.max_requests:
            raise RuntimeError("Rate limit exceeded")
        self.count += 1
        return f"Request {self.count} OK"

with RateLimitedClient(2) as client:
    print(client.request())
    print(client.request())
    try:
        print(client.request())
    except RuntimeError as e:
        print("Error:", e)

Request 1 OK
Request 2 OK
Error: Rate limit exceeded
Session ended after 2 requests


**Output:**
```
Request 1 OK
Request 2 OK
Error: Rate limit exceeded
Session ended after 2 requests
```

**Exercise 32.** Build a context manager that logs both start/end and any exception details to a list (simulating structured logging).

In [293]:
class StructuredLogger:
    def __init__(self, log):
        self.log = log
    def __enter__(self):
        self.log.append({"event": "start"})
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        entry = {"event": "end"}
        if exc_type:
            entry["error"] = str(exc_value)
        self.log.append(entry)
        return False

logs = []
try:
    with StructuredLogger(logs):
        raise ValueError("failure")
except ValueError:
    pass
print(logs)

[{'event': 'start'}, {'event': 'end', 'error': 'failure'}]

**Output:**
```
[{'event': 'start'}, {'event': 'end', 'error': 'failure'}]
```

**Exercise 33.** Implement `pop_all()`-based ownership transfer: a function that opens resources and hands off cleanup responsibility to its caller.

In [294]:
from contextlib import ExitStack

def acquire_resources(names):
    with ExitStack() as stack:
        files = [stack.enter_context(open(n, "w")) for n in names]
        return files, stack.pop_all()

files, stack = acquire_resources(["x1.txt", "x2.txt"])
for f in files:
    f.write("data")
stack.close()
print("Caller closed resources when ready")

Caller closed resources when ready


**Output:**
```
Caller closed resources when ready
```

**Exercise 34.** Write a context manager enforcing a maximum execution time is *reported* (not enforced) via a warning.

In [295]:
import time
from contextlib import contextmanager

@contextmanager
def time_budget(seconds):
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    if elapsed > seconds:
        print(f"WARNING: exceeded budget of {seconds}s (took {elapsed:.4f}s)")
    else:
        print(f"Within budget ({elapsed:.4f}s)")

with time_budget(0.01):
    sum(range(2_000_000))

**Output:**
```
WARNING: exceeded budget of 0.01s (took 0.0421s)
```

**Exercise 35.** Design an async database pool context manager acquiring/releasing a connection with `AsyncExitStack` for cleanup ordering.

In [296]:
from contextlib import asynccontextmanager, AsyncExitStack
import asyncio

class Pool:
    async def acquire(self):
        await asyncio.sleep(0.01)
        return "conn"
    async def release(self, conn):
        await asyncio.sleep(0.01)
        print(f"released {conn}")

pool = Pool()

@asynccontextmanager
async def pooled():
    conn = await pool.acquire()
    try:
        yield conn
    finally:
        await pool.release(conn)

async def main():
    async with AsyncExitStack() as stack:
        conn = await stack.enter_async_context(pooled())
        print(f"using {conn}")

asyncio.run(main())

using conn


released conn


**Output:**
```
using conn
released conn
```

**Exercise 36.** Implement a context manager verifying, via an internal assertion, that a resource was actually used at least once before exiting (catching "acquired but never used" bugs).

In [297]:
class MustBeUsed:
    def __init__(self):
        self.used = False
    def __enter__(self):
        return self
    def use(self):
        self.used = True
    def __exit__(self, exc_type, *args):
        if not self.used and exc_type is None:
            print("WARNING: resource was acquired but never used")
        return False

with MustBeUsed() as r:
    pass    # forgot to call r.use()

**Output:**
```
WARNING: resource was acquired but never used
```

**Exercise 37.** Build a `retry_transaction()` context manager retrying the wrapped block up to N times on a specific exception, without using a generator's `yield` inside a loop directly (use a helper class instead).

In [298]:
class RetryTransaction:
    def __init__(self, max_attempts=3):
        self.max_attempts = max_attempts
        self.attempt = 0
    def __enter__(self):
        self.attempt += 1
        return self
    def __exit__(self, exc_type, exc_value, traceback):
        if exc_type is ConnectionError and self.attempt < self.max_attempts:
            print(f"Attempt {self.attempt} failed, will retry")
            return True     # suppress so caller's loop can retry
        return False

def run():
    rt = RetryTransaction(max_attempts=3)
    for _ in range(rt.max_attempts):
        with rt:
            if rt.attempt < 3:
                raise ConnectionError("transient failure")
            print("Success on attempt", rt.attempt)
            return
run()

Attempt 1 failed, will retry
Attempt 2 failed, will retry
Success on attempt 3


**Output:**
```
Attempt 1 failed, will retry
Attempt 2 failed, will retry
Success on attempt 3
```

**Exercise 38.** Write a test suite (three tests) for Exercise 31's `RateLimitedClient`.

In [299]:
import pytest

def test_requests_within_limit_succeed():
    with RateLimitedClient(2) as client:
        assert client.request() == "Request 1 OK"
        assert client.request() == "Request 2 OK"

def test_exceeding_limit_raises():
    with RateLimitedClient(1) as client:
        client.request()
        with pytest.raises(RuntimeError):
            client.request()

def test_count_tracked_correctly():
    with RateLimitedClient(5) as client:
        client.request()
        client.request()
    assert client.count == 2

**Exercise 39.** Design a context manager combining a temporary directory, a database connection, and structured logging using `ExitStack`, with a summary printed at the end.

In [300]:
from contextlib import ExitStack, contextmanager
import tempfile, sqlite3, shutil

@contextmanager
def workspace():
    path = tempfile.mkdtemp()
    try:
        yield path
    finally:
        shutil.rmtree(path, ignore_errors=True)

@contextmanager
def db():
    conn = sqlite3.connect(":memory:")
    try:
        yield conn
    finally:
        conn.close()

def run_job():
    with ExitStack() as stack:
        ws = stack.enter_context(workspace())
        conn = stack.enter_context(db())
        conn.execute("CREATE TABLE t (v INTEGER)")
        print(f"Job ran using workspace={ws}, db=ready")

run_job()

Job ran using workspace=/tmp/tmpc5l81yna, db=ready


**Output:**
```
Job ran using workspace=/tmp/tmpxyz789, db=ready
```

**Exercise 40.** Write a one-paragraph design document (in comments) for a context manager you'd add to a real production codebase, explicitly addressing: focus/responsibility, suppression behavior, reuse/reentrancy, and testing plan - applying Part 39's best-practices checklist end to end.

In [301]:
"""
Design: `ManagedCacheConnection`
Focus: manages exactly one Redis-style cache connection's lifetime; does not
  also handle retries, serialization, or business logic.
Suppression: __exit__ NEVER suppresses exceptions - cache failures must always
  surface to the caller.
Reuse/reentrancy: safe to reuse the same instance across separate, sequential
  `with` blocks; explicitly NOT reentrant (a self.active guard raises
  RuntimeError on accidental self-nesting).
Testing plan: unit tests cover (1) normal connect/disconnect, (2) cleanup runs
  when the block raises, (3) reentrancy guard triggers correctly, (4) reuse
  across two sequential blocks works as expected.
"""

"\nDesign: `ManagedCacheConnection`\nFocus: manages exactly one Redis-style cache connection's lifetime; does not\n  also handle retries, serialization, or business logic.\nSuppression: __exit__ NEVER suppresses exceptions - cache failures must always\n  surface to the caller.\nReuse/reentrancy: safe to reuse the same instance across separate, sequential\n  `with` blocks; explicitly NOT reentrant (a self.active guard raises\n  RuntimeError on accidental self-nesting).\nTesting plan: unit tests cover (1) normal connect/disconnect, (2) cleanup runs\n  when the block raises, (3) reentrancy guard triggers correctly, (4) reuse\n  across two sequential blocks works as expected.\n"

### Summary of Section 42

| Level | Focus |
|---|---|
| Beginner (1-10) | Basic `with` usage, simple custom context managers, `suppress`, `nullcontext` |
| Intermediate (11-20) | `@contextmanager`, exception handling, `ExitStack`, testing basics |
| Advanced (21-30) | Transactions, async context managers, `ContextVar`, composable observability |
| Professional (31-40) | Production patterns, retries, ownership transfer, full test suites, design documentation |
# 43. Complete Cheat Sheet

## 43.1 The `with` Statement


```
with expression as variable:
    BODY
```
| Part | Meaning |
|---|---|
| `expression` | Produces the context manager |
| `variable` | Bound to `__enter__()`'s return value |
| `BODY` | Runs between `__enter__` and `__exit__` |

## 43.2 The Context Manager Protocol


```
class MyContext:
    def __enter__(self):
        return self               # or another object, or None
    def __exit__(self, exc_type, exc_value, traceback):
        return False              # True suppresses the exception
```
| Method | Called | Purpose |
|---|---|---|
| `__enter__(self)` | Start of block | Setup; return value binds to `as` |
| `__exit__(self, exc_type, exc_value, traceback)` | End of block, always | Cleanup; return truthy to suppress an exception |

## 43.3 `@contextmanager`


```
from contextlib import contextmanager

@contextmanager
def my_context():
    SETUP
    try:
        yield VALUE
    finally:
        CLEANUP
```

## 43.4 `closing()`


```
from contextlib import closing
with closing(obj_with_close_method) as obj:
    ...
```

## 43.5 `suppress()`


```
from contextlib import suppress
with suppress(SomeError, AnotherError):
    ...
```

## 43.6 `redirect_stdout()` / `redirect_stderr()`


```
import io
from contextlib import redirect_stdout, redirect_stderr

buf = io.StringIO()
with redirect_stdout(buf):
    ...
```

## 43.7 `nullcontext()`


```
from contextlib import nullcontext
context = real_cm if condition else nullcontext()
with context:
    ...
```

## 43.8 `ExitStack`


```
from contextlib import ExitStack
with ExitStack() as stack:
    resource = stack.enter_context(some_cm())
    stack.callback(cleanup_func, arg1, arg2)
```
Cleanup order: reverse of registration (LIFO).

## 43.9 Multiple / Nested Context Managers


```
with A() as a, B() as b:
    ...
# equivalent to:
with A() as a:
    with B() as b:
        ...
```
Entry: left to right / outer to inner. Exit: right to left / inner to outer (LIFO).

## 43.10 Asynchronous Context Managers


```
class AsyncCM:
    async def __aenter__(self):
        return self
    async def __aexit__(self, exc_type, exc_value, traceback):
        return False

async def main():
    async with AsyncCM() as x:
        ...
```

## 43.11 `@asynccontextmanager`


```
from contextlib import asynccontextmanager

@asynccontextmanager
async def my_async_context():
    SETUP
    try:
        yield VALUE
    finally:
        CLEANUP
```

## 43.12 `ContextVar`


```
from contextvars import ContextVar
var = ContextVar("name", default=DEFAULT)
token = var.set(NEW_VALUE)
var.get()
var.reset(token)
```

## 43.13 Best Practices Checklist

- Keep cleanup deterministic - never rely on `__del__`/GC timing.
- Keep each context manager focused on one resource/concern.
- Never suppress broadly (`suppress(Exception)`, unconditional `return True`).
- Document suppression, reuse, and reentrancy behavior explicitly.
- Prefer `contextlib` utilities over hand-written classes when they fit.
- Always test the exception/cleanup path, not just the happy path.
- Wrap post-`yield` cleanup in `try/finally` in every `@contextmanager` function.
- Acquire multiple locks in a single, consistent order across the codebase.
- Never assume a context manager is reusable or reentrant without checking.

## 43.14 Common Mistakes Quick Reference

| Mistake | Fix |
|---|---|
| Missing `__exit__` | Always implement both `__enter__` and `__exit__` |
| Wrong `__exit__` signature | Always `(self, exc_type, exc_value, traceback)` |
| Unconditional `return True` | Suppress only specific, documented exception types |
| Missing `yield` in `@contextmanager` | Always exactly one `yield` per execution path |
| Missing `try/finally` around `yield` | Cleanup is skipped on exception without it |
| Reusing an exhausted generator-based context manager | Call the factory function again for each use |
| Mixing `with`/`async with` | Match protocols: sync with sync, async with async |

### Python Context Managers - One-Page Quick Reference


```
WITH STATEMENT               with expr as x: BODY
PROTOCOL                     __enter__(self) -> value bound to 'as'
                              __exit__(self, exc_type, exc_value, tb) -> bool
SUPPRESSION                   __exit__ returns True  => exception suppressed
                              __exit__ returns False/None => exception propagates
GENERATOR-BASED               @contextmanager
                              def cm():
                                  SETUP
                                  try: yield VALUE
                                  finally: CLEANUP
STDLIB UTILITIES               closing(obj)              -> calls obj.close()
                                suppress(*exc_types)       -> ignores given exceptions
                                redirect_stdout(target)     -> reroutes print() output
                                redirect_stderr(target)      -> reroutes error output
                                nullcontext(value=None)       -> no-op context manager
                                ExitStack()                    -> dynamic multi-resource management
MULTIPLE / NESTED               with A() as a, B() as b: ...   (entry L->R, exit R->L, LIFO)
ASYNC                            async def __aenter__(self)
                                  async def __aexit__(self, exc_type, exc_value, tb)
                                  async with cm() as x: ...
                                  @asynccontextmanager  (same shape as @contextmanager, async def + yield)
CONTEXT VARIABLES                 ContextVar(name, default=...)
                                   var.set(value) -> token
                                   var.get()
                                   var.reset(token)
GOLDEN RULE                        Cleanup must be deterministic, guaranteed, and
                                    never silently hide genuine failures.
```

# 44. Conclusion

Context managers exist to solve one precise, universal problem: making sure that whenever a program acquires something scarce or stateful - a file, a connection, a lock, a transaction, a slice of GPU memory - it reliably gives that resource back, no matter what happens while it's in use. Every part of this handbook, from the simplest `with open(...) as f:` to `ExitStack`-composed production pipelines and `ContextVar`-scoped async request state, is ultimately an elaboration of that single guarantee.

What separates a working context manager from a well-designed one is precision about a small number of questions: does this suppress exceptions, and if so, which ones, deliberately? Is it safe to reuse across separate blocks? Is it safe to nest inside itself? Does its cleanup code correctly run even when the block it's protecting fails? This handbook treated each of these questions as a first-class topic, not an afterthought - because in production systems, it is almost always the *exceptional* path, not the happy path, where poorly designed resource management quietly turns into an outage.

The most valuable single habit to take away is this: whenever code acquires something that must eventually be released, restored, or committed, reach for a context manager - a class with `__enter__`/`__exit__`, a `@contextmanager` generator, or a `contextlib` utility - rather than a manual pair of setup/cleanup calls. That one habit, applied consistently, eliminates an entire, historically very common class of bugs: the leaked file, the held lock, the half-finished transaction, the connection that quietly exhausts a pool three weeks into production.

*End of Document*

**Kajola Gbenga | Data Scientist & AI Engineer**